<a href="https://colab.research.google.com/github/spirosChv/neuro208/blob/main/practicals/practical3_part5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simulating dendrites - Part 5

## Coincidence detection in the apical tree through calcium spike (BAC model)

In this tutorial we will see how coincident inputs at the dendrite and soma produce a supralinear response.

This exercise is based on [Schaefer et al. 2003](https://www.physiology.org/doi/full/10.1152/jn.00046.2003), with code modified from [ModelDB](https://senselab.med.yale.edu/modeldb/showmodel.cshtml?model=83344&file=%2FBACFiring%2FBACModel.hoc#tabs-1).

In [ ]:
!pip install neuron --quiet

## Compile ion channel models (`.mod` files)

In [ ]:
# @title Create the mod file `cad.mod`
# @markdown Execute this cell.
with open("cad2.mod", "w") as file:
  file.write("""

    TITLE decay of internal calcium concentration
    
    COMMENT
    : Internal calcium concentration due to calcium currents and pump.
    : Differential equations.
    :
    : Simple model of ATPase pump with 3 kinetic constants (Destexhe 92)
    :     Cai + P <-> CaP -> Cao + P  (k1,k2,k3)
    : A Michaelis-Menten approximation is assumed, which reduces the complexity
    : of the system to 2 parameters: 
    :       kt = <tot enzyme concentration> * k3  -> TIME CONSTANT OF THE PUMP
    :	kd = k2/k1 (dissociation constant)    -> EQUILIBRIUM CALCIUM VALUE
    : The values of these parameters are chosen assuming a high affinity of 
    : the pump to calcium and a low transport capacity (cfr. Blaustein, 
    : TINS, 11: 438, 1988, and references therein).  
    :
    : Units checked using "modlunit" -> factor 10000 needed in ca entry
    :
    : VERSION OF PUMP + DECAY (decay can be viewed as simplified buffering)
    :
    : All variables are range variables
    :
    : adopted from the lower model by AS 102199
    :
    : This mechanism was published in:  Destexhe, A. Babloyantz, A. and 
    : Sejnowski, TJ.  Ionic mechanisms for intrinsic slow oscillations in
    : thalamic relay neurons. Biophys. J. 65: 1538-1552, 1993)
    :
    : Written by Alain Destexhe, Salk Institute, Nov 12, 1992
    :
    ENDCOMMENT

    INDEPENDENT {t FROM 0 TO 1 WITH 1 (ms)}

    NEURON {
        SUFFIX cad2
        USEION ca READ ica, cai WRITE cai
        RANGE ca
        GLOBAL depth,cainf,taur
    }

    UNITS {
        (molar) = (1/liter)  : moles do not appear in units
        (mM) = (millimolar)
        (um) = (micron)
        (mA) = (milliamp)
        (msM)	= (ms mM)
        FARADAY = (faraday) (coulomb)
    }

    PARAMETER {
        depth	= .1	(um) : depth of shell
        taur = 80	(ms) : rate of calcium removal, changed from 200 to 80 (H.Markram)
        cainf	= 100e-6 (mM)
        cai (mM)
    }

    STATE {
        ca (mM) 
    }

    INITIAL {
        ca = cainf
    }

    ASSIGNED {
      ica  (mA/cm2)
      drive_channel	(mM/ms)
    }

    BREAKPOINT {
        SOLVE state METHOD euler
    }

    DERIVATIVE state { 
        drive_channel =  - (10000) * ica / (2 * FARADAY * depth)
        if (drive_channel <= 0.) { drive_channel = 0. }	: cannot pump inward

        ca' = drive_channel + (cainf-ca)/taur
        cai = ca
    }
  """)

In [ ]:
# @title Create the mod file `child.mod`
# @markdown Execute this cell.
with open("child.mod", "w") as file:
  file.write("""

    COMMENT
    iterator for traversing all the descendants of the currently accessed section

    section subtree_traverse("statement")

    executes statement for section and every descendant of section.
    Just before the statement is executed the currently accessed section is set.
    ENDCOMMENT

    NEURON {
      SUFFIX nothing
    }

    VERBATIM
    static subtree(sec, sym) Section* sec; Symbol* sym; {
      Section* child;

      nrn_pushsec(sec);	/* move these three (sec becomes child) */
      hoc_run_stmt(sym);	/* into the loop to do only the first level */
      nrn_popsec();

      for (child = sec->child; child; child = child->sibling) {
        subtree(child, sym);
      }
    }
    ENDVERBATIM

    PROCEDURE subtree_traverse_all() {
    VERBATIM
    {
      Section* chk_access();
      Symbol* hoc_parse_stmt();
      Symlist* symlist = (Symlist*)0;
      subtree(chk_access(), hoc_parse_stmt(gargstr(1), &symlist));
      /* if following not executed (ie hoc error in statement),
      some memory will leak */
      hoc_free_list(&symlist);
    }
    ENDVERBATIM
    }
  """)

In [ ]:
# @title Create the mod file `childa.mod`
# @markdown Execute this cell.
with open("childa.mod", "w") as file:
  file.write("""

    COMMENT
    iterator for traversing all the daughters of the currently accessed section

    section subtree_traverse("statement")

    executes statement for every daughter of section.
    Just before the statement is executed the currently accessed section is set.
    ENDCOMMENT

    NEURON {
      SUFFIX nothing
    }

    VERBATIM
    static subtree(sec, sym) Section* sec; Symbol* sym; {
      Section* child;
      for (child = sec->child; child; child = child->sibling) {
        nrn_pushsec(child);       /* move these three (sec becomes child) */
        hoc_run_stmt(sym);      /* into the loop to do only the first level */
        nrn_popsec();
      }
    }
    ENDVERBATIM

    PROCEDURE subtree_traverse() {
    VERBATIM
      {
      Section* chk_access();
      Symbol* hoc_parse_stmt();
      Symlist* symlist = (Symlist*)0;
      subtree(chk_access(), hoc_parse_stmt(gargstr(1), &symlist));
      /* if following not executed (ie hoc error in statement),
      some memory will leak */
      hoc_free_list(&symlist);
      }
    ENDVERBATIM
    }
  """)

In [ ]:
# @title Create the mod file `epsp.mod`
# @markdown Execute this cell.
with open("epsp.mod", "w") as file:
  file.write("""
    TITLE this model is built-in to neuron with suffix epsp

    COMMENT
    modified from syn2.mod
    injected current with exponential rise and decay current defined by
    i = 0 for t < onset and
    i=amp*((1-exp(-(t-onset)/tau0))-(1-exp(-(t-onset)/tau1)))
    for t > onset

    compare to experimental current injection:
    i = - amp*(1-exp(-t/t1))*(exp(-t/t2))

    -> tau1==t2   tau0 ^-1 = t1^-1 + t2^-1
    ENDCOMMENT

    INDEPENDENT {t FROM 0 TO 1 WITH 1 (ms)}

    NEURON {
        POINT_PROCESS epsp
        RANGE onset, tau0, tau1, imax, i, myv
        NONSPECIFIC_CURRENT i
    }

    UNITS {
        (nA) = (nanoamp)
        (mV) = (millivolt)
        (umho) = (micromho)
    }

    PARAMETER {
        onset=0 (ms)
        tau0=0.2 (ms)
        tau1=3.0 (ms)
        imax=0 (nA)
        v (mV)
    }

    ASSIGNED { 
        i (nA)
        myv (mV)
    }

    LOCAL a[2]
    LOCAL tpeak
    LOCAL adjust
    LOCAL amp

    BREAKPOINT {
        myv = v
        i = curr(t)
    }

    FUNCTION myexp(x) {
        if (x < -100) {
        myexp = 0
        } else {
        myexp = exp(x)
        }
    }

    FUNCTION curr(x) {
      tpeak=tau0*tau1*log(tau0/tau1)/(tau0-tau1)
      adjust=1/((1-myexp(-tpeak/tau0))-(1-myexp(-tpeak/tau1)))
      amp=adjust*imax
      if (x < onset) {
          curr = 0
      } else {
          a[0]=1-myexp(-(x-onset)/tau0)
          a[1]=1-myexp(-(x-onset)/tau1)
          curr = -amp*(a[0]-a[1])
      }
    }
  """)

In [ ]:
# @title Create the mod file `it2.mod`
# @markdown Execute this cell.
with open("it2.mod", "w") as file:
  file.write("""
    COMMENT
    changed from (AS Oct0899)
    ca.mod to lead to thalamic ca current inspired by destexhe and huguenrd
    Uses fixed eca instead of GHK eqn
    ENDCOMMENT

    INDEPENDENT {t FROM 0 TO 1 WITH 1 (ms)}

    NEURON {
        SUFFIX it2
        USEION ca READ eca WRITE ica
        RANGE m, h, gca, gcabar
        RANGE minf, hinf, mtau, htau, inactF, actF
        GLOBAL vshift,vmin,vmax, v12m, v12h, vwm, vwh
        GLOBAL am, ah, vm1, vm2, vh1, vh2, wm1, wm2, wh1, wh2
    }

    PARAMETER {
        gcabar = 0.0008 (mho/cm2)	: 0.12 mho/cm2
        vshift = 0 (mV) : voltage shift (affects all)
        cao = 2.5 (mM) : external ca concentration
        cai (mM)

        v (mV)
        dt (ms)
        celsius	(degC)
        vmin = -120	(mV)
        vmax = 100 (mV)

        v12m=50 (mV)
        v12h=78 (mV)
        vwm =7.4 (mV)
        vwh=5.0 (mV)
        am=3 (mV)
        ah=85 (mV)
        vm1=25 (mV)
        vm2=100 (mV)
        vh1=46 (mV)
        vh2=405 (mV)
        wm1=20 (mV)
        wm2=15 (mV)
        wh1=4 (mV)
        wh2=50 (mV)
    }

    UNITS {
        (mA) = (milliamp)
        (mV) = (millivolt)
        (pS) = (picosiemens)
        (um) = (micron)
        FARADAY = (faraday) (coulomb)
        R = (k-mole) (joule/degC)
        PI = (pi) (1)
    } 

    ASSIGNED {
        ica (mA/cm2)
        gca (pS/um2)
        eca (mV)
        minf hinf
        mtau (ms)
        htau (ms)
        tadj
    }

    STATE { m h }

    INITIAL { 
        trates(v+vshift)
        m = minf
        h = hinf
    }

    BREAKPOINT {
        SOLVE states
        gca = gcabar*m*m*h
        ica = gca * (v - eca)
    } 

    LOCAL mexp, hexp

    PROCEDURE states() {
        trates(v+vshift)
        m = m + mexp*(minf-m)
        h = h + hexp*(hinf-h)
        VERBATIM
        return 0;
        ENDVERBATIM
    }

    PROCEDURE trates(v) {  
      
        LOCAL tinc
        TABLE minf, mexp, hinf, hexp
        DEPEND dt
        FROM vmin TO vmax WITH 199
        rates(v): not consistently executed from here if usetable == 1
        tinc = -dt
        mexp = 1 - exp(tinc/mtau)
        hexp = 1 - exp(tinc/htau)
    }

    PROCEDURE rates(v_) {  
        LOCAL a, b

        minf = 1.0 / ( 1 + exp(-(v_+v12m)/vwm) )
        hinf = 1.0 / ( 1 + exp((v_+v12h)/vwh) )
        mtau = ( am + 1.0 / ( exp((v_+vm1)/wm1) + exp(-(v_+vm2)/wm2) ) )
        htau = ( ah + 1.0 / ( exp((v_+vh1)/wh1) + exp(-(v_+vh2)/wh2) ) )
    }
  """)

In [ ]:
# @title Create the mod file `kaprox.mod`
# @markdown Execute this cell.
with open("kaprox.mod", "w") as file:
  file.write("""
    TITLE K-A channel from Klee Ficker and Heinemann

    COMMENT
    modified to account for Dax A Current
    M.Migliore Jun 1997
    ENDCOMMENT

    NEURON {
      SUFFIX kap
      USEION k READ ek WRITE ik
      RANGE gkabar, gka
      GLOBAL ninf, linf, taul, taun, lmin
    }

    UNITS {
        (mA) = (milliamp)
        (mV) = (millivolt)
    }

    INDEPENDENT {t FROM 0 TO 1 WITH 1 (ms)}

    PARAMETER {
        dt (ms)
        v (mV)
        ek (mV) : must be explicitely def. in hoc
        celsius = 24	(degC)
        gkabar=.008 (mho/cm2)
        vhalfn=11 (mV)
        vhalfl=-56 (mV)
        a0l=0.05 (/ms)
        a0n=0.05 (/ms)
        zetan=-1.5 (1)
        zetal=3 (1)
        gmn=0.55 (1)
        gml=1 (1)
        lmin=2 (mS)
        nmin=0.1 (mS)
        pw=-1 (1)
        tq=-40
        qq=5
        q10=5
        qtl=1
    }

    STATE {
        n
        l
    }

    ASSIGNED {
        ik (mA/cm2)
        ninf
        linf
        taul
        taun
        gka
    }

    BREAKPOINT {
        SOLVE states
        gka = gkabar*n*l
        ik = gka*(v-ek)
    }

    FUNCTION alpn(v (mV)) {
        LOCAL zeta
        zeta=zetan+pw/(1+exp((v-tq)/qq))
        alpn = exp(1.e-3*zeta*(v-vhalfn)*9.648e4/(8.315*(273.16+celsius))) 
    }

    FUNCTION betn(v (mV)) {
        LOCAL zeta
        zeta=zetan+pw/(1+exp((v-tq)/qq))
        betn = exp(1.e-3*zeta*gmn*(v-vhalfn)*9.648e4/(8.315*(273.16+celsius))) 
    }

    FUNCTION alpl(v (mV)) {
        alpl = exp(1.e-3*zetal*(v-vhalfl)*9.648e4/(8.315*(273.16+celsius))) 
    }

    FUNCTION betl(v (mV)) {
        betl = exp(1.e-3*zetal*gml*(v-vhalfl)*9.648e4/(8.315*(273.16+celsius))) 
    }

    LOCAL facn,facl

    :if state_borgka is called from hoc, garbage or segmentation violation will
    :result because range variables won't have correct pointer.  This is because
    : only BREAKPOINT sets up the correct pointers to range variables.
    PROCEDURE states() {     : exact when v held constant; integrates over dt step
        rates(v)
        n = n + facn*(ninf - n)
        l = l + facl*(linf - l)
        VERBATIM
        return 0;
        ENDVERBATIM
    }

    PROCEDURE rates(v (mV)) { :callable from hoc
        LOCAL a,qt
        qt=q10^((celsius-24)/10)
        a = alpn(v)
        ninf = 1/(1 + a)
        taun = betn(v)/(qt*a0n*(1+a))
        if (taun<nmin) {taun=nmin}
        facn = (1 - exp(-dt/taun))
        a = alpl(v)
        linf = 1/(1+ a)
        taul = 0.26*(v+50)/qtl
        if (taul<lmin/qtl) {taul=lmin/qtl}
        facl = (1 - exp(-dt/taul))
    }
  """)

In [ ]:
# @title Create the mod file `kca.mod`
# @markdown Execute this cell.
with open("kca.mod", "w") as file:
  file.write("""

    TITLE Calcium-dependent potassium channel

    COMMENT
    Based on Pennefather (1990) -- sympathetic ganglion cells
    taken from Reuveni et al (1993) -- neocortical cells

    Author: Zach Mainen, Salk Institute, 1995, zach@salk.edu
    ENDCOMMENT

    INDEPENDENT {t FROM 0 TO 1 WITH 1 (ms)}

    NEURON {
        SUFFIX kca
        USEION k READ ek WRITE ik
        USEION ca READ cai
        RANGE n, gk, gbar
        RANGE ninf, ntau
        GLOBAL Ra, Rb, caix
        GLOBAL q10, temp, tadj, vmin, vmax
    }

    UNITS {
        (mA) = (milliamp)
        (mV) = (millivolt)
        (pS) = (picosiemens)
        (um) = (micron)
    } 

    PARAMETER {
      gbar = 10 (pS/um2) : 0.03 mho/cm2
      v (mV)
      cai (mM)
      caix = 1

      Ra = 0.01	(/ms) : max act rate
      Rb = 0.02	(/ms) : max deact rate

      dt (ms)
      celsius (degC)
      temp = 23	(degC) : original temp
      q10 = 2.3 : temperature sensitivity

      vmin = -120 (mV)
      vmax = 100 (mV)
    } 


    ASSIGNED {
        a (/ms)
        b (/ms)
        ik (mA/cm2)
        gk (pS/um2)
        ek (mV)
        ninf
        ntau (ms)
        tadj
    }

    STATE { n }

    INITIAL { 
        rates(cai)
        n = ninf
    }

    BREAKPOINT {
        SOLVE states
        gk = tadj*gbar*n
        ik = (1e-4) * gk * (v - ek)
    } 

    LOCAL nexp

    PROCEDURE states() { :Computes state variable n
      rates(cai) : at the current v and dt.
      n = n + nexp*(ninf-n)

      VERBATIM
      return 0;
      ENDVERBATIM
    }

    PROCEDURE rates(cai(mM)) {
      LOCAL tinc

      a = Ra * cai^caix
      b = Rb
      ntau = 1/(a+b)
      ninf = a*ntau

      tadj = q10^((celsius - temp)/10)

      tinc = -dt * tadj
      nexp = 1 - exp(tinc/ntau)
    }
  """)

In [ ]:
# @title Create the mod file `km.mod`
# @markdown Execute this cell.
with open("km.mod", "w") as file:
  file.write("""
    TITLE Potassium channel I-M (muscarinic K channel)

    COMMENT
    Potassium channel, Hodgkin-Huxley style kinetics
    Based on I-M, Slow, noninactivating

    Author: Zach Mainen, Salk Institute, 1995, zach@salk.edu
    ENDCOMMENT

    INDEPENDENT {t FROM 0 TO 1 WITH 1 (ms)}

    NEURON {
        SUFFIX km
        USEION k READ ek WRITE ik
        RANGE n, gk, gbar
        RANGE ninf, ntau
        GLOBAL Ra, Rb
        GLOBAL q10, temp, tadj, vmin, vmax
    }

    UNITS {
        (mA) = (milliamp)
        (mV) = (millivolt)
        (pS) = (picosiemens)
        (um) = (micron)
    } 

    PARAMETER {
        gbar = 10 (pS/um2) : 0.03 mho/cm2
        v (mV)

        tha = -30 (mV) : v 1/2 for inf
        qa = 9 (mV) : inf slope

        Ra = 0.001 (/ms) : max act rate  (slow)
        Rb = 0.001 (/ms) : max deact rate  (slow)

        dt (ms)
        celsius (degC)
        temp = 23	(degC) : original temp
        q10  = 2.3 : temperature sensitivity

        vmin = -120	(mV)
        vmax = 100	(mV)
    } 

    ASSIGNED {
        a (/ms)
        b (/ms)
        ik (mA/cm2)
        gk (pS/um2)
        ek (mV)
        ninf
        ntau (ms)
        tadj
    }

    STATE { n }

    INITIAL { 
        trates(v)
        n = ninf
    }

    BREAKPOINT {
        SOLVE states
        gk = tadj*gbar*n
        ik = (1e-4) * gk * (v - ek)
    } 

    LOCAL nexp

    PROCEDURE states() { :Computes state variable n
        trates(v) : at the current v and dt.
        n = n + nexp*(ninf-n)
        VERBATIM
        return 0;
        ENDVERBATIM
    }

    PROCEDURE trates(v) { :Computes rate and other constants at current v.
        :Call once from HOC to initialize inf at resting v.
        LOCAL tinc
        TABLE ninf, nexp
        DEPEND dt, celsius, temp, Ra, Rb, tha, qa

        FROM vmin TO vmax WITH 199

        rates(v): not consistently executed from here if usetable_hh == 1

        tadj = q10^((celsius - temp)/10)

        tinc = -dt * tadj
        nexp = 1 - exp(tinc/ntau)
    }

    PROCEDURE rates(v) { :Computes rate and other constants at current v.
        :Call once from HOC to initialize inf at resting v.

        a = Ra * (v - tha) / (1 - exp(-(v - tha)/qa))
        b = -Rb * (v - tha) / (1 - exp((v - tha)/qa))
        ntau = 1/(a+b)
        ninf = a*ntau
    }
  """)

In [ ]:
# @title Create the mod file `kv.mod`
# @markdown Execute this cell.
with open("kv.mod", "w") as file:
  file.write("""
    TITLE  Potassium channel
    COMMENT
    Potassium channel, Hodgkin-Huxley style kinetics
    Kinetic rates based roughly on Sah et al. and Hamill et al. (1991)

    Author: Zach Mainen, Salk Institute, 1995, zach@salk.edu
    ENDCOMMENT

    INDEPENDENT {t FROM 0 TO 1 WITH 1 (ms)}

    NEURON {
      SUFFIX kv
      USEION k READ ek WRITE ik
      RANGE n, gk, gbar
      RANGE ninf, ntau
      GLOBAL Ra, Rb
      GLOBAL q10, temp, tadj, vmin, vmax
    }

    UNITS {
      (mA) = (milliamp)
      (mV) = (millivolt)
      (pS) = (picosiemens)
      (um) = (micron)
    } 

    PARAMETER {
        gbar = 5 (pS/um2) : 0.03 mho/cm2
        v (mV)

        tha = 25 (mV) : v 1/2 for inf
        qa = 9 (mV) : inf slope

        Ra = 0.02	(/ms) : max act rate
        Rb = 0.002 (/ms) : max deact rate

        dt (ms)
        celsius (degC)
        temp = 23	(degC) : original temp
        q10 = 2.3 : temperature sensitivity

        vmin = -120	(mV)
        vmax = 100	(mV)
    } 

    ASSIGNED {
        a (/ms)
        b (/ms)
        ik (mA/cm2)
        gk (pS/um2)
        ek (mV)
        ninf
        ntau(ms)
        tadj
    }

    STATE { n }

    INITIAL { 
      trates(v)
      n = ninf
    }

    BREAKPOINT {
        SOLVE states
        gk = tadj*gbar*n
        ik = (1e-4) * gk * (v - ek)
    } 

    LOCAL nexp

    PROCEDURE states() { :Computes state variable n
        trates(v) : at the current v and dt.
        n = n + nexp*(ninf-n)
        VERBATIM
        return 0;
        ENDVERBATIM
    }

    PROCEDURE trates(v) { :Computes rate and other constants at current v.
      :Call once from HOC to initialize inf at resting v.
      LOCAL tinc
      TABLE ninf, nexp
      DEPEND dt, celsius, temp, Ra, Rb, tha, qa

      FROM vmin TO vmax WITH 199

      rates(v): not consistently executed from here if usetable_hh == 1

      tadj = q10^((celsius - temp)/10)

      tinc = -dt * tadj
      nexp = 1 - exp(tinc/ntau)
    }


    PROCEDURE rates(v) { :Computes rate and other constants at current v.
        :Call once from HOC to initialize inf at resting v.

        a = Ra * (v - tha) / (1 - exp(-(v - tha)/qa))
        b = -Rb * (v - tha) / (1 - exp((v - tha)/qa))
        ntau = 1/(a+b)
        ninf = a*ntau
    }
  """)

In [ ]:
# @title Create the mod file `na.mod`
# @markdown Execute this cell.
with open("na.mod", "w") as file:
  file.write("""

    TITLE Sodium channel

    COMMENT
    Sodium channel, Hodgkin-Huxley style kinetics.  

    Kinetics were fit to data from Huguenard et al. (1988) and Hamill et al. (1991)

    qi is not well constrained by the data, since there are no points
    between -80 and -55.  So this was fixed at 5 while the thi1,thi2,Rg,Rd
    were optimized using a simplex least square proc

    voltage dependencies are shifted approximately from the best
    fit to give higher threshold

    Author: Zach Mainen, Salk Institute, 1994, zach@salk.edu
    ENDCOMMENT

    INDEPENDENT {t FROM 0 TO 1 WITH 1 (ms)}

    NEURON {
        SUFFIX na
        USEION na READ ena WRITE ina
        RANGE m, h, gna, gbar
        GLOBAL tha, thi1, thi2, qa, qi, qinf, thinf
        RANGE minf, hinf, mtau, htau
        GLOBAL Ra, Rb, Rd, Rg
        GLOBAL q10, temp, tadj, vmin, vmax, vshift
    }

    PARAMETER {
        gbar = 1000 (pS/um2) : 0.12 mho/cm2
        vshift = -10 (mV) : voltage shift (affects all)

        tha = -35 (mV) : v 1/2 for act (-42)
        qa = 9 (mV) : act slope
        Ra = 0.182	(/ms) : open (v)
        Rb = 0.124 (/ms) : close (v)

        thi1 = -50 (mV) : v 1/2 for inact
        thi2 = -75 (mV) : v 1/2 for inact
        qi = 5 (mV) : inact tau slope
        thinf = -65	(mV) : inact inf slope
        qinf = 6.2 (mV) : inact inf slope
        Rg = 0.0091	(/ms) : inact (v)
        Rd = 0.024 (/ms) : inact recov (v)

        temp = 23	(degC) : original temp 
        q10 = 2.3 : temperature sensitivity

        v (mV)
        dt (ms)
        celsius (degC)
        vmin = -120	(mV)
        vmax = 100 (mV)
    }


    UNITS {
        (mA) = (milliamp)
        (mV) = (millivolt)
        (pS) = (picosiemens)
        (um) = (micron)
    } 

    ASSIGNED {
        ina (mA/cm2)
        gna (pS/um2)
        ena (mV)
        minf
        hinf
        mtau (ms)
        htau (ms)
        tadj
    }

    STATE { m h }

    INITIAL { 
        trates(v+vshift)
        m = minf
        h = hinf
    }

    BREAKPOINT {
        SOLVE states
        gna = tadj*gbar*m*m*m*h
        ina = (1e-4) * gna * (v - ena)
    } 

    LOCAL mexp, hexp 

    PROCEDURE states() { :Computes state variables m, h, and n
        trates(v+vshift) : at the current v and dt.
        m = m + mexp*(minf-m)
        h = h + hexp*(hinf-h)
        VERBATIM
        return 0;
        ENDVERBATIM
    }

    PROCEDURE trates(v) {
        LOCAL tinc
        TABLE minf, mexp, hinf, hexp
        DEPEND dt, celsius, temp, Ra, Rb, Rd, Rg, tha, thi1, thi2, qa, qi, qinf

        FROM vmin TO vmax WITH 199

        rates(v): not consistently executed from here if usetable == 1

        tadj = q10^((celsius - temp)/10)
        tinc = -dt * tadj

        mexp = 1 - exp(tinc/mtau)
        hexp = 1 - exp(tinc/htau)
    }

    PROCEDURE rates(vm) {
        LOCAL  a, b

        a = trap0(vm,tha,Ra,qa)
        b = trap0(-vm,-tha,Rb,qa)
        mtau = 1/(a+b)
        minf = a*mtau

        :"h" inactivation 

        a = trap0(vm,thi1,Rd,qi)
        b = trap0(-vm,-thi2,Rg,qi)
        htau = 1/(a+b)
        hinf = 1/(1+exp((vm-thinf)/qinf))
    }

    FUNCTION trap0(v,th,a,q) {
        if (fabs(v/th) > 1e-6) {
            trap0 = a * (v - th) / (1 - exp(-(v - th)/q))
        } else {
            trap0 = a * q
        }
    }
  """)

In [ ]:
# @title Create the mod file `SlowCa.mod`
# @markdown Execute this cell.
with open("SlowCa.mod", "w") as file:
  file.write("""
    TITLE HVA Ca2+ current

    COMMENT
    Uses fixed eca instead of GHK eqn
    Based on Reuveni, Friedman, Amitai and Gutnick (1993) J. Neurosci. 13: 4609-4621.
    Changed from (AS Oct0899) ca.mod
    
    Author: Zach Mainen, Salk Institute, 1994, zach@salk.edu
    ENDCOMMENT

    INDEPENDENT {t FROM 0 TO 1 WITH 1 (ms)}

    NEURON {
        SUFFIX sca
        USEION ca READ eca WRITE ica
        RANGE m, h, gca, gbar
        RANGE minf, hinf, mtau, htau, inactF, actF
        GLOBAL q10, temp, tadj, vmin, vmax, vshift
    }

    UNITS {
        (mA) = (milliamp)
        (mV) = (millivolt)
        (pS) = (picosiemens)
        (um) = (micron)
        FARADAY = (faraday) (coulomb)
        R = (k-mole) (joule/degC)
        PI = (pi) (1)
    }

    PARAMETER {
        inactF = 3
        actF = 1
        gbar = 0.1 (pS/um2) : 0.12 mho/cm2
        vshift = 0 (mV) : voltage shift (affects all)

        cao = 2.5 (mM) : external ca concentration
        cai (mM)

        temp = 23 (degC) : original temp 
        q10 = 2.3 : temperature sensitivity

        v (mV)
        dt (ms)
        celsius (degC)
        vmin = -120	(mV)
        vmax = 100 (mV)
    }

    ASSIGNED {
        ica (mA/cm2)
        gca (pS/um2)
        eca (mV)
        minf
        hinf
        mtau (ms)
        htau (ms)
        tadj
    }

    STATE { m h }

    INITIAL { 
        trates(v+vshift)
        m = minf
        h = hinf
    }

    BREAKPOINT {
        SOLVE states
        gca = tadj*gbar*m*m*h
        ica = (1e-4) * gca * (v - eca)
    } 

    LOCAL mexp, hexp

    PROCEDURE states() {
        trates(v+vshift)
        m = m + mexp*(minf-m)
        h = h + hexp*(hinf-h)
        VERBATIM
        return 0;
        ENDVERBATIM
    }

    PROCEDURE trates(v) {
        LOCAL tinc
        TABLE minf, mexp, hinf, hexp
        DEPEND dt, celsius, temp, inactF

        FROM vmin TO vmax WITH 199

        rates(v): not consistently executed from here if usetable == 1

        tadj = q10^((celsius - temp)/10)
        tinc = -dt * tadj

        mexp = 1 - exp(tinc/mtau)
        hexp = 1 - exp(tinc/htau)
    }

    PROCEDURE rates(vm) {  
        LOCAL a, b

        a = 0.055*(-27 - vm)/(exp((-27-vm)/3.8) - 1)/actF
        b = 0.94*exp((-75-vm)/17)/actF

        mtau = 1/(a+b)
        minf = a*mtau

        :"h" inactivation 

        a = 0.000457*exp((-13-vm)/50)/inactF
        b = 0.0065/(exp((-vm-15)/28) + 1)/inactF

        htau = 1/(a+b) : originally *1
        hinf = a*htau
    }

    FUNCTION efun(z) {
        if (fabs(z) < 1e-4) {
        efun = 1 - z/2
        }else{
        efun = z/(exp(z) - 1)
        }
    }
  """)

In [ ]:
# @title Create the morphology file `morphology.nrn`
# @markdown Execute this cell.
with open("morphology.nrn", "w") as file:
  file.write("""
    //Morphology of the model neuron
    {create somaA}
    {access somaA}
    {nseg = 10}
    {pt3dclear()}
    {pt3dadd(3.670000, 4.120000, 2.320000, 10.000000)}
    {pt3dadd(3.661211, 4.500661, 2.320000, 10.000000)}
    {pt3dadd(3.769587, 5.038524, 2.320000, 10.000000)}
    {pt3dadd(3.874381, 5.398300, 2.320000, 10.000000)}
    {pt3dadd(4.644151, 6.511971, 2.320000, 10.000000)}
    {pt3dadd(3.873478, 7.274557, 2.320000, 10.000000)}
    {pt3dadd(4.924000, 8.377978, 2.320000, 11.835451)}
    {pt3dadd(6.613045, 9.470419, 2.320000, 16.140561)}
    {pt3dadd(6.626815, 10.107840, 2.320000, 16.923574)}
    {pt3dadd(6.640585, 10.745261, 2.320000, 17.707203)}
    {pt3dadd(5.751317, 11.818075, 2.320000, 16.895580)}
    {pt3dadd(5.589602, 12.509800, 2.320000, 17.374173)}
    {pt3dadd(5.290544, 13.105153, 2.320000, 18.151169)}
    {pt3dadd(6.046100, 13.917766, 2.320000, 20.629677)}
    {pt3dadd(5.644771, 14.546938, 2.320000, 20.765994)}
    {pt3dadd(5.265636, 15.427633, 2.320000, 20.788740)}
    {pt3dadd(4.848639, 16.037748, 2.320000, 21.019496)}
    {pt3dadd(4.398120, 16.620047, 2.320000, 21.277036)}
    {pt3dadd(3.963878, 17.218584, 2.320000, 21.498635)}
    {pt3dadd(3.452440, 17.945405, 2.320000, 21.608513)}
    {pt3dadd(2.673044, 18.872636, 2.320000, 22.306155)}
    {pt3dadd(2.220901, 19.455799, 2.320000, 22.538449)}
    {pt3dadd(1.546263, 20.241334, 2.320000, 23.158168)}
    {pt3dadd(1.086689, 20.912560, 2.320000, 23.399890)}
    {pt3dadd(2.011039, 21.833973, 2.320000, 20.774073)}
    {pt3dadd(1.928306, 22.495736, 2.320000, 20.082567)}
    {pt3dadd(0.313701, 22.845447, 2.320000, 22.607161)}
    {pt3dadd(0.440136, 23.368777, 2.320000, 21.640430)}
    {pt3dadd(0.523812, 24.025403, 2.320000, 20.646519)}
    {pt3dadd(1.940963, 25.491034, 2.320000, 16.797485)}
    {pt3dadd(2.216292, 26.234986, 2.320000, 16.497949)}
    {pt3dadd(3.077789, 27.390510, 2.320000, 14.125388)}
    {pt3dadd(3.229365, 28.037974, 2.320000, 13.296790)}
    {pt3dadd(2.933186, 28.767380, 2.320000, 12.094480)}
    {pt3dadd(3.018492, 29.474478, 2.320000, 11.642879)}
    {pt3dadd(2.899895, 30.292807, 2.320000, 11.558316)}
    {pt3dadd(3.018818, 30.981991, 2.320000, 11.046083)}
    {pt3dadd(1.693964, 30.638115, 2.320000, 10.000000)}
    {pt3dadd(1.607618, 31.120694, 2.320000, 10.000000)}
    {pt3dadd(2.935852, 32.479522, 2.320000, 10.000000)}
    {pt3dadd(2.764193, 32.832828, 2.320000, 10.000000)}
    {pt3dadd(0.000000, 33.740000, 2.320000, 10.000000)}

    {create dendA1_0}
    {somaA connect dendA1_0(0), 1.000000}
    {access dendA1_0}
    {nseg = 1}
    {pt3dclear()}
    {pt3dadd(4.13, 33.33, 3.12, 9.18)} // diameter corrected from 0.46 A.R. 17.8.99
    {pt3dadd(4.13, 33.33, 3.12, 9.18)}
    {pt3dadd(5.05, 45.68, 3.12, 7.34)}
    {pt3dadd(5.05, 55.14, 3.12, 9.18)}

    {create dendA1_00}
    {dendA1_0 connect dendA1_00(0), 1}
    {access dendA1_00}
    {nseg = 2}
    {pt3dclear()}
    {pt3dadd(5.05, 55.14, 3.12, 6.42)}
    {pt3dadd(3.67, 65.02, 3.12, 6.42)}
    {pt3dadd(3.21, 70.78, 3.12, 7.34)}
    {pt3dadd(3.67, 77.37, 3.12, 7.34)}
    {pt3dadd(3.93, 83.37, 5.16, 6.42)}
    {pt3dadd(4.85, 88.72, 5.16, 6.42)}

    {create dendA1_000}
    {dendA1_00 connect dendA1_000(0), 1}
    {access dendA1_000}
    {nseg = 2}
    {pt3dclear()}
    {pt3dadd(4.85, 88.72, 5.16, 5.50)}
    {pt3dadd(5.31, 94.07, 5.16, 5.50)}
    {pt3dadd(6.68, 99.00, 5.16, 5.50)}
    {pt3dadd(6.68, 105.18, 5.16, 5.50)}
    {pt3dadd(5.31, 111.35, 5.16, 5.50)}
    {pt3dadd(4.39, 117.11, 5.04, 5.50)}
    {pt3dadd(3.47, 122.87, 5.00, 5.50)}
    {pt3dadd(1.18, 128.22, 4.92, 5.50)}

    {create dendA1_0000}
    {dendA1_000 connect dendA1_0000(0), 1}
    {access dendA1_0000}
    {nseg = 2}
    {pt3dclear()}
    {pt3dadd(1.18, 128.22, 4.92, 5.50)}
    {pt3dadd(-0.66, 132.75, 4.88, 5.50)}
    {pt3dadd(-1.58, 137.69, 5.76, 5.50)}
    {pt3dadd(-1.12, 143.04, 5.64, 5.50)}
    {pt3dadd(-1.12, 146.33, 5.96, 5.50)}
    {pt3dadd(-2.49, 149.62, 5.08, 7.34)}
    {pt3dadd(-2.03, 153.74, 7.52, 5.50)}
    {pt3dadd(-1.31, 158.84, 8.96, 5.50)}

    {create dendA1_00000}
    {dendA1_0000 connect dendA1_00000(0), 1}
    {access dendA1_00000}
    {nseg = 1}
    {pt3dclear()}
    {pt3dadd(-1.31, 158.84, 8.96, 5.50)}
    {pt3dadd(-0.85, 162.96, 9.12, 5.50)}
    {pt3dadd(-0.39, 166.25, 8.04, 5.50)}

    {create dendA1_000000}
    {dendA1_00000 connect dendA1_000000(0), 1}
    {access dendA1_000000}
    {nseg = 2}
    {pt3dclear()}
    {pt3dadd(-0.39, 166.25, 8.04, 5.50)}
    {pt3dadd(-0.85, 171.60, 8.04, 5.50)}
    {pt3dadd(-1.31, 178.19, 8.04, 5.50)}
    {pt3dadd(-1.31, 184.77, 8.04, 5.50)}
    {pt3dadd(-2.23, 192.18, 8.68, 5.50)}
    {pt3dadd(-1.77, 198.35, 8.20, 5.50)}
    {pt3dadd(-2.23, 204.93, 7.72, 5.50)}
    {pt3dadd(-3.15, 207.40, 8.56, 5.50)}

    {create dendA1_0000000}
    {dendA1_000000 connect dendA1_0000000(0), 1}
    {access dendA1_0000000}
    {nseg = 1}
    {pt3dclear()}
    {pt3dadd(-3.15, 207.40, 8.56, 5.50)}
    {pt3dadd(-1.31, 212.75, 2.00, 5.50)}
    {pt3dadd(-0.85, 216.87, 1.92, 5.50)}

    {create dendA1_00000000}
    {dendA1_0000000 connect dendA1_00000000(0), 1}
    {access dendA1_00000000}
    {nseg = 2}
    {pt3dclear()}
    {pt3dadd(-0.85, 216.87, 1.92, 5.50)}
    {pt3dadd(-1.31, 222.22, 4.56, 5.50)}
    {pt3dadd(-0.85, 228.80, 4.28, 5.50)}
    {pt3dadd(0.71, 233.94, 3.40, 5.50)}
    {pt3dadd(0.71, 241.76, 3.76, 5.50)}

    {create dendA1_000000000}
    {dendA1_00000000 connect dendA1_000000000(0), 1}
    {access dendA1_000000000}
    {nseg = 1}
    {pt3dclear()}
    {pt3dadd(0.71, 241.76, 3.76, 5.50)}
    {pt3dadd(1.62, 246.70, 3.64, 5.50)}
    {pt3dadd(1.16, 252.05, 3.64, 5.50)}
    {pt3dadd(0.71, 256.99, 3.64, 5.50)}

    {create dendA1_0000000000}
    {dendA1_000000000 connect dendA1_0000000000(0), 1}
    {access dendA1_0000000000}
    {nseg = 5}
    {pt3dclear()}
    {pt3dadd(0.71, 256.99, 3.64, 5.50)}
    {pt3dadd(2.54, 261.93, 3.64, 5.50)}
    {pt3dadd(2.08, 266.87, 5.40, 5.50)}
    {pt3dadd(1.62, 273.86, 5.36, 5.50)}
    {pt3dadd(1.62, 279.21, 5.28, 5.50)}
    {pt3dadd(1.62, 281.68, 5.24, 5.50)}
    {pt3dadd(3.00, 286.21, 5.20, 5.50)}
    {pt3dadd(2.54, 291.15, 5.20, 5.50)}
    {pt3dadd(1.62, 294.85, 5.16, 5.50)}
    {pt3dadd(2.54, 300.20, 5.16, 5.50)}
    {pt3dadd(2.54, 304.31, 5.12, 5.50)}
    {pt3dadd(2.27, 307.80, 5.12, 5.50)}
    {pt3dadd(2.73, 311.50, 5.12, 5.50)}
    {pt3dadd(2.73, 314.79, 5.12, 5.50)}
    {pt3dadd(3.65, 318.91, 5.12, 5.50)}
    {pt3dadd(3.65, 322.20, 5.12, 5.50)}
    {pt3dadd(3.19, 326.32, 5.12, 5.50)}
    {pt3dadd(3.65, 331.25, 5.12, 5.50)}
    {pt3dadd(6.86, 334.14, 5.08, 5.50)}
    {pt3dadd(7.32, 339.07, 5.08, 5.50)}
    {pt3dadd(7.32, 344.01, 6.20, 5.50)}
    {pt3dadd(7.78, 349.77, 5.96, 5.50)}
    {pt3dadd(9.15, 354.30, 5.80, 5.50)}
    {pt3dadd(9.15, 360.47, 5.80, 5.50)}

    {create dendA1_00000000000}
    {dendA1_0000000000 connect dendA1_00000000000(0), 1}
    {access dendA1_00000000000}
    {nseg = 1}
    {pt3dclear()}
    {pt3dadd(9.15, 360.47, 5.80, 5.50)}
    {pt3dadd(9.61, 364.59, 7.40, 5.50)}
    {pt3dadd(11.90, 369.94, 8.12, 5.50)}

    {create dendA1_000000000000}
    {dendA1_00000000000 connect dendA1_000000000000(0), 1}
    {access dendA1_000000000000}
    {nseg = 13}
    {pt3dclear()}
    {pt3dadd(11.90, 369.94, 8.12, 5.50)}
    {pt3dadd(12.36, 375.70, 8.12, 5.50)}
    {pt3dadd(12.82, 378.99, 8.12, 5.50)}
    {pt3dadd(12.11, 383.73, 7.60, 5.50)}
    {pt3dadd(13.95, 388.67, 7.52, 5.50)}
    {pt3dadd(15.32, 393.61, 7.48, 5.50)}
    {pt3dadd(15.32, 397.72, 7.48, 5.50)}
    {pt3dadd(14.41, 401.02, 7.48, 5.50)}
    {pt3dadd(16.24, 405.13, 7.48, 5.50)}
    {pt3dadd(17.16, 410.07, 7.40, 5.50)}
    {pt3dadd(17.16, 414.60, 7.40, 5.50)}
    {pt3dadd(17.16, 419.95, 7.40, 5.50)}
    {pt3dadd(17.16, 426.53, 7.36, 5.50)}
    {pt3dadd(17.62, 433.53, 7.24, 5.50)}
    {pt3dadd(17.62, 437.64, 7.24, 5.50)}
    {pt3dadd(17.16, 443.81, 7.24, 5.50)}
    {pt3dadd(15.78, 448.34, 7.24, 5.50)}
    {pt3dadd(17.62, 452.05, 7.24, 5.50)}
    {pt3dadd(18.08, 456.16, 7.24, 5.50)}
    {pt3dadd(18.79, 467.11, 7.24, 5.50)}
    {pt3dadd(20.16, 479.86, 7.24, 5.50)}
    {pt3dadd(22.92, 493.03, 7.24, 5.50)}
    {pt3dadd(25.21, 502.08, 7.24, 5.50)}
    {pt3dadd(23.83, 507.85, 7.24, 5.50)}
    {pt3dadd(23.83, 515.25, 7.24, 5.50)}
    {pt3dadd(22.92, 522.25, 7.24, 4.58)}
    {pt3dadd(22.92, 528.01, 7.24, 4.58)}
    {pt3dadd(24.75, 533.36, 7.24, 4.58)}
    {pt3dadd(25.03, 539.37, 8.44, 4.58)}
    {pt3dadd(25.95, 545.13, 8.20, 4.58)}
    {pt3dadd(26.86, 550.89, 8.08, 7.34)}
    {pt3dadd(27.32, 555.42, 8.08, 10.10)}
    {pt3dadd(27.32, 557.89, 8.04, 12.84)}
    {pt3dadd(27.32, 558.30, 8.04, 14.68)}
    {pt3dadd(29.16, 563.24, 8.04, 10.10)}
    {pt3dadd(30.08, 568.17, 8.04, 6.42)}
    {pt3dadd(30.53, 571.88, 8.04, 5.50)}
    {pt3dadd(32.83, 576.82, 8.04, 5.50)}
    {pt3dadd(31.45, 582.17, 8.04, 5.50)}
    {pt3dadd(31.91, 588.34, 8.04, 5.50)}
    {pt3dadd(32.37, 592.45, 8.04, 5.50)}
    {pt3dadd(32.37, 596.98, 8.04, 5.50)}
    {pt3dadd(30.53, 602.74, 8.04, 5.50)}
    {pt3dadd(31.91, 608.91, 8.04, 5.50)}
    {pt3dadd(33.05, 613.63, 10.76, 4.58)}
    {pt3dadd(34.42, 620.63, 10.76, 4.58)}
    {pt3dadd(35.80, 624.33, 10.76, 4.58)}
    {pt3dadd(35.80, 629.68, 10.76, 4.58)}
    {pt3dadd(37.18, 634.62, 10.76, 4.58)}
    {pt3dadd(36.72, 639.56, 10.76, 4.58)}
    {pt3dadd(35.34, 644.09, 10.76, 4.58)}
    {pt3dadd(36.26, 649.43, 10.76, 4.58)}
    {pt3dadd(36.72, 656.02, 10.76, 4.58)}
    {pt3dadd(34.88, 661.37, 11.96, 4.58)}
    {pt3dadd(34.88, 667.95, 11.96, 4.58)}
    {pt3dadd(34.88, 676.18, 11.96, 4.58)}
    {pt3dadd(33.96, 684.00, 11.92, 4.58)}
    {pt3dadd(34.23, 688.73, 13.80, 3.66)}
    {pt3dadd(35.61, 694.08, 12.32, 3.66)}

    {create dendA1_0000000000000}
    {dendA1_000000000000 connect dendA1_0000000000000(0), 1}
    {access dendA1_0000000000000}
    {nseg = 5}
    {pt3dclear()}
    {pt3dadd(35.61, 694.08, 12.32, 3.66)}
    {pt3dadd(40.19, 698.20, 12.32, 3.66)}
    {pt3dadd(43.40, 701.90, 13.64, 2.76)}
    {pt3dadd(42.94, 707.66, 13.60, 2.76)}
    {pt3dadd(41.57, 712.60, 13.60, 2.76)}
    {pt3dadd(41.57, 718.77, 15.08, 2.76)}
    {pt3dadd(43.40, 723.71, 14.92, 2.76)}
    {pt3dadd(44.78, 732.35, 14.88, 2.76)}
    {pt3dadd(44.78, 736.88, 15.08, 2.76)}
    {pt3dadd(46.61, 739.35, 15.08, 2.76)}
    {pt3dadd(45.70, 745.11, 15.88, 2.76)}
    {pt3dadd(49.37, 750.87, 14.84, 2.76)}
    {pt3dadd(48.45, 753.75, 17.48, 1.84)}
    {pt3dadd(50.74, 757.05, 17.40, 1.84)}
    {pt3dadd(51.20, 760.34, 17.88, 1.84)}
    {pt3dadd(50.97, 762.97, 19.32, 1.84)}
    {pt3dadd(50.05, 768.74, 18.64, 1.84)}

    {create dendA1_0000000000001}
    {dendA1_000000000000 connect dendA1_0000000000001(0), 1}
    {access dendA1_0000000000001}
    {nseg = 3}
    {pt3dclear()}
    {pt3dadd(35.61, 694.08, 12.32, 3.66)}
    {pt3dadd(34.00, 699.19, 9.16, 3.66)}
    {pt3dadd(34.46, 704.54, 9.04, 3.66)}
    {pt3dadd(36.29, 711.95, 9.04, 3.66)}
    {pt3dadd(38.13, 717.71, 9.88, 3.66)}
    {pt3dadd(41.80, 721.82, 10.92, 3.66)}
    {pt3dadd(41.80, 728.00, 10.60, 3.66)}
    {pt3dadd(41.34, 734.58, 10.52, 3.66)}
    {pt3dadd(39.96, 740.75, 10.32, 3.66)}
    {pt3dadd(39.50, 745.28, 7.68, 3.66)}

    {create dendA1_00000000000010}
    {dendA1_0000000000001 connect dendA1_00000000000010(0), 1}
    {access dendA1_00000000000010}
    {nseg = 12}
    {pt3dclear()}
    {pt3dadd(39.50, 745.28, 7.68, 2.76)}
    {pt3dadd(41.80, 749.39, 7.64, 2.76)}
    {pt3dadd(44.55, 755.57, 7.64, 2.76)}
    {pt3dadd(47.76, 758.86, 7.64, 2.76)}
    {pt3dadd(50.05, 761.74, 7.64, 2.76)}
    {pt3dadd(52.35, 770.79, 7.64, 2.76)}
    {pt3dadd(56.02, 777.79, 7.64, 2.76)}
    {pt3dadd(58.31, 786.84, 7.60, 2.76)}
    {pt3dadd(59.69, 793.02, 7.60, 2.76)}
    {pt3dadd(58.77, 796.31, 7.60, 2.76)}
    {pt3dadd(62.44, 802.07, 7.60, 2.76)}
    {pt3dadd(67.03, 806.18, 7.56, 2.76)}
    {pt3dadd(67.94, 812.77, 7.56, 2.76)}
    {pt3dadd(67.48, 821.82, 7.56, 2.76)}
    {pt3dadd(67.03, 826.76, 7.56, 2.76)}
    {pt3dadd(68.86, 834.58, 7.20, 2.76)}
    {pt3dadd(73.26, 837.69, 7.04, 2.76)}
    {pt3dadd(76.01, 841.80, 7.00, 2.76)}
    {pt3dadd(75.09, 845.92, 7.00, 1.84)}
    {pt3dadd(76.93, 850.03, 6.96, 1.84)}
    {pt3dadd(78.76, 852.91, 6.92, 1.84)}
    {pt3dadd(79.68, 856.62, 6.92, 1.84)}
    {pt3dadd(79.22, 860.32, 6.92, 1.84)}
    {pt3dadd(83.35, 863.20, 6.92, 1.84)}
    {pt3dadd(85.64, 865.26, 6.92, 1.84)}
    {pt3dadd(87.94, 867.32, 6.92, 1.84)}
    {pt3dadd(87.02, 873.08, 6.92, 1.84)}
    {pt3dadd(87.94, 878.02, 6.92, 1.84)}
    {pt3dadd(88.85, 881.31, 5.48, 1.84)}
    {pt3dadd(89.31, 884.19, 5.48, 1.84)}
    {pt3dadd(88.85, 887.48, 5.48, 1.84)}
    {pt3dadd(91.61, 891.60, 5.48, 1.84)}
    {pt3dadd(94.36, 894.07, 5.32, 1.84)}
    {pt3dadd(93.44, 899.83, 4.08, 1.84)}
    {pt3dadd(93.90, 905.18, 4.08, 1.84)}
    {pt3dadd(96.19, 907.23, 4.08, 1.84)}
    {pt3dadd(100.78, 911.35, 4.84, 1.84)}

    {create dendA1_000000000000100}
    {dendA1_00000000000010 connect dendA1_000000000000100(0), 1}
    {access dendA1_000000000000100}
    {nseg = 1}
    {pt3dclear()}
    {pt3dadd(100.78, 911.35, 4.84, 1.84)}
    {pt3dadd(102.86, 914.39, 5.48, 1.84)}
    {pt3dadd(102.86, 917.27, 3.48, 1.84)}

    {create dendA1_0000000000001000}
    {dendA1_000000000000100 connect dendA1_0000000000001000(0), 1}
    {access dendA1_0000000000001000}
    {nseg = 13}
    {pt3dclear()}
    {pt3dadd(102.86, 917.27, 3.48, 1.84)}
    {pt3dadd(100.57, 923.03, 3.20, 1.84)}
    {pt3dadd(100.57, 926.73, 2.32, 1.84)}
    {pt3dadd(102.41, 930.44, 1.88, 1.84)}
    {pt3dadd(104.24, 932.08, 1.60, 0.92)}
    {pt3dadd(101.49, 934.55, 1.48, 0.92)}
    {pt3dadd(98.28, 937.02, 1.48, 0.92)}
    {pt3dadd(96.90, 941.14, -0.24, 0.92)}
    {pt3dadd(97.82, 944.02, -0.64, 0.92)}
    {pt3dadd(97.36, 948.54, -2.36, 0.92)}
    {pt3dadd(96.90, 951.42, -3.00, 0.92)}
    {pt3dadd(96.44, 954.72, -4.76, 0.92)}
    {pt3dadd(96.90, 956.36, -6.36, 0.92)}
    {pt3dadd(94.15, 958.83, -6.36, 0.92)}
    {pt3dadd(93.69, 963.77, -7.76, 0.92)}
    {pt3dadd(94.15, 968.30, -7.76, 0.92)}
    {pt3dadd(92.77, 972.82, -7.88, 0.92)}
    {pt3dadd(90.48, 976.94, -8.44, 0.92)}
    {pt3dadd(88.64, 979.41, -9.32, 0.92)}
    {pt3dadd(87.73, 983.11, -6.16, 0.92)}
    {pt3dadd(87.27, 986.81, -6.28, 0.92)}
    {pt3dadd(85.24, 989.47, -5.48, 0.92)}
    {pt3dadd(83.40, 993.18, -4.48, 0.92)}
    {pt3dadd(82.94, 998.12, -6.16, 0.92)}
    {pt3dadd(82.48, 1002.64, -1.04, 0.92)}
    {pt3dadd(81.57, 1007.17, 0.64, 0.92)}
    {pt3dadd(80.65, 1009.64, 2.04, 0.92)}
    {pt3dadd(81.11, 1011.29, 3.60, 0.92)}
    {pt3dadd(77.90, 1012.52, 5.28, 0.92)}
    {pt3dadd(74.69, 1014.58, 5.28, 0.92)}
    {pt3dadd(71.47, 1016.22, 5.76, 0.92)}
    {pt3dadd(66.89, 1016.64, 6.80, 0.92)}
    {pt3dadd(65.05, 1014.17, 6.80, 0.92)}
    {pt3dadd(62.76, 1015.40, 6.08, 0.92)}

    {create dendA1_00000000000010000}
    {dendA1_0000000000001000 connect dendA1_00000000000010000(0), 1}
    {access dendA1_00000000000010000}
    {nseg = 10}
    {pt3dclear()}
    {pt3dadd(62.76, 1015.40, 6.08, 0.92)}
    {pt3dadd(59.55, 1015.40, 4.52, 0.92)}
    {pt3dadd(55.88, 1014.99, 5.92, 0.92)}
    {pt3dadd(52.67, 1014.58, 3.96, 0.92)}
    {pt3dadd(46.24, 1016.64, 3.60, 0.92)}
    {pt3dadd(43.95, 1017.46, 2.88, 0.92)}
    {pt3dadd(39.82, 1019.10, 3.56, 0.46)}
    {pt3dadd(34.78, 1019.93, 3.72, 0.46)}
    {pt3dadd(32.48, 1020.34, 4.08, 0.46)}
    {pt3dadd(31.11, 1019.10, 3.56, 0.46)}
    {pt3dadd(28.81, 1018.28, 3.20, 0.46)}
    {pt3dadd(27.90, 1019.93, 2.80, 0.46)}
    {pt3dadd(24.69, 1019.10, 2.80, 0.46)}
    {pt3dadd(20.56, 1021.57, 4.60, 0.46)}
    {pt3dadd(16.43, 1022.40, 7.00, 0.46)}
    {pt3dadd(13.68, 1021.99, 6.04, 0.46)}
    {pt3dadd(10.01, 1022.40, 7.32, 0.46)}
    {pt3dadd(5.42, 1022.81, 8.12, 0.46)}
    {pt3dadd(1.75, 1022.81, 9.68, 0.46)}
    {pt3dadd(-1.00, 1021.57, 10.52, 0.46)}
    {pt3dadd(-4.67, 1021.57, 10.56, 0.46)}
    {pt3dadd(-8.80, 1021.99, 9.28, 0.46)}

    {create dendA1_00000000000010001}
    {dendA1_0000000000001000 connect dendA1_00000000000010001(0), 1}
    {access dendA1_00000000000010001}
    {nseg = 5}
    {pt3dclear()}
    {pt3dadd(62.76, 1015.40, 6.08, 1.84)}
    {pt3dadd(59.55, 1015.81, 7.36, 1.84)}
    {pt3dadd(55.88, 1017.46, 7.24, 0.92)}
    {pt3dadd(53.58, 1016.64, 8.52, 0.92)}
    {pt3dadd(49.00, 1015.40, 10.00, 0.92)}
    {pt3dadd(42.58, 1014.58, 11.28, 0.92)}
    {pt3dadd(39.36, 1014.17, 11.40, 0.92)}
    {pt3dadd(38.45, 1015.81, 11.16, 0.92)}
    {pt3dadd(36.61, 1014.58, 12.24, 0.92)}
    {pt3dadd(34.32, 1014.99, 13.44, 0.92)}
    {pt3dadd(33.40, 1017.05, 13.76, 0.92)}
    {pt3dadd(31.57, 1016.22, 14.40, 0.92)}
    {pt3dadd(28.81, 1017.05, 13.36, 0.92)}
    {pt3dadd(27.90, 1014.17, 16.08, 0.92)}
    {pt3dadd(23.77, 1014.17, 17.40, 0.92)}
    {pt3dadd(23.31, 1012.52, 16.40, 0.92)}

    {create dendA1_0000000000001001}
    {dendA1_000000000000100 connect dendA1_0000000000001001(0), 1}
    {access dendA1_0000000000001001}
    {nseg = 25}
    {pt3dclear()}
    {pt3dadd(102.86, 917.27, 3.48, 2.76)}
    {pt3dadd(105.42, 917.46, 2.28, 2.76)}
    {pt3dadd(108.63, 921.16, 2.48, 1.84)}
    {pt3dadd(112.30, 921.16, 1.76, 0.92)}
    {pt3dadd(116.43, 921.99, 0.12, 0.92)}
    {pt3dadd(119.64, 924.04, 0.12, 0.92)}
    {pt3dadd(121.93, 924.45, 0.12, 0.92)}
    {pt3dadd(125.14, 924.87, 0.84, 0.92)}
    {pt3dadd(127.44, 925.28, 0.68, 0.92)}
    {pt3dadd(131.11, 923.63, 0.96, 0.92)}
    {pt3dadd(135.24, 925.69, 0.96, 0.92)}
    {pt3dadd(138.45, 926.92, -1.52, 0.92)}
    {pt3dadd(142.12, 926.10, -1.52, 0.92)}
    {pt3dadd(143.03, 925.28, -1.56, 0.92)}
    {pt3dadd(145.33, 927.75, -1.56, 0.92)}
    {pt3dadd(147.16, 928.16, -2.60, 0.92)}
    {pt3dadd(149.46, 927.75, -2.84, 0.92)}
    {pt3dadd(151.29, 927.75, -3.52, 0.92)}
    {pt3dadd(152.67, 928.98, -4.16, 0.92)}
    {pt3dadd(154.04, 928.57, -5.08, 0.92)}
    {pt3dadd(158.63, 927.75, -6.28, 0.92)}
    {pt3dadd(160.47, 929.39, -6.28, 0.92)}
    {pt3dadd(163.22, 931.04, -6.36, 0.92)}
    {pt3dadd(165.97, 929.39, -6.84, 0.92)}
    {pt3dadd(170.10, 931.45, -7.92, 0.92)}
    {pt3dadd(173.77, 932.27, -7.96, 0.92)}
    {pt3dadd(178.36, 932.27, -8.08, 0.92)}
    {pt3dadd(179.27, 931.86, -8.08, 0.92)}
    {pt3dadd(182.48, 931.45, -8.08, 0.92)}
    {pt3dadd(185.24, 933.92, -8.88, 0.92)}
    {pt3dadd(186.15, 936.80, -10.08, 0.92)}
    {pt3dadd(189.36, 936.39, -12.52, 0.92)}
    {pt3dadd(193.49, 936.80, -12.92, 0.92)}
    {pt3dadd(198.54, 938.03, -13.36, 0.92)}
    {pt3dadd(199.00, 942.15, -13.40, 0.92)}
    {pt3dadd(202.21, 943.80, -13.40, 0.92)}
    {pt3dadd(204.04, 944.21, -13.40, 0.92)}
    {pt3dadd(208.17, 945.85, -14.12, 0.92)}
    {pt3dadd(210.92, 947.50, -14.60, 0.92)}
    {pt3dadd(213.68, 948.32, -15.28, 0.92)}
    {pt3dadd(220.10, 950.38, -16.00, 0.92)}
    {pt3dadd(220.33, 953.49, -15.48, 0.92)}
    {pt3dadd(223.54, 955.54, -15.48, 0.92)}
    {pt3dadd(228.13, 958.42, -15.48, 0.92)}
    {pt3dadd(232.26, 959.66, -14.20, 0.92)}
    {pt3dadd(237.30, 961.72, -14.44, 0.46)}
    {pt3dadd(239.60, 964.19, -14.52, 0.46)}
    {pt3dadd(242.81, 963.36, -14.72, 0.46)}
    {pt3dadd(246.48, 965.83, -14.36, 0.92)}
    {pt3dadd(248.31, 967.89, -14.36, 0.92)}
    {pt3dadd(251.06, 968.30, -14.36, 0.92)}
    {pt3dadd(258.40, 972.01, -13.20, 0.92)}
    {pt3dadd(261.16, 974.89, -11.80, 0.92)}
    {pt3dadd(264.37, 974.06, -12.20, 0.92)}
    {pt3dadd(268.04, 978.18, -12.32, 0.92)}
    {pt3dadd(269.87, 981.47, -11.60, 0.92)}
    {pt3dadd(272.16, 981.47, -11.60, 0.92)}
    {pt3dadd(275.83, 984.76, -12.96, 0.92)}
    {pt3dadd(279.04, 988.47, -13.24, 0.92)}
    {pt3dadd(285.01, 991.35, -10.80, 0.92)}
    {pt3dadd(287.30, 992.99, -9.72, 0.92)}
    {pt3dadd(288.68, 995.46, -9.44, 0.92)}
    {pt3dadd(292.35, 995.05, -9.64, 0.92)}
    {pt3dadd(296.02, 998.75, -12.44, 0.92)}
    {pt3dadd(298.31, 999.17, -11.40, 0.46)}
    {pt3dadd(300.60, 995.05, -10.24, 0.46)}
    {pt3dadd(302.90, 993.82, -12.88, 0.46)}
    {pt3dadd(307.94, 996.70, -13.28, 0.46)}

    {create dendA1_000000000000101}
    {dendA1_00000000000010 connect dendA1_000000000000101(0), 1}
    {access dendA1_000000000000101}
    {nseg = 11}
    {pt3dclear()}
    {pt3dadd(100.78, 911.35, 4.84, 0.92)}
    {pt3dadd(104.27, 913.16, 7.48, 0.92)}
    {pt3dadd(107.49, 914.80, 7.56, 0.92)}
    {pt3dadd(109.78, 914.39, 9.88, 0.92)}
    {pt3dadd(113.45, 917.27, 10.40, 0.92)}
    {pt3dadd(116.20, 916.45, 11.60, 0.92)}
    {pt3dadd(121.71, 921.39, 11.56, 0.92)}
    {pt3dadd(124.46, 924.27, 13.88, 0.92)}
    {pt3dadd(126.29, 923.03, 14.16, 0.92)}
    {pt3dadd(129.50, 925.91, 14.56, 0.92)}
    {pt3dadd(131.80, 929.62, 15.08, 0.92)}
    {pt3dadd(134.55, 931.26, 15.40, 0.92)}
    {pt3dadd(136.38, 929.62, 16.56, 0.92)}
    {pt3dadd(137.30, 931.26, 17.08, 0.92)}
    {pt3dadd(137.30, 933.32, 17.36, 0.92)}
    {pt3dadd(140.97, 934.97, 17.76, 0.92)}
    {pt3dadd(142.81, 936.20, 17.80, 0.92)}
    {pt3dadd(146.02, 937.03, 17.88, 0.92)}
    {pt3dadd(149.23, 938.67, 18.64, 0.92)}
    {pt3dadd(151.52, 939.91, 19.56, 0.92)}
    {pt3dadd(152.44, 942.79, 20.28, 0.92)}
    {pt3dadd(150.15, 943.61, 20.40, 0.46)}
    {pt3dadd(148.31, 943.61, 20.44, 0.46)}
    {pt3dadd(145.56, 944.84, 20.80, 0.46)}
    {pt3dadd(141.89, 948.55, 21.32, 0.46)}
    {pt3dadd(139.14, 951.84, 20.80, 0.46)}
    {pt3dadd(136.38, 953.90, 20.80, 0.46)}
    {pt3dadd(134.09, 955.13, 20.84, 0.46)}
    {pt3dadd(132.71, 957.19, 20.88, 0.46)}
    {pt3dadd(134.09, 960.07, 20.92, 0.46)}

    {create dendA1_00000000000011}
    {dendA1_0000000000001 connect dendA1_00000000000011(0), 1}
    {access dendA1_00000000000011}
    {nseg = 2}
    {pt3dclear()}
    {pt3dadd(39.50, 745.28, 7.68, 3.66)}
    {pt3dadd(39.27, 749.62, 8.64, 3.66)}
    {pt3dadd(40.65, 756.21, 8.04, 3.66)}
    {pt3dadd(41.11, 761.56, 8.04, 3.66)}
    {pt3dadd(42.48, 765.26, 8.76, 2.76)}

    {create dendA1_000000000000110}
    {dendA1_00000000000011 connect dendA1_000000000000110(0), 1}
    {access dendA1_000000000000110}
    {nseg = 12}
    {pt3dclear()}
    {pt3dadd(42.48, 765.26, 8.76, 2.76)}
    {pt3dadd(43.40, 769.79, 8.64, 2.76)}
    {pt3dadd(43.86, 774.72, 7.40, 2.76)}
    {pt3dadd(43.40, 779.25, 7.44, 2.76)}
    {pt3dadd(43.40, 785.01, 7.24, 2.76)}
    {pt3dadd(45.23, 790.36, 7.16, 2.76)}
    {pt3dadd(43.86, 796.95, 7.00, 2.76)}
    {pt3dadd(42.48, 803.53, 6.84, 2.76)}
    {pt3dadd(42.94, 814.64, 6.76, 2.76)}
    {pt3dadd(44.32, 821.23, 6.12, 2.76)}
    {pt3dadd(46.82, 824.29, 3.64, 2.76)}
    {pt3dadd(48.66, 828.40, 2.96, 2.76)}
    {pt3dadd(47.74, 831.28, 0.92, 2.76)}
    {pt3dadd(44.07, 833.34, -0.24, 2.76)}
    {pt3dadd(45.91, 834.16, -3.12, 2.76)}
    {pt3dadd(49.58, 834.57, -3.84, 2.76)}
    {pt3dadd(53.24, 834.57, -4.88, 2.76)}
    {pt3dadd(55.54, 837.46, -5.68, 2.76)}
    {pt3dadd(53.70, 843.63, -6.60, 2.76)}
    {pt3dadd(56.00, 849.39, -8.32, 2.76)}
    {pt3dadd(58.75, 855.15, -9.52, 2.76)}
    {pt3dadd(57.37, 862.56, -7.68, 2.76)}
    {pt3dadd(57.37, 869.14, -7.68, 2.76)}
    {pt3dadd(58.75, 879.02, -8.36, 2.76)}
    {pt3dadd(60.58, 883.55, -10.48, 2.76)}
    {pt3dadd(60.58, 888.07, -12.12, 2.76)}
    {pt3dadd(56.91, 891.78, -14.56, 2.76)}
    {pt3dadd(54.16, 895.89, -15.76, 2.76)}
    {pt3dadd(51.22, 900.22, -14.76, 2.76)}
    {pt3dadd(48.92, 905.57, -16.12, 2.76)}
    {pt3dadd(48.47, 912.57, -14.68, 2.76)}
    {pt3dadd(46.63, 917.51, -15.84, 2.76)}
    {pt3dadd(48.01, 922.44, -10.60, 2.76)}
    {pt3dadd(52.14, 925.74, -9.68, 2.76)}
    {pt3dadd(53.05, 931.09, -7.68, 2.76)}
    {pt3dadd(54.43, 932.73, -7.48, 2.76)}

    {create dendA1_0000000000001100}
    {dendA1_000000000000110 connect dendA1_0000000000001100(0), 1}
    {access dendA1_0000000000001100}
    {nseg = 3}
    {pt3dclear()}
    {pt3dadd(54.43, 932.73, -7.48, 1.84)}
    {pt3dadd(54.89, 936.02, -7.24, 1.84)}
    {pt3dadd(53.51, 938.08, -6.92, 0.92)}
    {pt3dadd(51.22, 939.32, -8.16, 0.92)}
    {pt3dadd(53.05, 944.67, -10.84, 0.92)}
    {pt3dadd(53.51, 947.96, -11.16, 1.84)}
    {pt3dadd(55.35, 951.25, -11.60, 1.84)}
    {pt3dadd(58.56, 954.95, -13.64, 1.84)}
    {pt3dadd(58.10, 960.71, -14.84, 1.84)}
    {pt3dadd(55.81, 963.60, -15.40, 1.84)}

    {create dendA1_00000000000011000}
    {dendA1_0000000000001100 connect dendA1_00000000000011000(0), 1}
    {access dendA1_00000000000011000}
    {nseg = 2}
    {pt3dclear()}
    {pt3dadd(55.81, 963.60, -15.40, 1.84)}
    {pt3dadd(57.18, 966.89, -15.40, 1.84)}
    {pt3dadd(57.18, 971.83, -16.64, 1.84)}
    {pt3dadd(56.97, 976.12, -16.00, 1.84)}
    {pt3dadd(56.51, 982.71, -16.16, 1.84)}

    {create dendA1_000000000000110000}
    {dendA1_00000000000011000 connect dendA1_000000000000110000(0), 1}
    {access dendA1_000000000000110000}
    {nseg = 4}
    {pt3dclear()}
    {pt3dadd(56.51, 982.71, -16.16, 1.84)}
    {pt3dadd(57.43, 988.88, -16.16, 1.84)}
    {pt3dadd(58.81, 991.76, -16.16, 1.84)}
    {pt3dadd(60.64, 995.05, -16.16, 1.84)}
    {pt3dadd(59.72, 1000.40, -16.16, 1.84)}
    {pt3dadd(60.18, 1003.69, -16.16, 0.92)}
    {pt3dadd(62.93, 1009.04, -17.76, 0.92)}
    {pt3dadd(64.31, 1013.16, -18.32, 0.92)}
    {pt3dadd(65.23, 1017.68, -20.04, 0.92)}
    {pt3dadd(62.02, 1020.15, -22.68, 0.92)}

    {create dendA1_0000000000001100000}
    {dendA1_000000000000110000 connect dendA1_0000000000001100000(0), 1}
    {access dendA1_0000000000001100000}
    {nseg = 13}
    {pt3dclear()}
    {pt3dadd(62.02, 1020.15, -22.68, 0.92)}
    {pt3dadd(67.06, 1023.03, -22.68, 0.92)}
    {pt3dadd(68.44, 1026.33, -22.24, 0.92)}
    {pt3dadd(71.65, 1029.62, -22.96, 0.92)}
    {pt3dadd(74.86, 1029.21, -23.40, 0.92)}
    {pt3dadd(78.99, 1028.80, -23.04, 0.92)}
    {pt3dadd(85.41, 1029.62, -22.96, 0.92)}
    {pt3dadd(89.54, 1029.21, -22.72, 0.92)}
    {pt3dadd(93.67, 1029.21, -21.68, 0.92)}
    {pt3dadd(96.88, 1028.38, -21.00, 0.92)}
    {pt3dadd(101.47, 1026.74, -20.16, 0.92)}
    {pt3dadd(104.22, 1027.15, -25.68, 0.92)}
    {pt3dadd(106.97, 1026.33, -26.32, 0.92)}
    {pt3dadd(108.81, 1027.56, -26.40, 0.92)}
    {pt3dadd(110.64, 1027.15, -26.12, 0.92)}
    {pt3dadd(114.31, 1030.03, -26.84, 0.92)}
    {pt3dadd(116.14, 1030.44, -27.24, 0.92)}
    {pt3dadd(118.90, 1029.21, -27.36, 0.92)}
    {pt3dadd(121.65, 1030.85, -29.12, 0.92)}
    {pt3dadd(124.40, 1030.44, -30.16, 0.92)}
    {pt3dadd(127.61, 1030.85, -31.20, 0.92)}
    {pt3dadd(129.91, 1032.09, -31.92, 0.92)}
    {pt3dadd(131.74, 1030.85, -32.80, 0.92)}
    {pt3dadd(134.49, 1029.21, -33.36, 0.92)}
    {pt3dadd(137.70, 1029.21, -33.92, 0.92)}
    {pt3dadd(142.75, 1029.21, -34.04, 0.92)}
    {pt3dadd(146.42, 1028.38, -34.08, 0.92)}
    {pt3dadd(149.17, 1027.15, -34.12, 0.92)}
    {pt3dadd(151.92, 1029.21, -34.20, 0.92)}
    {pt3dadd(156.05, 1030.85, -35.04, 0.92)}
    {pt3dadd(160.64, 1030.85, -35.36, 0.92)}
    {pt3dadd(164.77, 1030.85, -36.32, 0.92)}
    {pt3dadd(166.60, 1032.09, -37.12, 0.92)}
    {pt3dadd(172.11, 1034.15, -38.24, 0.92)}
    {pt3dadd(177.15, 1036.61, -38.84, 0.92)}

    {create dendA1_0000000000001100001}
    {dendA1_000000000000110000 connect dendA1_0000000000001100001(0), 1}
    {access dendA1_0000000000001100001}
    {nseg = 9}
    {pt3dclear()}
    {pt3dadd(62.02, 1020.15, -22.68, 0.92)}
    {pt3dadd(65.23, 1024.27, -24.40, 0.92)}
    {pt3dadd(64.77, 1026.74, -24.64, 0.92)}
    {pt3dadd(61.56, 1027.15, -27.04, 0.92)}
    {pt3dadd(60.64, 1024.27, -32.40, 0.92)}
    {pt3dadd(59.72, 1027.15, -33.76, 0.92)}
    {pt3dadd(59.26, 1029.21, -34.68, 0.92)}
    {pt3dadd(56.97, 1026.74, -35.72, 0.92)}
    {pt3dadd(56.51, 1026.74, -37.16, 0.92)}
    {pt3dadd(57.43, 1031.27, -37.52, 0.92)}
    {pt3dadd(54.68, 1030.44, -38.84, 0.92)}
    {pt3dadd(52.84, 1028.80, -40.96, 0.92)}
    {pt3dadd(51.47, 1029.62, -41.56, 0.92)}
    {pt3dadd(51.01, 1031.68, -43.04, 0.92)}
    {pt3dadd(49.17, 1031.27, -44.12, 0.92)}
    {pt3dadd(47.34, 1033.32, -45.52, 0.92)}
    {pt3dadd(47.34, 1035.79, -49.12, 0.92)}
    {pt3dadd(46.88, 1037.85, -50.48, 0.92)}
    {pt3dadd(44.59, 1037.44, -51.92, 0.92)}
    {pt3dadd(41.83, 1040.32, -52.64, 0.92)}
    {pt3dadd(43.67, 1042.38, -53.92, 0.92)}
    {pt3dadd(44.13, 1046.08, -54.76, 0.92)}
    {pt3dadd(43.41, 1049.60, -56.80, 0.92)}
    {pt3dadd(41.58, 1050.83, -59.16, 0.92)}
    {pt3dadd(39.28, 1050.42, -60.08, 0.92)}
    {pt3dadd(38.82, 1054.12, -61.12, 0.92)}
    {pt3dadd(38.37, 1055.77, -63.56, 0.92)}
    {pt3dadd(36.99, 1056.18, -62.56, 0.92)}

    {create dendA1_000000000000110001}
    {dendA1_00000000000011000 connect dendA1_000000000000110001(0), 1}
    {access dendA1_000000000000110001}
    {nseg = 8}
    {pt3dclear()}
    {pt3dadd(56.51, 982.71, -16.16, 0.92)}
    {pt3dadd(53.96, 986.64, -12.92, 0.92)}
    {pt3dadd(53.96, 989.93, -12.88, 0.92)}
    {pt3dadd(53.96, 991.98, -12.00, 0.92)}
    {pt3dadd(54.88, 996.51, -13.64, 0.92)}
    {pt3dadd(53.04, 999.39, -13.84, 0.92)}
    {pt3dadd(53.96, 1004.33, -14.04, 0.92)}
    {pt3dadd(53.50, 1008.86, -14.32, 0.92)}
    {pt3dadd(54.88, 1010.91, -10.80, 0.92)}
    {pt3dadd(53.50, 1015.03, -9.76, 0.92)}
    {pt3dadd(54.88, 1018.73, -9.28, 0.92)}
    {pt3dadd(53.50, 1022.44, -7.32, 0.92)}
    {pt3dadd(54.88, 1026.14, -5.76, 0.92)}
    {pt3dadd(58.09, 1030.26, -6.64, 0.92)}
    {pt3dadd(60.84, 1031.49, -6.00, 0.92)}
    {pt3dadd(63.14, 1030.67, -6.00, 0.92)}
    {pt3dadd(65.43, 1034.37, -5.40, 0.92)}
    {pt3dadd(67.27, 1032.73, -5.12, 0.92)}
    {pt3dadd(72.31, 1035.19, -10.76, 0.92)}

    {create dendA1_00000000000011001}
    {dendA1_0000000000001100 connect dendA1_00000000000011001(0), 1}
    {access dendA1_00000000000011001}
    {nseg = 13}
    {pt3dclear()}
    {pt3dadd(55.81, 963.60, -15.40, 0.92)}
    {pt3dadd(51.90, 967.92, -15.00, 0.92)}
    {pt3dadd(48.23, 972.86, -15.00, 0.92)}
    {pt3dadd(48.69, 977.39, -16.16, 0.92)}
    {pt3dadd(46.86, 980.68, -13.12, 0.92)}
    {pt3dadd(43.64, 982.32, -12.12, 0.92)}
    {pt3dadd(40.89, 988.09, -11.08, 0.92)}
    {pt3dadd(39.97, 990.97, -10.08, 0.92)}
    {pt3dadd(37.68, 991.79, -8.00, 0.92)}
    {pt3dadd(36.30, 994.26, -6.68, 0.92)}
    {pt3dadd(34.47, 997.96, -5.64, 0.92)}
    {pt3dadd(32.63, 1002.08, -6.52, 0.92)}
    {pt3dadd(30.34, 1006.60, -2.24, 0.92)}
    {pt3dadd(28.97, 1010.72, -0.52, 0.92)}
    {pt3dadd(28.51, 1013.60, 1.64, 0.92)}
    {pt3dadd(23.46, 1012.78, 2.92, 0.92)}
    {pt3dadd(19.79, 1011.13, 3.32, 0.92)}
    {pt3dadd(17.04, 1012.78, 5.12, 0.92)}
    {pt3dadd(13.37, 1012.37, 5.72, 0.92)}
    {pt3dadd(10.62, 1014.01, 7.80, 0.92)}
    {pt3dadd(8.78, 1010.31, 9.32, 0.92)}
    {pt3dadd(6.95, 1009.48, 10.56, 0.92)}
    {pt3dadd(4.65, 1006.60, 10.96, 0.92)}
    {pt3dadd(3.74, 1009.48, 11.00, 0.92)}
    {pt3dadd(-0.85, 1009.48, 11.00, 0.92)}
    {pt3dadd(-3.60, 1007.84, 11.20, 0.92)}
    {pt3dadd(-7.27, 1007.02, 11.20, 0.46)}
    {pt3dadd(-9.57, 1007.02, 12.20, 0.46)}
    {pt3dadd(-14.15, 1010.31, 8.72, 0.46)}
    {pt3dadd(-13.70, 1014.83, 9.96, 0.46)}

    {create dendA1_0000000000001101}
    {dendA1_000000000000110 connect dendA1_0000000000001101(0), 1}
    {access dendA1_0000000000001101}
    {nseg = 1}
    {pt3dclear()}
    {pt3dadd(54.43, 932.73, -7.48, 2.76)}
    {pt3dadd(52.82, 938.70, -10.52, 2.76)}
    {pt3dadd(49.15, 942.41, -8.60, 2.76)}

    {create dendA1_00000000000011010}
    {dendA1_0000000000001101 connect dendA1_00000000000011010(0), 1}
    {access dendA1_00000000000011010}
    {nseg = 16}
    {pt3dclear()}
    {pt3dadd(49.15, 942.41, -8.60, 1.84)}
    {pt3dadd(48.69, 946.11, -6.68, 1.84)}
    {pt3dadd(47.31, 949.40, -6.68, 1.84)}
    {pt3dadd(46.40, 953.93, -6.52, 0.92)}
    {pt3dadd(45.02, 957.63, -6.52, 0.92)}
    {pt3dadd(43.19, 959.28, -6.52, 0.92)}
    {pt3dadd(39.97, 962.57, -6.52, 0.92)}
    {pt3dadd(39.52, 965.45, -6.20, 0.92)}
    {pt3dadd(38.60, 968.33, -5.08, 0.92)}
    {pt3dadd(36.76, 973.68, -5.12, 0.92)}
    {pt3dadd(34.01, 976.97, -5.12, 0.92)}
    {pt3dadd(26.67, 979.86, -5.12, 0.92)}
    {pt3dadd(22.54, 983.56, -5.12, 0.92)}
    {pt3dadd(19.33, 986.03, -4.72, 0.92)}
    {pt3dadd(16.58, 986.44, -8.60, 0.92)}
    {pt3dadd(11.53, 990.14, -8.60, 0.92)}
    {pt3dadd(8.32, 994.26, -9.00, 0.92)}
    {pt3dadd(6.49, 997.55, -5.96, 0.92)}
    {pt3dadd(2.82, 1001.25, -5.48, 0.92)}
    {pt3dadd(0.98, 1004.55, -3.04, 0.92)}
    {pt3dadd(-2.23, 1008.25, -1.88, 0.92)}
    {pt3dadd(-8.19, 1010.72, -3.68, 0.92)}
    {pt3dadd(-10.03, 1014.83, 0.40, 0.92)}
    {pt3dadd(-11.86, 1016.89, 1.64, 0.92)}
    {pt3dadd(-13.70, 1020.18, 3.00, 0.92)}
    {pt3dadd(-17.37, 1021.01, 4.00, 0.92)}
    {pt3dadd(-19.20, 1022.65, 4.04, 0.92)}
    {pt3dadd(-24.70, 1022.24, 4.12, 0.92)}
    {pt3dadd(-28.37, 1023.48, 3.60, 0.92)}
    {pt3dadd(-31.13, 1023.89, 3.20, 0.92)}
    {pt3dadd(-35.71, 1023.07, 2.72, 0.92)}
    {pt3dadd(-41.22, 1024.71, 1.84, 0.92)}
    {pt3dadd(-43.05, 1026.77, 1.52, 0.92)}
    {pt3dadd(-49.93, 1026.36, 1.52, 0.92)}
    {pt3dadd(-52.69, 1027.59, 0.88, 0.92)}
    {pt3dadd(-54.06, 1026.77, -1.24, 0.92)}
    {pt3dadd(-57.27, 1028.00, -1.68, 0.92)}
    {pt3dadd(-60.03, 1028.41, -2.56, 0.92)}
    {pt3dadd(-62.78, 1030.06, -4.96, 0.92)}
    {pt3dadd(-65.53, 1030.47, -4.96, 0.92)}

    {create dendA1_00000000000011011}
    {dendA1_0000000000001101 connect dendA1_00000000000011011(0), 1}
    {access dendA1_00000000000011011}
    {nseg = 8}
    {pt3dclear()}
    {pt3dadd(49.15, 942.41, -8.60, 1.84)}
    {pt3dadd(46.40, 944.05, -9.52, 1.84)}
    {pt3dadd(42.27, 946.11, -12.36, 1.84)}
    {pt3dadd(37.22, 949.40, -13.48, 0.92)}
    {pt3dadd(33.09, 951.87, -13.72, 0.92)}
    {pt3dadd(31.26, 953.11, -14.92, 0.92)}
    {pt3dadd(28.51, 955.16, -15.28, 0.92)}
    {pt3dadd(27.59, 958.87, -16.04, 0.92)}
    {pt3dadd(26.21, 962.16, -16.12, 0.92)}
    {pt3dadd(21.17, 961.75, -15.76, 0.92)}
    {pt3dadd(17.50, 964.22, -15.72, 0.92)}
    {pt3dadd(12.91, 967.51, -15.72, 0.92)}
    {pt3dadd(11.53, 971.62, -15.72, 0.92)}
    {pt3dadd(8.32, 973.27, -15.72, 0.92)}
    {pt3dadd(2.82, 972.86, -15.72, 0.92)}
    {pt3dadd(-2.69, 976.97, -16.68, 0.92)}
    {pt3dadd(-4.06, 977.80, -14.28, 0.92)}
    {pt3dadd(-6.36, 980.27, -13.36, 0.92)}
    {pt3dadd(-9.11, 981.50, -12.16, 0.92)}
    {pt3dadd(-12.78, 981.09, -11.28, 0.92)}

    {create dendA1_000000000000110110}
    {dendA1_00000000000011011 connect dendA1_000000000000110110(0), 1}
    {access dendA1_000000000000110110}
    {nseg = 16}
    {pt3dclear()}
    {pt3dadd(-12.78, 981.09, -11.28, 0.92)}
    {pt3dadd(-12.32, 984.79, -11.28, 0.92)}
    {pt3dadd(-15.99, 988.91, -11.24, 0.92)}
    {pt3dadd(-18.28, 988.50, -10.28, 0.92)}
    {pt3dadd(-22.87, 991.79, -9.84, 0.92)}
    {pt3dadd(-26.08, 997.55, -9.60, 0.92)}
    {pt3dadd(-29.75, 1001.67, -8.80, 0.92)}
    {pt3dadd(-32.50, 1002.08, -8.20, 0.92)}
    {pt3dadd(-37.09, 1003.72, -7.64, 0.92)}
    {pt3dadd(-41.68, 1008.25, -7.04, 0.92)}
    {pt3dadd(-43.51, 1009.90, -8.32, 0.92)}
    {pt3dadd(-48.10, 1010.31, -6.00, 0.92)}
    {pt3dadd(-52.69, 1014.01, -5.04, 0.46)}
    {pt3dadd(-56.36, 1015.25, -6.96, 0.46)}
    {pt3dadd(-63.24, 1017.72, -4.08, 0.46)}
    {pt3dadd(-67.82, 1018.54, -2.84, 0.46)}
    {pt3dadd(-68.74, 1019.77, -1.68, 0.46)}
    {pt3dadd(-69.66, 1023.89, -1.36, 0.46)}
    {pt3dadd(-73.79, 1027.18, 0.48, 0.46)}
    {pt3dadd(-72.65, 1030.70, 1.44, 0.46)}
    {pt3dadd(-74.95, 1031.93, 1.44, 0.46)}
    {pt3dadd(-76.32, 1031.11, -2.04, 0.46)}
    {pt3dadd(-79.08, 1032.34, -2.80, 0.46)}
    {pt3dadd(-82.74, 1033.58, -3.00, 0.46)}
    {pt3dadd(-85.04, 1032.76, -2.56, 0.46)}
    {pt3dadd(-87.79, 1036.05, -4.76, 0.46)}
    {pt3dadd(-90.08, 1037.28, -6.32, 0.46)}
    {pt3dadd(-94.21, 1036.46, -8.48, 0.46)}
    {pt3dadd(-97.42, 1035.22, -9.04, 0.46)}
    {pt3dadd(-102.01, 1036.87, -9.72, 0.46)}
    {pt3dadd(-102.93, 1035.22, -8.44, 0.46)}
    {pt3dadd(-103.39, 1035.22, -5.64, 0.46)}

    {create dendA1_000000000000110111}
    {dendA1_00000000000011011 connect dendA1_000000000000110111(0), 1}
    {access dendA1_000000000000110111}
    {nseg = 11}
    {pt3dclear()}
    {pt3dadd(-12.78, 981.09, -11.28, 0.92)}
    {pt3dadd(-14.40, 978.85, -11.12, 0.92)}
    {pt3dadd(-16.69, 979.67, -11.12, 0.92)}
    {pt3dadd(-19.90, 978.02, -8.96, 0.92)}
    {pt3dadd(-22.65, 978.85, -8.44, 0.92)}
    {pt3dadd(-24.49, 980.49, -8.64, 0.92)}
    {pt3dadd(-26.32, 981.73, -8.80, 0.92)}
    {pt3dadd(-29.99, 982.55, -8.84, 0.92)}
    {pt3dadd(-31.83, 980.49, -9.92, 0.92)}
    {pt3dadd(-34.12, 980.49, -10.60, 0.92)}
    {pt3dadd(-37.79, 978.85, -9.88, 0.92)}
    {pt3dadd(-41.00, 978.02, -6.60, 0.92)}
    {pt3dadd(-43.30, 979.26, -5.04, 0.92)}
    {pt3dadd(-45.13, 981.32, -3.64, 0.92)}
    {pt3dadd(-47.42, 981.32, -3.20, 0.92)}
    {pt3dadd(-52.01, 984.61, -3.68, 0.46)}
    {pt3dadd(-53.85, 982.96, -3.04, 0.46)}
    {pt3dadd(-58.43, 987.08, -1.28, 0.46)}
    {pt3dadd(-61.19, 987.90, -0.36, 0.46)}
    {pt3dadd(-62.56, 990.78, 0.00, 0.46)}
    {pt3dadd(-64.40, 991.19, 3.16, 0.46)}
    {pt3dadd(-68.52, 992.43, 4.72, 0.46)}
    {pt3dadd(-71.28, 995.31, 5.36, 0.46)}
    {pt3dadd(-72.19, 998.19, 5.88, 0.46)}
    {pt3dadd(-72.19, 1000.25, 9.44, 0.46)}
    {pt3dadd(-71.28, 1001.89, 12.20, 0.46)}
    {pt3dadd(-71.28, 1005.18, 16.40, 0.46)}
    {pt3dadd(-68.52, 1006.01, 16.88, 0.46)}

    {create dendA1_000000000000111}
    {dendA1_00000000000011 connect dendA1_000000000000111(0), 1}
    {access dendA1_000000000000111}
    {nseg = 11}
    {pt3dclear()}
    {pt3dadd(42.48, 765.26, 8.76, 1.84)}
    {pt3dadd(41.35, 770.86, 9.48, 1.84)}
    {pt3dadd(39.06, 774.15, 10.52, 1.84)}
    {pt3dadd(39.06, 778.26, 11.08, 1.84)}
    {pt3dadd(37.68, 782.79, 11.08, 1.84)}
    {pt3dadd(38.14, 786.91, 11.08, 1.84)}
    {pt3dadd(37.22, 791.84, 11.08, 1.84)}
    {pt3dadd(33.10, 796.37, 11.52, 1.84)}
    {pt3dadd(31.72, 798.02, 12.72, 1.84)}
    {pt3dadd(33.56, 803.37, 13.00, 1.84)}
    {pt3dadd(32.64, 807.48, 13.24, 1.84)}
    {pt3dadd(30.34, 811.18, 13.24, 1.84)}
    {pt3dadd(29.89, 816.12, 13.36, 1.84)}
    {pt3dadd(29.89, 821.47, 13.32, 1.84)}
    {pt3dadd(28.05, 826.41, 13.44, 1.84)}
    {pt3dadd(25.76, 828.88, 13.68, 1.84)}
    {pt3dadd(24.38, 831.76, 13.68, 0.92)}
    {pt3dadd(21.63, 834.23, 13.68, 0.92)}
    {pt3dadd(21.63, 837.93, 13.68, 0.92)}
    {pt3dadd(19.79, 841.23, 13.64, 0.92)}
    {pt3dadd(18.42, 844.93, 13.56, 0.92)}
    {pt3dadd(16.79, 849.22, 15.44, 0.92)}
    {pt3dadd(15.42, 852.51, 15.56, 0.92)}
    {pt3dadd(13.12, 854.16, 15.80, 0.92)}
    {pt3dadd(7.62, 857.86, 13.08, 0.92)}
    {pt3dadd(5.79, 862.80, 13.08, 0.92)}
    {pt3dadd(3.49, 864.04, 12.28, 0.92)}
    {pt3dadd(0.74, 866.92, 15.52, 0.92)}
    {pt3dadd(-0.18, 869.39, 16.72, 0.92)}
    {pt3dadd(-0.18, 875.15, 17.68, 0.92)}
    {pt3dadd(-0.64, 878.85, 17.84, 0.92)}
    {pt3dadd(-2.01, 881.73, 18.08, 0.92)}

    {create dendA1_000000000001}
    {dendA1_00000000000 connect dendA1_000000000001(0), 1}
    {access dendA1_000000000001}
    {nseg = 3}
    {pt3dclear()}
    {pt3dadd(11.90, 369.94, 8.12, 1.84)}
    {pt3dadd(15.80, 370.52, 7.80, 1.84)}
    {pt3dadd(19.93, 369.28, 8.00, 1.84)}
    {pt3dadd(19.47, 372.99, 10.08, 1.84)}
    {pt3dadd(19.01, 375.87, 10.92, 1.84)}
    {pt3dadd(21.76, 377.92, 15.28, 1.84)}
    {pt3dadd(23.14, 380.39, 15.28, 0.92)}
    {pt3dadd(21.30, 383.69, 18.16, 0.92)}

    {create dendA1_00000000001}
    {dendA1_0000000000 connect dendA1_00000000001(0), 1}
    {access dendA1_00000000001}
    {nseg = 4}
    {pt3dclear()}
    {pt3dadd(9.15, 360.47, 5.80, 1.84)}
    {pt3dadd(3.87, 360.64, 2.92, 1.84)}
    {pt3dadd(2.04, 365.17, 2.88, 1.84)}
    {pt3dadd(1.12, 368.87, 2.88, 1.84)}
    {pt3dadd(-1.63, 370.93, 4.12, 1.84)}
    {pt3dadd(-7.14, 372.99, 1.68, 1.84)}
    {pt3dadd(-10.35, 375.46, 1.36, 1.84)}
    {pt3dadd(-17.23, 377.92, -0.40, 1.84)}
    {pt3dadd(-20.44, 382.86, -1.92, 1.84)}
    {pt3dadd(-22.27, 386.98, -2.68, 1.84)}

    {create dendA1_000000000010}
    {dendA1_00000000001 connect dendA1_000000000010(0), 1}
    {access dendA1_000000000010}
    {nseg = 29}
    {pt3dclear()}
    {pt3dadd(-22.27, 386.98, -2.68, 1.84)}
    {pt3dadd(-19.52, 391.09, -3.96, 1.84)}
    {pt3dadd(-22.73, 393.56, -4.60, 1.84)}
    {pt3dadd(-27.78, 396.03, -5.32, 1.84)}
    {pt3dadd(-24.57, 398.50, -7.76, 1.84)}
    {pt3dadd(-27.32, 400.97, -9.00, 1.84)}
    {pt3dadd(-30.53, 400.97, -11.40, 0.92)}
    {pt3dadd(-32.82, 402.62, -12.48, 0.92)}
    {pt3dadd(-35.12, 405.50, -12.60, 0.92)}
    {pt3dadd(-35.12, 410.43, -14.64, 0.92)}
    {pt3dadd(-35.58, 414.14, -15.60, 0.92)}
    {pt3dadd(-37.41, 416.61, -16.80, 0.92)}
    {pt3dadd(-41.54, 417.84, -17.48, 0.92)}
    {pt3dadd(-44.75, 419.90, -17.48, 0.92)}
    {pt3dadd(-45.21, 423.60, -17.48, 0.92)}
    {pt3dadd(-47.50, 426.90, -19.32, 0.92)}
    {pt3dadd(-50.26, 426.90, -20.60, 0.92)}
    {pt3dadd(-53.01, 428.13, -21.20, 0.92)}
    {pt3dadd(-55.30, 431.42, -22.20, 0.92)}
    {pt3dadd(-56.68, 433.89, -22.92, 0.92)}
    {pt3dadd(-61.72, 438.01, -23.16, 0.92)}
    {pt3dadd(-64.93, 441.30, -23.32, 0.92)}
    {pt3dadd(-66.05, 445.21, -23.44, 0.92)}
    {pt3dadd(-66.96, 448.09, -23.48, 0.92)}
    {pt3dadd(-72.01, 449.74, -23.48, 0.92)}
    {pt3dadd(-76.14, 451.38, -23.68, 0.92)}
    {pt3dadd(-77.06, 456.73, -25.40, 0.92)}
    {pt3dadd(-78.89, 459.61, -27.08, 0.92)}
    {pt3dadd(-80.73, 462.90, -27.36, 0.92)}
    {pt3dadd(-82.56, 465.37, -28.24, 0.92)}
    {pt3dadd(-85.77, 471.14, -28.28, 0.92)}
    {pt3dadd(-88.07, 477.72, -28.44, 0.92)}
    {pt3dadd(-88.07, 485.95, -28.44, 0.92)}
    {pt3dadd(-88.98, 491.30, -28.44, 0.92)}
    {pt3dadd(-88.98, 498.30, -28.44, 0.92)}
    {pt3dadd(-89.44, 504.06, -28.44, 0.92)}
    {pt3dadd(-92.65, 509.41, -28.56, 0.92)}
    {pt3dadd(-94.03, 514.76, -31.08, 0.92)}
    {pt3dadd(-94.49, 517.23, -31.12, 0.92)}
    {pt3dadd(-95.65, 520.29, -30.00, 0.92)}
    {pt3dadd(-95.65, 525.22, -28.96, 0.92)}
    {pt3dadd(-99.32, 529.75, -29.04, 0.92)}
    {pt3dadd(-103.90, 533.46, -28.48, 0.92)}
    {pt3dadd(-106.66, 536.75, -27.84, 0.92)}
    {pt3dadd(-108.95, 542.92, -27.00, 0.92)}
    {pt3dadd(-111.24, 547.04, -26.64, 0.92)}
    {pt3dadd(-113.08, 550.74, -25.68, 0.92)}
    {pt3dadd(-114.91, 554.85, -24.88, 0.92)}
    {pt3dadd(-115.37, 558.56, -24.40, 0.92)}
    {pt3dadd(-117.67, 563.50, -26.48, 0.92)}
    {pt3dadd(-119.50, 568.02, -26.56, 0.92)}
    {pt3dadd(-121.79, 572.96, -24.68, 0.92)}
    {pt3dadd(-125.00, 577.08, -24.32, 0.92)}
    {pt3dadd(-129.13, 579.55, -23.92, 0.92)}
    {pt3dadd(-133.72, 582.43, -23.16, 0.92)}
    {pt3dadd(-136.93, 585.31, -24.20, 0.92)}
    {pt3dadd(-140.60, 586.13, -27.36, 0.92)}
    {pt3dadd(-142.89, 585.72, -28.20, 0.92)}
    {pt3dadd(-146.11, 584.07, -28.28, 0.92)}
    {pt3dadd(-151.15, 586.54, -27.64, 0.92)}
    {pt3dadd(-155.28, 586.95, -26.08, 0.92)}
    {pt3dadd(-158.03, 587.36, -25.04, 0.92)}
    {pt3dadd(-164.45, 589.42, -24.88, 0.92)}
    {pt3dadd(-168.58, 590.66, -24.64, 0.92)}
    {pt3dadd(-171.33, 592.71, -24.64, 0.92)}
    {pt3dadd(-174.28, 596.61, -26.36, 0.92)}
    {pt3dadd(-176.58, 601.14, -27.16, 0.46)}

    {create dendA1_000000000011}
    {dendA1_00000000001 connect dendA1_000000000011(0), 1}
    {access dendA1_000000000011}
    {nseg = 8}
    {pt3dclear()}
    {pt3dadd(-22.27, 386.98, -2.68, 0.92)}
    {pt3dadd(-22.99, 385.15, -0.04, 0.92)}
    {pt3dadd(-25.28, 384.74, -0.04, 0.92)}
    {pt3dadd(-28.49, 383.09, 0.00, 0.92)}
    {pt3dadd(-31.24, 382.27, 3.16, 0.92)}
    {pt3dadd(-34.00, 380.62, 4.00, 0.92)}
    {pt3dadd(-35.83, 381.03, 4.48, 0.92)}
    {pt3dadd(-40.42, 378.98, 4.80, 0.92)}
    {pt3dadd(-41.80, 373.63, 5.12, 0.92)}
    {pt3dadd(-45.47, 372.80, 5.52, 0.92)}
    {pt3dadd(-49.59, 369.92, 6.12, 0.92)}
    {pt3dadd(-51.89, 369.10, 6.96, 0.92)}
    {pt3dadd(-56.47, 364.57, 8.72, 0.92)}
    {pt3dadd(-59.23, 362.93, 6.72, 0.92)}
    {pt3dadd(-61.52, 364.98, 6.40, 0.92)}
    {pt3dadd(-64.73, 367.04, 8.92, 0.92)}
    {pt3dadd(-69.78, 370.74, 10.48, 0.92)}
    {pt3dadd(-71.15, 371.98, 12.40, 0.92)}
    {pt3dadd(-74.36, 372.80, 12.84, 0.92)}
    {pt3dadd(-73.91, 369.92, 12.96, 0.92)}
    {pt3dadd(-76.66, 366.63, 11.08, 0.92)}

    {create dendA1_0000000001}
    {dendA1_000000000 connect dendA1_0000000001(0), 1}
    {access dendA1_0000000001}
    {nseg = 2}
    {pt3dclear()}
    {pt3dadd(0.71, 256.99, 3.64, 3.66)}
    {pt3dadd(-4.58, 258.03, 0.52, 3.66)}
    {pt3dadd(-10.54, 260.50, 1.08, 3.66)}
    {pt3dadd(-16.96, 261.32, 2.08, 3.66)}
    {pt3dadd(-21.55, 260.09, 2.44, 3.66)}
    {pt3dadd(-27.51, 257.62, 2.60, 3.66)}
    {pt3dadd(-30.26, 255.97, -2.76, 3.66)}
    {pt3dadd(-33.93, 257.62, -4.44, 3.66)}

    {create dendA1_00000000010}
    {dendA1_0000000001 connect dendA1_00000000010(0), 1}
    {access dendA1_00000000010}
    {nseg = 2}
    {pt3dclear()}
    {pt3dadd(-33.93, 257.62, -4.44, 3.66)}
    {pt3dadd(-31.64, 259.26, -7.32, 3.66)}
    {pt3dadd(-32.10, 262.56, -6.96, 3.66)}
    {pt3dadd(-40.36, 264.61, -7.52, 2.76)}
    {pt3dadd(-43.11, 263.79, -13.32, 2.76)}
    {pt3dadd(-44.02, 263.79, -13.44, 1.84)}

    {create dendA1_000000000100}
    {dendA1_00000000010 connect dendA1_000000000100(0), 1}
    {access dendA1_000000000100}
    {nseg = 3}
    {pt3dclear()}
    {pt3dadd(-44.02, 263.79, -13.44, 1.84)}
    {pt3dadd(-43.57, 268.73, -15.72, 1.84)}
    {pt3dadd(-41.27, 270.79, -22.72, 1.84)}
    {pt3dadd(-40.81, 272.43, -24.64, 1.84)}
    {pt3dadd(-40.81, 274.90, -26.16, 1.84)}
    {pt3dadd(-38.98, 276.96, -26.80, 1.84)}
    {pt3dadd(-35.77, 276.96, -27.04, 1.84)}
    {pt3dadd(-33.02, 279.43, -27.04, 1.84)}
    {pt3dadd(-30.26, 277.78, -27.08, 0.92)}
    {pt3dadd(-27.05, 279.02, -28.20, 0.92)}

    {create dendA1_000000000101}
    {dendA1_00000000010 connect dendA1_000000000101(0), 1}
    {access dendA1_000000000101}
    {nseg = 11}
    {pt3dclear()}
    {pt3dadd(-44.02, 263.79, -13.44, 1.84)}
    {pt3dadd(-44.48, 261.32, -13.12, 1.84)}
    {pt3dadd(-43.57, 258.44, -13.96, 1.84)}
    {pt3dadd(-41.27, 255.56, -15.36, 1.84)}
    {pt3dadd(-40.81, 251.03, -17.00, 1.84)}
    {pt3dadd(-40.81, 245.68, -17.72, 1.84)}
    {pt3dadd(-40.36, 242.80, -19.08, 1.84)}
    {pt3dadd(-40.81, 240.74, -20.12, 1.84)}
    {pt3dadd(-42.65, 237.86, -20.72, 1.84)}
    {pt3dadd(-44.02, 236.22, -21.96, 1.84)}
    {pt3dadd(-43.11, 231.69, -22.16, 1.84)}
    {pt3dadd(-40.81, 228.81, -22.36, 1.84)}
    {pt3dadd(-42.19, 225.11, -25.12, 1.84)}
    {pt3dadd(-44.94, 224.28, -25.24, 1.84)}
    {pt3dadd(-45.86, 222.23, -26.24, 1.84)}
    {pt3dadd(-49.07, 219.35, -26.32, 1.84)}
    {pt3dadd(-48.15, 215.64, -26.32, 0.92)}
    {pt3dadd(-47.24, 208.23, -27.12, 0.92)}
    {pt3dadd(-44.02, 204.53, -29.04, 0.92)}
    {pt3dadd(-43.57, 202.47, -30.68, 0.92)}
    {pt3dadd(-39.90, 197.95, -31.84, 0.92)}
    {pt3dadd(-38.98, 194.65, -32.96, 0.92)}
    {pt3dadd(-37.14, 189.30, -33.00, 0.92)}
    {pt3dadd(-36.41, 186.21, -37.16, 0.92)}
    {pt3dadd(-35.49, 183.33, -38.72, 0.92)}
    {pt3dadd(-34.58, 180.45, -39.84, 0.92)}
    {pt3dadd(-33.66, 175.10, -40.64, 0.92)}
    {pt3dadd(-33.66, 171.40, -43.28, 0.92)}
    {pt3dadd(-35.04, 169.34, -44.04, 0.92)}
    {pt3dadd(-33.20, 164.40, -41.04, 0.92)}
    {pt3dadd(-30.45, 161.93, -42.68, 0.92)}
    {pt3dadd(-29.99, 156.17, -44.88, 0.92)}
    {pt3dadd(-28.61, 152.88, -42.88, 0.92)}

    {create dendA1_00000000011}
    {dendA1_0000000001 connect dendA1_00000000011(0), 1}
    {access dendA1_00000000011}
    {nseg = 2}
    {pt3dclear()}
    {pt3dadd(-33.93, 257.62, -4.44, 2.76)}
    {pt3dadd(-37.33, 259.05, -2.08, 2.76)}
    {pt3dadd(-41.92, 259.46, -3.36, 2.76)}
    {pt3dadd(-47.42, 261.11, -4.36, 2.76)}
    {pt3dadd(-51.09, 263.17, -2.72, 2.76)}
    {pt3dadd(-56.38, 265.05, -1.12, 1.84)}
    {pt3dadd(-60.50, 266.70, -1.16, 1.84)}
    {pt3dadd(-63.26, 268.34, -0.12, 1.84)}

    {create dendA1_000000000110}
    {dendA1_00000000011 connect dendA1_000000000110(0), 1}
    {access dendA1_000000000110}
    {nseg = 9}
    {pt3dclear()}
    {pt3dadd(-63.26, 268.34, -0.12, 1.84)}
    {pt3dadd(-65.09, 273.28, -0.12, 1.84)}
    {pt3dadd(-66.47, 275.34, 1.12, 1.84)}
    {pt3dadd(-69.68, 276.99, 2.72, 1.84)}
    {pt3dadd(-76.10, 280.28, 3.68, 0.92)}
    {pt3dadd(-77.02, 285.22, 3.68, 0.92)}
    {pt3dadd(-78.85, 289.33, 3.68, 0.92)}
    {pt3dadd(-80.23, 292.21, 3.68, 0.92)}
    {pt3dadd(-82.06, 295.92, 4.76, 0.92)}
    {pt3dadd(-82.52, 301.68, 4.84, 0.92)}
    {pt3dadd(-83.44, 307.44, 5.12, 0.92)}
    {pt3dadd(-83.90, 314.02, 5.60, 0.92)}
    {pt3dadd(-83.44, 321.43, 3.64, 0.92)}
    {pt3dadd(-85.27, 325.13, 3.64, 0.92)}
    {pt3dadd(-83.90, 329.66, 3.52, 0.92)}
    {pt3dadd(-84.82, 336.25, 5.40, 0.92)}
    {pt3dadd(-85.27, 338.30, 5.40, 0.92)}
    {pt3dadd(-85.99, 342.21, 7.72, 0.92)}
    {pt3dadd(-87.82, 349.21, 7.72, 0.92)}
    {pt3dadd(-89.20, 352.91, 8.36, 0.92)}

    {create dendA1_000000000111}
    {dendA1_00000000011 connect dendA1_000000000111(0), 1}
    {access dendA1_000000000111}
    {nseg = 18}
    {pt3dclear()}
    {pt3dadd(-63.26, 268.34, -0.12, 1.84)}
    {pt3dadd(-68.10, 267.72, -2.32, 1.84)}
    {pt3dadd(-74.29, 266.67, -1.84, 1.84)}
    {pt3dadd(-79.34, 265.44, -0.52, 1.84)}
    {pt3dadd(-84.84, 265.03, -1.68, 1.84)}
    {pt3dadd(-90.35, 264.20, -1.36, 1.84)}
    {pt3dadd(-95.39, 261.73, -2.32, 1.84)}
    {pt3dadd(-103.19, 260.50, -2.56, 0.92)}
    {pt3dadd(-108.24, 259.26, -2.80, 0.92)}
    {pt3dadd(-114.66, 260.09, -2.84, 0.92)}
    {pt3dadd(-122.00, 259.68, -0.52, 0.92)}
    {pt3dadd(-125.21, 259.68, 0.64, 0.92)}
    {pt3dadd(-132.09, 260.09, 1.32, 0.92)}
    {pt3dadd(-134.38, 258.03, 1.68, 0.92)}
    {pt3dadd(-139.43, 257.21, 3.28, 0.92)}
    {pt3dadd(-140.81, 256.79, 3.84, 0.92)}
    {pt3dadd(-143.56, 260.50, 3.96, 0.92)}
    {pt3dadd(-146.77, 260.09, 4.44, 0.92)}
    {pt3dadd(-150.44, 257.62, 5.44, 0.92)}
    {pt3dadd(-153.19, 256.79, 6.04, 0.92)}
    {pt3dadd(-156.86, 256.38, 6.92, 0.92)}
    {pt3dadd(-160.53, 254.74, 7.88, 0.92)}
    {pt3dadd(-165.58, 251.86, 8.64, 0.92)}
    {pt3dadd(-168.33, 250.62, 9.12, 0.92)}
    {pt3dadd(-174.29, 253.50, 9.84, 0.92)}
    {pt3dadd(-178.42, 255.97, 10.24, 0.92)}
    {pt3dadd(-184.84, 260.50, 10.24, 0.92)}
    {pt3dadd(-188.51, 262.14, 10.64, 0.92)}
    {pt3dadd(-194.93, 264.61, 10.96, 0.92)}
    {pt3dadd(-202.27, 265.44, 11.04, 0.92)}
    {pt3dadd(-205.28, 268.90, 10.68, 0.92)}
    {pt3dadd(-214.00, 270.13, 11.08, 0.92)}
    {pt3dadd(-216.29, 271.37, 12.48, 0.92)}
    {pt3dadd(-221.34, 273.42, 11.80, 0.92)}
    {pt3dadd(-226.38, 275.48, 8.92, 0.92)}
    {pt3dadd(-232.35, 277.13, 7.44, 0.92)}
    {pt3dadd(-237.85, 277.13, 6.56, 0.92)}
    {pt3dadd(-242.44, 278.77, 5.88, 0.92)}
    {pt3dadd(-245.65, 282.07, 5.88, 0.92)}

    {create dendA1_000000001}
    {dendA1_00000000 connect dendA1_000000001(0), 1}
    {access dendA1_000000001}
    {nseg = 9}
    {pt3dclear()}
    {pt3dadd(0.71, 241.76, 3.76, 1.84)}
    {pt3dadd(-5.03, 244.47, -7.28, 1.84)}
    {pt3dadd(-10.08, 247.77, -7.92, 1.84)}
    {pt3dadd(-14.21, 249.82, -9.04, 1.84)}
    {pt3dadd(-20.17, 252.71, -9.64, 0.92)}
    {pt3dadd(-22.47, 256.41, -10.92, 0.92)}
    {pt3dadd(-23.84, 259.70, -11.08, 0.92)}
    {pt3dadd(-26.59, 262.58, -11.00, 0.92)}
    {pt3dadd(-31.64, 265.46, -9.64, 0.92)}
    {pt3dadd(-35.31, 267.11, -9.32, 0.92)}
    {pt3dadd(-37.60, 269.99, -10.12, 0.92)}
    {pt3dadd(-38.98, 274.52, -11.44, 0.92)}
    {pt3dadd(-39.90, 278.22, -11.44, 0.92)}
    {pt3dadd(-41.27, 282.75, -11.52, 0.92)}
    {pt3dadd(-44.48, 286.86, -11.28, 0.92)}
    {pt3dadd(-45.86, 292.62, -10.84, 0.92)}
    {pt3dadd(-45.86, 295.92, -9.24, 0.92)}
    {pt3dadd(-46.32, 298.38, -9.00, 0.92)}
    {pt3dadd(-49.07, 302.09, -10.56, 0.92)}

    {create dendA1_00000001}
    {dendA1_0000000 connect dendA1_00000001(0), 1}
    {access dendA1_00000001}
    {nseg = 11}
    {pt3dclear()}
    {pt3dadd(-0.85, 216.87, 1.92, 2.76)}
    {pt3dadd(1.85, 220.20, -4.20, 2.76)}
    {pt3dadd(3.68, 221.84, -4.28, 2.76)}
    {pt3dadd(7.35, 220.20, -5.08, 2.76)}
    {pt3dadd(5.98, 223.49, -9.56, 2.76)}
    {pt3dadd(3.68, 228.01, -9.92, 2.76)}
    {pt3dadd(5.98, 230.07, -10.12, 2.76)}
    {pt3dadd(10.10, 225.13, -10.92, 2.76)}
    {pt3dadd(14.69, 222.66, -13.32, 2.76)}
    {pt3dadd(16.53, 224.31, -18.40, 2.76)}
    {pt3dadd(12.86, 226.78, -21.56, 2.76)}
    {pt3dadd(10.56, 229.66, -22.80, 2.76)}
    {pt3dadd(12.86, 231.31, -23.20, 2.76)}
    {pt3dadd(17.44, 227.60, -24.76, 2.76)}
    {pt3dadd(22.95, 222.66, -28.08, 2.76)}
    {pt3dadd(25.24, 218.55, -28.52, 2.76)}
    {pt3dadd(29.83, 214.43, -29.24, 1.84)}
    {pt3dadd(33.50, 211.96, -29.68, 1.84)}
    {pt3dadd(37.17, 214.43, -30.08, 1.84)}
    {pt3dadd(42.67, 215.26, -30.88, 0.92)}
    {pt3dadd(45.42, 217.73, -31.92, 0.92)}
    {pt3dadd(50.47, 218.55, -31.24, 0.92)}
    {pt3dadd(54.60, 220.20, -31.20, 0.92)}
    {pt3dadd(59.19, 221.02, -33.92, 0.92)}
    {pt3dadd(61.94, 221.02, -35.28, 0.92)}
    {pt3dadd(66.07, 219.37, -35.92, 0.92)}
    {pt3dadd(66.98, 216.49, -36.52, 0.92)}
    {pt3dadd(69.28, 211.14, -36.72, 0.92)}
    {pt3dadd(69.28, 206.61, -38.20, 0.92)}
    {pt3dadd(68.36, 204.56, -40.24, 0.92)}
    {pt3dadd(72.03, 197.56, -36.44, 0.92)}
    {pt3dadd(75.24, 195.09, -34.92, 0.92)}
    {pt3dadd(78.91, 192.21, -36.92, 0.92)}

    {create dendA1_0000001}
    {dendA1_000000 connect dendA1_0000001(0), 1}
    {access dendA1_0000001}
    {nseg = 3}
    {pt3dclear()}
    {pt3dadd(-3.15, 207.40, 8.56, 3.66)}
    {pt3dadd(2.31, 213.61, 1.56, 3.66)}
    {pt3dadd(3.22, 216.49, -5.84, 3.66)}
    {pt3dadd(4.60, 220.61, -12.76, 3.66)}
    {pt3dadd(6.43, 223.08, -14.20, 3.66)}
    {pt3dadd(12.40, 221.43, -16.92, 3.66)}
    {pt3dadd(17.90, 221.43, -20.04, 3.66)}
    {pt3dadd(19.74, 220.20, -5.28, 3.66)}

    {create dendA1_000001}
    {dendA1_00000 connect dendA1_000001(0), 1}
    {access dendA1_000001}
    {nseg = 3}
    {pt3dclear()}
    {pt3dadd(-0.39, 166.25, 8.04, 2.76)}
    {pt3dadd(5.79, 171.44, 1.56, 2.76)}
    {pt3dadd(11.76, 173.91, -2.56, 2.76)}
    {pt3dadd(16.34, 173.50, -2.56, 2.76)}
    {pt3dadd(24.14, 175.15, -2.60, 2.76)}
    {pt3dadd(26.43, 183.38, -5.00, 2.76)}
    {pt3dadd(31.02, 188.73, -7.20, 2.76)}
    {pt3dadd(34.23, 193.67, -9.32, 2.76)}

    {create dendA1_0000010}
    {dendA1_000001 connect dendA1_0000010(0), 1}
    {access dendA1_0000010}
    {nseg = 1}
    {pt3dclear()}
    {pt3dadd(34.23, 193.67, -9.32, 2.76)}
    {pt3dadd(36.53, 196.55, -8.04, 2.76)}

    {create dendA1_00000100}
    {dendA1_0000010 connect dendA1_00000100(0), 1}
    {access dendA1_00000100}
    {nseg = 14}
    {pt3dclear()}
    {pt3dadd(36.53, 196.55, -8.04, 2.76)}
    {pt3dadd(42.49, 194.49, -7.52, 2.76)}
    {pt3dadd(46.16, 192.84, -7.04, 2.76)}
    {pt3dadd(48.91, 187.90, -5.68, 2.76)}
    {pt3dadd(51.21, 184.61, -5.28, 2.76)}
    {pt3dadd(53.04, 180.91, -4.80, 1.84)}
    {pt3dadd(55.79, 178.85, -4.80, 1.84)}
    {pt3dadd(58.54, 176.79, -3.52, 1.84)}
    {pt3dadd(66.34, 175.97, -2.12, 1.84)}
    {pt3dadd(71.39, 173.50, -0.96, 1.84)}
    {pt3dadd(75.98, 172.68, -0.84, 1.84)}
    {pt3dadd(80.10, 172.27, -0.60, 1.84)}
    {pt3dadd(85.61, 171.03, -0.60, 1.84)}
    {pt3dadd(88.36, 169.80, 0.48, 1.84)}
    {pt3dadd(93.41, 172.68, 1.20, 1.84)}
    {pt3dadd(96.62, 171.44, 1.00, 1.84)}
    {pt3dadd(100.75, 168.15, 1.00, 0.92)}
    {pt3dadd(102.58, 166.51, 1.00, 0.92)}
    {pt3dadd(105.33, 168.15, 2.64, 0.92)}
    {pt3dadd(108.09, 169.80, 6.08, 0.92)}
    {pt3dadd(113.59, 168.56, 7.44, 0.92)}
    {pt3dadd(115.43, 166.92, 7.28, 0.92)}
    {pt3dadd(117.72, 164.86, 7.16, 0.92)}
    {pt3dadd(122.77, 162.80, 7.16, 0.92)}
    {pt3dadd(124.14, 161.16, 7.04, 0.92)}
    {pt3dadd(123.68, 154.98, 7.04, 0.92)}
    {pt3dadd(125.98, 153.75, 7.04, 0.92)}
    {pt3dadd(128.73, 152.10, 7.04, 0.92)}
    {pt3dadd(127.55, 148.60, 10.32, 0.92)}
    {pt3dadd(129.85, 143.25, 11.00, 0.92)}
    {pt3dadd(132.60, 140.37, 11.04, 0.92)}
    {pt3dadd(134.43, 136.26, 11.08, 0.92)}
    {pt3dadd(136.73, 131.32, 11.08, 0.92)}
    {pt3dadd(139.48, 128.03, 11.08, 0.92)}
    {pt3dadd(144.98, 123.91, 12.60, 0.92)}
    {pt3dadd(148.20, 124.32, 13.76, 0.92)}
    {pt3dadd(156.91, 125.97, 14.64, 0.92)}
    {pt3dadd(161.50, 128.85, 13.68, 0.92)}

    {create dendA1_00000101}
    {dendA1_0000010 connect dendA1_00000101(0), 1}
    {access dendA1_00000101}
    {nseg = 22}
    {pt3dclear()}
    {pt3dadd(36.53, 196.55, -8.04, 1.84)}
    {pt3dadd(39.94, 198.40, -10.92, 1.84)}
    {pt3dadd(43.15, 200.04, -10.92, 1.84)}
    {pt3dadd(48.65, 198.81, -10.92, 0.92)}
    {pt3dadd(52.32, 199.63, -10.92, 0.92)}
    {pt3dadd(55.08, 201.69, -10.64, 0.92)}
    {pt3dadd(57.83, 205.80, -9.36, 0.92)}
    {pt3dadd(61.04, 208.68, -8.68, 0.92)}
    {pt3dadd(65.17, 210.74, -7.64, 0.92)}
    {pt3dadd(69.30, 215.27, -7.16, 0.92)}
    {pt3dadd(74.34, 216.09, -6.84, 0.92)}
    {pt3dadd(78.01, 218.97, -5.88, 0.92)}
    {pt3dadd(78.47, 222.68, -5.24, 0.92)}
    {pt3dadd(80.31, 225.56, -5.12, 0.92)}
    {pt3dadd(85.35, 226.79, -1.48, 0.92)}
    {pt3dadd(92.05, 234.82, -1.60, 0.92)}
    {pt3dadd(92.51, 238.94, -1.68, 0.92)}
    {pt3dadd(95.26, 239.76, -1.68, 0.92)}
    {pt3dadd(97.10, 245.11, -1.68, 0.92)}
    {pt3dadd(97.56, 252.11, -0.60, 0.92)}
    {pt3dadd(101.23, 253.75, -0.80, 0.92)}
    {pt3dadd(106.73, 259.93, 0.60, 0.92)}
    {pt3dadd(110.40, 264.04, 1.76, 0.92)}
    {pt3dadd(114.99, 269.39, -0.96, 0.92)}
    {pt3dadd(116.36, 274.33, -2.32, 0.92)}
    {pt3dadd(119.57, 282.15, -3.16, 0.92)}
    {pt3dadd(119.12, 287.09, -3.72, 0.92)}
    {pt3dadd(125.08, 292.44, -3.96, 0.92)}
    {pt3dadd(125.54, 298.20, -4.32, 0.92)}
    {pt3dadd(126.00, 302.31, -4.60, 0.92)}
    {pt3dadd(128.98, 306.21, -7.64, 0.92)}
    {pt3dadd(130.82, 314.03, -8.24, 0.92)}
    {pt3dadd(133.11, 318.14, -8.64, 0.92)}
    {pt3dadd(135.86, 321.02, -8.96, 0.92)}
    {pt3dadd(138.16, 328.02, -8.96, 0.92)}
    {pt3dadd(138.61, 335.84, -9.84, 0.92)}
    {pt3dadd(140.91, 340.37, -10.68, 0.92)}
    {pt3dadd(141.37, 346.95, -7.32, 0.92)}
    {pt3dadd(144.58, 354.36, -6.28, 0.92)}
    {pt3dadd(145.04, 360.53, -6.24, 0.92)}
    {pt3dadd(142.28, 365.88, -5.68, 0.92)}

    {create dendA1_0000011}
    {dendA1_000001 connect dendA1_0000011(0), 1}
    {access dendA1_0000011}
    {nseg = 12}
    {pt3dclear()}
    {pt3dadd(34.23, 193.67, -9.32, 2.76)}
    {pt3dadd(30.76, 195.96, -12.44, 2.76)}
    {pt3dadd(29.38, 198.43, -12.44, 2.76)}
    {pt3dadd(29.84, 202.14, -12.08, 2.76)}
    {pt3dadd(32.59, 204.19, -11.88, 1.84)}
    {pt3dadd(31.68, 206.25, -10.32, 1.84)}
    {pt3dadd(28.47, 207.90, -14.04, 1.84)}
    {pt3dadd(25.26, 209.54, -16.32, 1.84)}
    {pt3dadd(26.17, 213.25, -18.44, 1.84)}
    {pt3dadd(25.26, 216.54, -19.88, 1.84)}
    {pt3dadd(28.47, 219.83, -24.16, 1.84)}
    {pt3dadd(24.34, 222.71, -24.16, 1.84)}
    {pt3dadd(23.42, 226.42, -25.40, 1.84)}
    {pt3dadd(20.67, 227.65, -26.52, 1.84)}
    {pt3dadd(17.00, 232.18, -27.20, 1.84)}
    {pt3dadd(17.00, 233.41, -29.16, 1.84)}
    {pt3dadd(20.21, 235.47, -30.08, 1.84)}
    {pt3dadd(20.67, 237.53, -31.48, 0.92)}
    {pt3dadd(18.37, 239.17, -32.96, 0.92)}
    {pt3dadd(16.08, 243.29, -33.20, 0.92)}
    {pt3dadd(15.16, 248.23, -35.16, 0.92)}
    {pt3dadd(19.29, 248.64, -40.64, 0.92)}
    {pt3dadd(22.04, 249.05, -41.36, 0.92)}
    {pt3dadd(19.75, 251.93, -39.12, 0.92)}
    {pt3dadd(21.13, 253.58, -43.16, 0.92)}
    {pt3dadd(25.71, 254.40, -43.92, 0.92)}
    {pt3dadd(25.26, 257.69, -45.88, 0.92)}
    {pt3dadd(26.17, 260.98, -46.00, 0.92)}
    {pt3dadd(28.92, 260.98, -47.76, 0.92)}
    {pt3dadd(28.92, 266.33, -48.64, 0.92)}
    {pt3dadd(30.30, 267.98, -48.72, 0.92)}
    {pt3dadd(31.45, 271.09, -50.24, 0.92)}
    {pt3dadd(31.91, 274.79, -51.44, 0.92)}
    {pt3dadd(32.82, 281.79, -52.64, 0.92)}
    {pt3dadd(33.74, 285.90, -52.64, 0.92)}
    {pt3dadd(31.91, 288.78, -52.64, 0.92)}
    {pt3dadd(29.15, 291.25, -52.64, 0.92)}
    {pt3dadd(29.61, 294.54, -53.76, 0.92)}

    {create dendA1_00001}
    {dendA1_0000 connect dendA1_00001(0), 1}
    {access dendA1_00001}
    {nseg = 4}
    {pt3dclear()}
    {pt3dadd(-1.31, 158.84, 8.96, 1.84)}
    {pt3dadd(-5.20, 162.71, 4.04, 1.84)}
    {pt3dadd(-8.87, 163.12, 3.08, 1.84)}
    {pt3dadd(-9.79, 162.30, 5.28, 1.84)}
    {pt3dadd(-10.70, 161.06, 8.56, 1.84)}
    {pt3dadd(-9.79, 159.42, 10.40, 1.84)}
    {pt3dadd(-7.95, 160.24, 11.40, 1.84)}
    {pt3dadd(-8.41, 163.94, 11.80, 1.84)}
    {pt3dadd(-9.79, 166.00, 12.24, 1.84)}
    {pt3dadd(-12.08, 166.82, 12.40, 1.84)}
    {pt3dadd(-13.91, 168.06, 13.36, 1.84)}
    {pt3dadd(-13.46, 171.35, 14.88, 1.84)}
    {pt3dadd(-11.16, 173.00, 15.28, 1.84)}
    {pt3dadd(-7.95, 173.41, 15.64, 1.84)}
    {pt3dadd(-7.03, 175.88, 15.32, 0.92)}
    {pt3dadd(-6.58, 177.94, 15.84, 0.92)}
    {pt3dadd(-6.58, 182.05, 15.96, 0.92)}

    {create dendA1_0001}
    {dendA1_000 connect dendA1_0001(0), 1}
    {access dendA1_0001}
    {nseg = 1}
    {pt3dclear()}
    {pt3dadd(1.18, 128.22, 4.92, 2.76)}
    {pt3dadd(-5.20, 131.02, -2.88, 2.76)}
    {pt3dadd(-7.49, 134.73, -1.00, 2.76)}

    {create dendA1_00010}
    {dendA1_0001 connect dendA1_00010(0), 1}
    {access dendA1_00010}
    {nseg = 2}
    {pt3dclear()}
    {pt3dadd(-7.49, 134.73, -1.00, 2.76)}
    {pt3dadd(-9.33, 138.02, -3.08, 2.76)}
    {pt3dadd(-14.83, 138.84, 1.08, 2.76)}
    {pt3dadd(-18.96, 138.84, 1.56, 2.76)}
    {pt3dadd(-22.17, 138.84, 2.80, 2.76)}

    {create dendA1_000100}
    {dendA1_00010 connect dendA1_000100(0), 1}
    {access dendA1_000100}
    {nseg = 1}
    {pt3dclear()}
    {pt3dadd(-22.17, 138.84, 2.80, 1.84)}
    {pt3dadd(-24.01, 142.54, 2.40, 1.84)}
    {pt3dadd(-26.30, 142.13, 5.12, 1.84)}

    {create dendA1_0001000}
    {dendA1_000100 connect dendA1_0001000(0), 1}
    {access dendA1_0001000}
    {nseg = 3}
    {pt3dclear()}
    {pt3dadd(-26.30, 142.13, 5.12, 1.84)}
    {pt3dadd(-24.01, 145.84, 7.24, 1.84)}
    {pt3dadd(-20.80, 144.60, 7.84, 1.84)}
    {pt3dadd(-20.80, 148.31, 10.88, 0.92)}
    {pt3dadd(-24.01, 151.60, 13.04, 0.92)}
    {pt3dadd(-26.30, 151.60, 15.28, 0.92)}
    {pt3dadd(-28.13, 152.83, 15.44, 0.92)}
    {pt3dadd(-26.30, 155.30, 14.48, 0.92)}

    {create dendA1_0001001}
    {dendA1_000100 connect dendA1_0001001(0), 1}
    {access dendA1_0001001}
    {nseg = 7}
    {pt3dclear()}
    {pt3dadd(-26.30, 142.13, 5.12, 1.84)}
    {pt3dadd(-27.68, 144.60, 4.24, 1.84)}
    {pt3dadd(-29.97, 147.07, 4.96, 1.84)}
    {pt3dadd(-32.72, 147.07, 5.08, 1.84)}
    {pt3dadd(-37.31, 146.66, 7.16, 1.84)}
    {pt3dadd(-41.44, 148.31, 7.80, 1.84)}
    {pt3dadd(-44.19, 154.07, 7.72, 1.84)}
    {pt3dadd(-48.32, 155.71, 7.44, 1.84)}
    {pt3dadd(-51.53, 156.13, 7.40, 0.92)}
    {pt3dadd(-53.82, 158.59, 10.20, 0.92)}
    {pt3dadd(-57.95, 160.24, 11.00, 0.92)}
    {pt3dadd(-60.24, 161.89, 11.60, 0.92)}
    {pt3dadd(-64.83, 163.94, 12.04, 0.92)}
    {pt3dadd(-68.50, 165.18, 12.68, 0.92)}
    {pt3dadd(-71.25, 165.59, 13.20, 0.92)}
    {pt3dadd(-73.09, 169.71, 14.24, 0.92)}
    {pt3dadd(-74.92, 172.17, 14.56, 0.92)}
    {pt3dadd(-76.76, 173.00, 15.32, 0.92)}
    {pt3dadd(-79.05, 176.29, 15.76, 0.92)}
    {pt3dadd(-81.35, 181.64, 17.72, 0.92)}
    {pt3dadd(-84.56, 184.93, 15.96, 0.92)}

    {create dendA1_000101}
    {dendA1_00010 connect dendA1_000101(0), 1}
    {access dendA1_000101}
    {nseg = 15}
    {pt3dclear()}
    {pt3dadd(-22.17, 138.84, 2.80, 1.84)}
    {pt3dadd(-24.47, 137.20, 1.72, 1.84)}
    {pt3dadd(-27.68, 139.66, 1.56, 1.84)}
    {pt3dadd(-33.18, 140.08, 1.24, 1.84)}
    {pt3dadd(-36.85, 140.49, 1.52, 1.84)}
    {pt3dadd(-41.44, 144.19, 2.84, 1.84)}
    {pt3dadd(-46.94, 146.66, 2.52, 1.84)}
    {pt3dadd(-53.36, 149.13, 2.48, 1.84)}
    {pt3dadd(-61.62, 149.54, 2.48, 1.84)}
    {pt3dadd(-66.21, 151.19, 2.40, 1.84)}
    {pt3dadd(-71.25, 156.13, 1.20, 1.84)}
    {pt3dadd(-79.05, 159.83, 0.28, 1.84)}
    {pt3dadd(-86.39, 164.36, -0.40, 0.92)}
    {pt3dadd(-92.81, 168.88, -0.72, 0.92)}
    {pt3dadd(-97.86, 172.59, -2.00, 0.92)}
    {pt3dadd(-105.66, 174.23, -2.00, 0.92)}
    {pt3dadd(-108.41, 179.99, -2.00, 0.92)}
    {pt3dadd(-112.54, 181.64, -2.00, 0.92)}
    {pt3dadd(-120.34, 184.11, -0.88, 0.92)}
    {pt3dadd(-126.30, 189.87, -2.52, 0.92)}
    {pt3dadd(-128.13, 194.81, -2.56, 0.92)}
    {pt3dadd(-128.13, 207.98, -2.60, 0.92)}
    {pt3dadd(-126.30, 212.92, -2.60, 0.92)}
    {pt3dadd(-126.76, 217.85, -3.16, 0.92)}
    {pt3dadd(-125.84, 221.97, 0.32, 0.92)}
    {pt3dadd(-126.30, 225.26, -0.28, 0.92)}
    {pt3dadd(-128.13, 230.61, -0.80, 0.92)}
    {pt3dadd(-126.04, 233.27, -0.24, 0.92)}
    {pt3dadd(-127.41, 240.27, -2.60, 0.92)}
    {pt3dadd(-128.79, 244.80, 1.44, 0.92)}

    {create dendA1_00011}
    {dendA1_0001 connect dendA1_00011(0), 1}
    {access dendA1_00011}
    {nseg = 3}
    {pt3dclear()}
    {pt3dadd(-7.49, 134.73, -1.00, 1.84)}
    {pt3dadd(-11.40, 133.67, -3.68, 1.84)}
    {pt3dadd(-13.23, 132.85, -3.92, 1.84)}
    {pt3dadd(-18.74, 137.37, -6.88, 1.84)}
    {pt3dadd(-21.03, 136.14, -9.16, 1.84)}
    {pt3dadd(-26.54, 134.91, -10.96, 1.84)}
    {pt3dadd(-34.33, 137.79, -12.20, 1.84)}
    {pt3dadd(-38.00, 139.43, -12.28, 1.84)}
    {pt3dadd(-40.30, 140.67, -13.36, 1.84)}

    {create dendA1_000110}
    {dendA1_00011 connect dendA1_000110(0), 1}
    {access dendA1_000110}
    {nseg = 2}
    {pt3dclear()}
    {pt3dadd(-40.30, 140.67, -13.36, 1.84)}
    {pt3dadd(-44.89, 142.31, -15.08, 1.84)}
    {pt3dadd(-47.64, 144.78, -16.04, 1.84)}
    {pt3dadd(-49.47, 147.66, -17.60, 1.84)}
    {pt3dadd(-51.31, 150.13, -19.52, 1.84)}
    {pt3dadd(-54.52, 153.01, -20.00, 1.84)}
    {pt3dadd(-54.52, 156.72, -20.96, 1.84)}
    {pt3dadd(-57.27, 155.89, -21.36, 1.84)}

    {create dendA1_0001100}
    {dendA1_000110 connect dendA1_0001100(0), 1}
    {access dendA1_0001100}
    {nseg = 20}
    {pt3dclear()}
    {pt3dadd(-57.27, 155.89, -21.36, 0.92)}
    {pt3dadd(-60.48, 155.89, -22.68, 0.92)}
    {pt3dadd(-65.07, 155.89, -23.60, 0.92)}
    {pt3dadd(-69.20, 157.54, -24.80, 0.92)}
    {pt3dadd(-69.20, 162.89, -25.32, 0.92)}
    {pt3dadd(-73.33, 162.07, -26.68, 0.92)}
    {pt3dadd(-77.00, 163.71, -27.16, 0.92)}
    {pt3dadd(-76.54, 160.42, -30.16, 0.92)}
    {pt3dadd(-79.29, 161.24, -30.40, 0.92)}
    {pt3dadd(-81.58, 165.77, -30.84, 0.92)}
    {pt3dadd(-87.09, 167.42, -32.88, 0.92)}
    {pt3dadd(-91.67, 167.83, -34.68, 0.92)}
    {pt3dadd(-96.26, 168.24, -35.60, 0.92)}
    {pt3dadd(-100.85, 169.47, -36.00, 0.92)}
    {pt3dadd(-104.52, 172.77, -36.68, 0.92)}
    {pt3dadd(-104.98, 176.47, -37.64, 0.92)}
    {pt3dadd(-109.11, 180.58, -38.44, 0.92)}
    {pt3dadd(-112.32, 181.82, -38.92, 0.92)}
    {pt3dadd(-117.36, 181.82, -39.76, 0.92)}
    {pt3dadd(-120.11, 187.17, -40.40, 0.92)}
    {pt3dadd(-123.78, 190.05, -41.04, 0.92)}
    {pt3dadd(-126.08, 194.17, -42.16, 0.92)}
    {pt3dadd(-129.75, 194.17, -42.96, 0.92)}
    {pt3dadd(-134.79, 193.34, -40.56, 0.92)}
    {pt3dadd(-137.55, 193.75, -38.84, 0.92)}
    {pt3dadd(-139.56, 197.71, -37.80, 0.92)}
    {pt3dadd(-145.53, 202.24, -37.68, 0.92)}
    {pt3dadd(-153.33, 206.35, -37.92, 0.92)}
    {pt3dadd(-157.91, 210.06, -35.60, 0.92)}
    {pt3dadd(-163.42, 210.06, -34.68, 0.92)}
    {pt3dadd(-170.76, 214.58, -32.84, 0.92)}
    {pt3dadd(-176.72, 218.70, -32.12, 0.92)}
    {pt3dadd(-181.31, 218.70, -29.56, 0.92)}
    {pt3dadd(-188.19, 218.70, -27.96, 0.92)}
    {pt3dadd(-192.78, 219.52, -27.28, 0.92)}
    {pt3dadd(-196.44, 216.64, -26.40, 0.92)}
    {pt3dadd(-203.33, 213.76, -26.24, 0.46)}
    {pt3dadd(-209.29, 211.29, -27.68, 0.46)}

    {create dendA1_0001101}
    {dendA1_000110 connect dendA1_0001101(0), 1}
    {access dendA1_0001101}
    {nseg = 6}
    {pt3dclear()}
    {pt3dadd(-57.27, 155.89, -21.36, 0.92)}
    {pt3dadd(-60.21, 155.32, -26.88, 0.92)}
    {pt3dadd(-62.96, 154.09, -28.88, 0.92)}
    {pt3dadd(-62.50, 149.56, -30.16, 0.92)}
    {pt3dadd(-61.58, 147.50, -32.12, 0.92)}
    {pt3dadd(-65.71, 146.68, -32.48, 0.92)}
    {pt3dadd(-64.33, 143.39, -35.28, 0.92)}
    {pt3dadd(-69.38, 140.10, -37.84, 0.92)}
    {pt3dadd(-66.63, 137.22, -40.92, 0.92)}
    {pt3dadd(-62.96, 136.39, -42.32, 0.92)}
    {pt3dadd(-59.29, 139.27, -43.40, 0.92)}
    {pt3dadd(-56.54, 136.39, -45.72, 0.92)}
    {pt3dadd(-54.24, 138.86, -47.32, 0.92)}
    {pt3dadd(-51.95, 142.57, -46.92, 0.92)}

    {create dendA1_000111}
    {dendA1_00011 connect dendA1_000111(0), 1}
    {access dendA1_000111}
    {nseg = 14}
    {pt3dclear()}
    {pt3dadd(-40.30, 140.67, -13.36, 1.84)}
    {pt3dadd(-42.32, 138.04, -11.60, 1.84)}
    {pt3dadd(-45.07, 135.98, -11.64, 1.84)}
    {pt3dadd(-47.82, 135.57, -11.28, 1.84)}
    {pt3dadd(-54.70, 132.28, -11.00, 1.84)}
    {pt3dadd(-56.54, 130.63, -10.72, 1.84)}
    {pt3dadd(-57.45, 127.34, -10.08, 1.84)}
    {pt3dadd(-60.21, 122.81, -9.96, 1.84)}
    {pt3dadd(-65.71, 119.52, -9.48, 1.84)}
    {pt3dadd(-66.85, 116.83, -11.32, 1.84)}
    {pt3dadd(-70.52, 113.94, -12.12, 1.84)}
    {pt3dadd(-73.73, 110.24, -13.48, 1.84)}
    {pt3dadd(-77.40, 108.59, -14.00, 1.84)}
    {pt3dadd(-81.98, 103.66, -14.40, 1.84)}
    {pt3dadd(-84.74, 99.13, -14.40, 1.84)}
    {pt3dadd(-88.87, 95.01, -14.12, 0.92)}
    {pt3dadd(-92.08, 91.72, -12.84, 0.92)}
    {pt3dadd(-95.75, 89.25, -12.16, 0.92)}
    {pt3dadd(-97.58, 86.37, -11.68, 0.92)}
    {pt3dadd(-99.42, 84.73, -11.32, 0.92)}
    {pt3dadd(-104.00, 81.85, -10.84, 0.92)}
    {pt3dadd(-107.67, 78.97, -10.44, 0.92)}
    {pt3dadd(-109.51, 75.26, -9.88, 0.92)}
    {pt3dadd(-110.42, 73.20, -12.64, 0.92)}
    {pt3dadd(-113.18, 71.15, -13.20, 0.92)}
    {pt3dadd(-114.55, 68.68, -14.08, 0.92)}
    {pt3dadd(-115.93, 64.97, -15.44, 0.92)}
    {pt3dadd(-116.85, 61.68, -16.04, 0.92)}
    {pt3dadd(-122.35, 57.98, -16.52, 0.92)}
    {pt3dadd(-126.02, 53.86, -14.60, 0.92)}
    {pt3dadd(-128.77, 49.34, -16.32, 0.92)}
    {pt3dadd(-129.69, 46.45, -16.40, 0.92)}
    {pt3dadd(-129.93, 43.41, -18.72, 0.92)}
    {pt3dadd(-131.76, 38.89, -18.76, 0.92)}
    {pt3dadd(-135.89, 32.30, -19.40, 0.92)}
    {pt3dadd(-139.56, 31.07, -20.08, 0.92)}
    {pt3dadd(-145.53, 28.19, -19.12, 0.92)}

    {create dendA1_001}
    {dendA1_00 connect dendA1_001(0), 1}
    {access dendA1_001}
    {nseg = 7}
    {pt3dclear()}
    {pt3dadd(4.85, 88.72, 5.16, 1.84)}
    {pt3dadd(9.16, 88.49, 1.00, 1.84)}
    {pt3dadd(10.08, 92.20, 1.00, 1.84)}
    {pt3dadd(13.74, 94.26, 1.00, 1.84)}
    {pt3dadd(17.41, 97.14, 1.00, 1.84)}
    {pt3dadd(21.08, 99.19, 3.44, 0.92)}
    {pt3dadd(24.30, 99.19, 4.00, 0.92)}
    {pt3dadd(28.88, 100.43, 3.68, 0.92)}
    {pt3dadd(32.09, 104.13, 4.72, 0.92)}
    {pt3dadd(33.47, 106.19, 5.08, 0.92)}
    {pt3dadd(36.68, 109.07, 6.48, 0.92)}
    {pt3dadd(36.22, 111.95, 7.40, 0.92)}
    {pt3dadd(36.68, 115.24, 9.04, 0.92)}
    {pt3dadd(38.97, 112.36, 9.16, 0.92)}
    {pt3dadd(41.27, 109.89, 12.20, 0.92)}
    {pt3dadd(41.73, 107.01, 15.00, 0.92)}
    {pt3dadd(41.73, 112.36, 15.04, 0.92)}
    {pt3dadd(44.02, 113.19, 15.08, 0.92)}
    {pt3dadd(45.40, 110.30, 16.12, 0.92)}
    {pt3dadd(49.52, 107.84, 15.72, 0.92)}

    {create dendA1_01}
    {dendA1_0 connect dendA1_01(0), 1}
    {access dendA1_01}
    {nseg = 1}
    {pt3dclear()}
    {pt3dadd(5.05, 55.14, 3.12, 2.76)}
    {pt3dadd(10.53, 53.93, -4.84, 2.76)}

    {create dendA1_010}
    {dendA1_01 connect dendA1_010(0), 1}
    {access dendA1_010}
    {nseg = 3}
    {pt3dclear()}
    {pt3dadd(10.53, 53.93, -4.84, 1.84)}
    {pt3dadd(14.66, 56.81, -4.92, 1.84)}
    {pt3dadd(17.87, 57.22, -4.88, 1.84)}
    {pt3dadd(24.75, 59.28, -2.84, 1.84)}
    {pt3dadd(28.42, 62.98, -2.48, 1.84)}
    {pt3dadd(32.55, 65.04, -1.80, 1.84)}
    {pt3dadd(36.22, 65.04, -0.68, 1.84)}
    {pt3dadd(38.52, 65.86, -0.40, 1.84)}

    {create dendA1_0100}
    {dendA1_010 connect dendA1_0100(0), 1}
    {access dendA1_0100}
    {nseg = 2}
    {pt3dclear()}
    {pt3dadd(38.52, 65.86, -0.40, 0.92)}
    {pt3dadd(42.64, 65.04, 0.44, 0.92)}
    {pt3dadd(45.86, 65.45, 0.44, 0.92)}
    {pt3dadd(50.44, 67.51, 1.40, 0.92)}
    {pt3dadd(55.95, 67.51, 2.76, 0.92)}

    {create dendA1_01000}
    {dendA1_0100 connect dendA1_01000(0), 1}
    {access dendA1_01000}
    {nseg = 17}
    {pt3dclear()}
    {pt3dadd(55.95, 67.51, 2.76, 0.92)}
    {pt3dadd(56.41, 70.39, 3.84, 0.92)}
    {pt3dadd(60.53, 71.62, 3.76, 0.92)}
    {pt3dadd(64.20, 72.86, 4.00, 0.92)}
    {pt3dadd(68.79, 73.27, 1.76, 0.92)}
    {pt3dadd(71.54, 77.79, 1.52, 0.92)}
    {pt3dadd(72.46, 79.44, 1.52, 0.92)}
    {pt3dadd(77.97, 81.91, 2.52, 0.92)}
    {pt3dadd(81.18, 81.50, 3.44, 0.92)}
    {pt3dadd(86.22, 83.56, -2.40, 0.92)}
    {pt3dadd(89.89, 86.44, -2.88, 0.92)}
    {pt3dadd(93.10, 88.91, -2.92, 0.92)}
    {pt3dadd(97.69, 91.79, -3.36, 0.92)}
    {pt3dadd(104.57, 95.49, -3.36, 0.92)}
    {pt3dadd(108.70, 96.72, -3.56, 0.92)}
    {pt3dadd(115.12, 100.84, -4.08, 0.92)}
    {pt3dadd(120.63, 104.54, -6.40, 0.92)}
    {pt3dadd(125.21, 107.42, -7.52, 0.92)}
    {pt3dadd(126.59, 113.60, -9.20, 0.92)}
    {pt3dadd(131.18, 120.59, -9.24, 0.92)}
    {pt3dadd(131.63, 125.94, -9.60, 0.92)}
    {pt3dadd(134.85, 130.06, -10.68, 0.92)}
    {pt3dadd(137.78, 134.83, -8.64, 0.92)}
    {pt3dadd(143.29, 139.36, -9.52, 0.92)}
    {pt3dadd(148.79, 139.77, -9.52, 0.92)}
    {pt3dadd(152.46, 145.53, -9.52, 0.92)}
    {pt3dadd(159.34, 150.88, -9.36, 0.92)}
    {pt3dadd(162.55, 156.64, -3.36, 0.92)}
    {pt3dadd(166.22, 159.52, -3.36, 0.92)}
    {pt3dadd(167.14, 168.58, -4.44, 0.92)}

    {create dendA1_01001}
    {dendA1_0100 connect dendA1_01001(0), 1}
    {access dendA1_01001}
    {nseg = 15}
    {pt3dclear()}
    {pt3dadd(55.95, 67.51, 2.76, 1.84)}
    {pt3dadd(58.88, 69.40, 1.36, 1.84)}
    {pt3dadd(62.09, 69.40, -0.48, 1.84)}
    {pt3dadd(65.76, 68.99, -1.28, 1.84)}
    {pt3dadd(69.89, 71.87, -1.32, 1.84)}
    {pt3dadd(74.48, 74.75, 4.04, 0.92)}
    {pt3dadd(75.40, 77.63, 4.04, 0.92)}
    {pt3dadd(79.98, 79.69, 6.80, 0.92)}
    {pt3dadd(85.03, 81.33, 7.80, 0.92)}
    {pt3dadd(90.08, 82.57, 9.12, 0.92)}
    {pt3dadd(95.12, 84.63, 9.40, 0.92)}
    {pt3dadd(98.79, 87.92, 9.52, 0.92)}
    {pt3dadd(102.92, 91.62, 9.32, 0.92)}
    {pt3dadd(109.80, 95.33, 9.00, 0.92)}
    {pt3dadd(113.01, 95.33, 8.76, 0.92)}
    {pt3dadd(120.35, 101.09, 9.08, 0.92)}
    {pt3dadd(123.56, 101.09, 9.00, 0.92)}
    {pt3dadd(126.77, 103.15, 9.00, 0.92)}
    {pt3dadd(131.36, 105.61, 8.96, 0.92)}
    {pt3dadd(139.16, 107.67, 9.48, 0.92)}
    {pt3dadd(142.37, 109.32, 9.48, 0.92)}
    {pt3dadd(154.75, 110.55, 9.48, 0.92)}
    {pt3dadd(159.34, 108.91, 9.48, 0.92)}
    {pt3dadd(164.39, 107.26, 9.44, 0.92)}
    {pt3dadd(169.89, 106.85, 9.56, 0.92)}
    {pt3dadd(171.27, 104.79, 10.92, 0.92)}
    {pt3dadd(181.36, 105.61, 7.72, 0.92)}
    {pt3dadd(184.11, 107.26, 6.92, 0.92)}
    {pt3dadd(190.53, 106.85, 6.60, 0.92)}
    {pt3dadd(196.50, 106.44, 6.40, 0.92)}

    {create dendA1_0101}
    {dendA1_010 connect dendA1_0101(0), 1}
    {access dendA1_0101}
    {nseg = 11}
    {pt3dclear()}
    {pt3dadd(38.52, 65.86, -0.40, 1.84)}
    {pt3dadd(40.99, 63.64, -0.48, 1.84)}
    {pt3dadd(44.66, 62.82, -2.12, 1.84)}
    {pt3dadd(50.17, 62.82, -2.48, 1.84)}
    {pt3dadd(55.67, 63.23, -2.64, 0.92)}
    {pt3dadd(59.34, 64.87, -3.96, 0.92)}
    {pt3dadd(61.63, 66.93, -4.40, 0.92)}
    {pt3dadd(66.22, 68.99, -4.72, 0.92)}
    {pt3dadd(74.94, 69.40, -4.96, 0.92)}
    {pt3dadd(79.07, 71.46, -4.00, 0.92)}
    {pt3dadd(81.36, 71.05, -4.00, 0.92)}
    {pt3dadd(85.03, 73.52, -5.08, 0.92)}
    {pt3dadd(87.78, 76.40, -5.60, 0.92)}
    {pt3dadd(96.50, 80.10, -5.72, 0.92)}
    {pt3dadd(102.92, 80.92, -6.96, 0.92)}
    {pt3dadd(107.05, 80.92, -8.20, 0.92)}
    {pt3dadd(107.97, 79.28, -8.44, 0.92)}
    {pt3dadd(110.26, 79.28, -9.60, 0.92)}
    {pt3dadd(113.47, 75.98, -10.56, 0.92)}
    {pt3dadd(116.22, 73.10, -11.92, 0.92)}
    {pt3dadd(119.89, 70.22, -12.04, 0.92)}
    {pt3dadd(120.35, 68.17, -12.92, 0.92)}
    {pt3dadd(122.64, 64.46, -13.36, 0.92)}
    {pt3dadd(123.56, 61.17, -13.76, 0.92)}
    {pt3dadd(124.94, 57.88, -13.76, 0.92)}
    {pt3dadd(124.67, 54.81, -14.64, 0.92)}
    {pt3dadd(126.51, 47.82, -16.48, 0.92)}

    {create dendA1_011}
    {dendA1_01 connect dendA1_011(0), 1}
    {access dendA1_011}
    {nseg = 4}
    {pt3dclear()}
    {pt3dadd(10.53, 53.93, -4.84, 1.84)}
    {pt3dadd(9.99, 56.46, -5.88, 1.84)}
    {pt3dadd(11.83, 60.16, -5.88, 1.84)}
    {pt3dadd(15.96, 61.40, -5.88, 1.84)}
    {pt3dadd(19.17, 60.57, -7.48, 1.84)}
    {pt3dadd(21.00, 62.63, -9.72, 1.84)}
    {pt3dadd(22.38, 65.51, -10.12, 1.84)}
    {pt3dadd(28.34, 65.92, -11.32, 1.84)}
    {pt3dadd(29.72, 67.16, -11.44, 1.84)}
    {pt3dadd(32.01, 69.63, -12.96, 1.84)}
    {pt3dadd(32.93, 70.86, -13.88, 1.84)}
    {pt3dadd(35.68, 72.92, -14.72, 1.84)}
    {pt3dadd(37.51, 74.57, -15.08, 1.84)}
    {pt3dadd(41.18, 76.62, -16.24, 1.84)}
    {pt3dadd(43.94, 79.92, -16.60, 1.84)}
    {pt3dadd(43.02, 82.39, -19.00, 1.84)}

    {create dendA1_0110}
    {dendA1_011 connect dendA1_0110(0), 1}
    {access dendA1_0110}
    {nseg = 18}
    {pt3dclear()}
    {pt3dadd(43.02, 82.39, -19.00, 0.92)}
    {pt3dadd(47.61, 81.15, -18.16, 0.92)}
    {pt3dadd(52.19, 81.56, -23.20, 0.92)}
    {pt3dadd(56.32, 83.21, -23.16, 0.92)}
    {pt3dadd(58.62, 86.09, -23.40, 0.92)}
    {pt3dadd(61.37, 89.38, -24.00, 0.92)}
    {pt3dadd(69.62, 90.20, -24.16, 0.92)}
    {pt3dadd(72.38, 93.09, -24.00, 0.92)}
    {pt3dadd(76.51, 98.02, -27.36, 0.92)}
    {pt3dadd(77.88, 101.32, -29.08, 0.92)}
    {pt3dadd(76.51, 106.25, -29.88, 0.92)}
    {pt3dadd(79.72, 110.37, -30.08, 0.92)}
    {pt3dadd(83.84, 113.25, -31.40, 0.92)}
    {pt3dadd(85.68, 116.95, -33.08, 0.92)}
    {pt3dadd(85.22, 121.07, -34.72, 0.92)}
    {pt3dadd(89.35, 126.01, -36.20, 0.92)}
    {pt3dadd(90.27, 128.48, -37.28, 0.92)}
    {pt3dadd(93.02, 126.01, -38.00, 0.92)}
    {pt3dadd(94.85, 128.89, -41.32, 0.92)}
    {pt3dadd(96.69, 130.95, -41.92, 0.92)}
    {pt3dadd(100.36, 131.77, -43.76, 0.92)}
    {pt3dadd(102.46, 134.47, -40.68, 0.92)}
    {pt3dadd(106.59, 133.65, -42.60, 0.92)}
    {pt3dadd(110.26, 133.65, -44.92, 0.92)}
    {pt3dadd(113.01, 137.36, -46.36, 0.92)}
    {pt3dadd(115.77, 136.94, -47.04, 0.92)}
    {pt3dadd(118.98, 135.30, -47.92, 0.92)}
    {pt3dadd(120.35, 139.41, -48.52, 0.92)}
    {pt3dadd(120.81, 140.65, -50.08, 0.92)}
    {pt3dadd(123.57, 139.00, -52.40, 0.92)}
    {pt3dadd(124.94, 137.77, -53.84, 0.92)}
    {pt3dadd(127.69, 139.00, -55.76, 0.92)}
    {pt3dadd(129.99, 137.36, -56.68, 0.92)}
    {pt3dadd(131.36, 136.12, -57.36, 0.92)}
    {pt3dadd(135.03, 136.53, -59.56, 0.92)}
    {pt3dadd(137.33, 136.94, -59.64, 0.92)}
    {pt3dadd(141.00, 139.00, -60.24, 0.46)}
    {pt3dadd(144.67, 137.36, -60.72, 0.46)}
    {pt3dadd(147.88, 136.94, -61.44, 0.46)}
    {pt3dadd(150.17, 135.71, -62.32, 0.46)}
    {pt3dadd(155.68, 135.71, -62.32, 0.46)}
    {pt3dadd(158.43, 132.01, -62.72, 0.46)}
    {pt3dadd(159.34, 129.54, -63.08, 0.46)}

    {create dendA1_0111}
    {dendA1_011 connect dendA1_0111(0), 1}
    {access dendA1_0111}
    {nseg = 1}
    {pt3dclear()}
    {pt3dadd(43.02, 82.39, -19.00, 1.84)}
    {pt3dadd(44.21, 85.09, -18.40, 1.84)}

    {create dendA1_01110}
    {dendA1_0111 connect dendA1_01110(0), 1}
    {access dendA1_01110}
    {nseg = 2}
    {pt3dclear()}
    {pt3dadd(44.21, 85.09, -18.40, 0.92)}
    {pt3dadd(47.42, 84.27, -17.68, 0.92)}
    {pt3dadd(50.17, 85.09, -22.28, 0.92)}
    {pt3dadd(54.76, 86.74, -22.32, 0.92)}

    {create dendA1_01111}
    {dendA1_0111 connect dendA1_01111(0), 1}
    {access dendA1_01111}
    {nseg = 9}
    {pt3dclear()}
    {pt3dadd(44.21, 85.09, -18.40, 1.84)}
    {pt3dadd(46.96, 87.15, -17.92, 1.84)}
    {pt3dadd(49.71, 86.74, -22.40, 1.84)}
    {pt3dadd(49.71, 89.62, -25.08, 1.84)}
    {pt3dadd(50.63, 92.09, -25.64, 1.84)}
    {pt3dadd(54.76, 93.73, -26.48, 1.84)}
    {pt3dadd(57.05, 96.20, -28.36, 1.84)}
    {pt3dadd(54.30, 101.14, -30.64, 1.84)}
    {pt3dadd(55.22, 104.85, -32.16, 1.84)}
    {pt3dadd(53.38, 107.31, -33.56, 0.92)}
    {pt3dadd(52.92, 109.78, -33.96, 0.92)}
    {pt3dadd(52.92, 112.66, -34.68, 0.92)}
    {pt3dadd(55.22, 115.96, -35.00, 0.92)}
    {pt3dadd(55.68, 119.66, -35.28, 0.92)}
    {pt3dadd(58.89, 123.78, -36.04, 0.92)}
    {pt3dadd(62.56, 126.24, -38.44, 0.92)}
    {pt3dadd(65.77, 127.89, -40.00, 0.92)}
    {pt3dadd(70.35, 132.01, -40.16, 0.92)}
    {pt3dadd(71.73, 135.30, -42.24, 0.92)}
    {pt3dadd(71.73, 139.82, -43.48, 0.92)}
    {pt3dadd(71.73, 143.94, -44.52, 0.92)}
    {pt3dadd(71.73, 146.00, -41.72, 0.92)}
    {pt3dadd(72.19, 147.64, -40.80, 0.92)}
    {pt3dadd(70.81, 153.40, -40.00, 0.92)}
    {pt3dadd(70.81, 157.52, -40.72, 0.92)}
    {pt3dadd(71.27, 161.22, -40.00, 0.92)}

    {create dendA2_0}
    {somaA connect dendA2_0(0), 0.624279}
    {access dendA2_0}
    {nseg = 1}
    {pt3dclear()}
    {pt3dadd(-10.81, 21.16, -5.88, 3.66)}
    {pt3dadd(-16.31, 21.57, -6.12, 3.66)}
    {pt3dadd(-18.61, 19.51, -6.16, 3.66)}
    {pt3dadd(-23.65, 19.51, -5.96, 3.66)}

    {create dendA2_00}
    {dendA2_0 connect dendA2_00(0), 1}
    {access dendA2_00}
    {nseg = 1}
    {pt3dclear()}
    {pt3dadd(-23.65, 19.51, -5.96, 3.66)}
    {pt3dadd(-21.36, 21.16, -4.56, 3.66)}

    {create dendA2_000}
    {dendA2_00 connect dendA2_000(0), 1}
    {access dendA2_000}
    {nseg = 13}
    {pt3dclear()}
    {pt3dadd(-21.36, 21.16, -4.56, 2.76)}
    {pt3dadd(-21.36, 24.45, -4.56, 2.76)}
    {pt3dadd(-23.19, 29.80, -6.24, 2.76)}
    {pt3dadd(-24.11, 35.97, -6.36, 1.84)}
    {pt3dadd(-27.32, 40.91, -6.36, 1.84)}
    {pt3dadd(-31.45, 46.26, -7.68, 1.84)}
    {pt3dadd(-35.12, 49.14, -6.68, 1.84)}
    {pt3dadd(-39.25, 52.02, -4.32, 1.84)}
    {pt3dadd(-44.75, 58.20, -4.44, 0.92)}
    {pt3dadd(-50.72, 61.90, -4.44, 0.92)}
    {pt3dadd(-56.68, 68.07, -4.44, 0.92)}
    {pt3dadd(-59.89, 73.01, -4.44, 0.92)}
    {pt3dadd(-64.48, 76.72, -3.52, 0.92)}
    {pt3dadd(-69.07, 80.83, -3.32, 0.92)}
    {pt3dadd(-73.19, 83.71, -2.64, 0.92)}
    {pt3dadd(-80.99, 88.24, -2.16, 0.92)}
    {pt3dadd(-85.12, 92.35, -5.52, 0.92)}
    {pt3dadd(-88.79, 95.65, -3.44, 0.92)}
    {pt3dadd(-92.00, 100.17, -3.44, 0.92)}
    {pt3dadd(-92.92, 102.64, -1.44, 0.92)}
    {pt3dadd(-97.51, 104.70, -0.56, 0.92)}
    {pt3dadd(-103.01, 109.23, -0.32, 0.92)}
    {pt3dadd(-105.76, 112.11, -0.32, 0.92)}
    {pt3dadd(-110.62, 114.77, 1.64, 0.92)}
    {pt3dadd(-114.29, 116.01, 1.80, 0.92)}
    {pt3dadd(-119.33, 119.30, 3.52, 0.92)}
    {pt3dadd(-122.08, 122.59, 4.24, 0.92)}
    {pt3dadd(-125.75, 125.47, 2.44, 0.92)}

    {create dendA2_0000}
    {dendA2_000 connect dendA2_0000(0), 1}
    {access dendA2_0000}
    {nseg = 6}
    {pt3dclear()}
    {pt3dadd(-125.75, 125.47, 2.44, 0.46)}
    {pt3dadd(-129.88, 125.47, 2.40, 0.46)}
    {pt3dadd(-132.18, 130.41, 2.24, 0.46)}
    {pt3dadd(-135.85, 133.29, 5.24, 0.46)}
    {pt3dadd(-138.14, 136.59, 1.72, 0.46)}
    {pt3dadd(-141.81, 140.29, 1.08, 0.46)}
    {pt3dadd(-143.64, 143.58, 0.88, 0.46)}
    {pt3dadd(-146.86, 146.87, 3.60, 0.46)}
    {pt3dadd(-150.52, 149.75, 3.76, 0.46)}

    {create dendA2_0001}
    {dendA2_000 connect dendA2_0001(0), 1}
    {access dendA2_0001}
    {nseg = 3}
    {pt3dclear()}
    {pt3dadd(-125.75, 125.47, 2.44, 0.92)}
    {pt3dadd(-128.51, 127.53, 3.64, 0.92)}
    {pt3dadd(-128.97, 133.29, 4.08, 0.92)}
    {pt3dadd(-133.09, 133.70, 5.76, 0.92)}
    {pt3dadd(-134.01, 136.59, 3.52, 0.92)}
    {pt3dadd(-135.85, 136.59, 2.64, 0.92)}
    {pt3dadd(-141.35, 137.82, 2.12, 0.92)}

    {create dendA2_001}
    {dendA2_00 connect dendA2_001(0), 1}
    {access dendA2_001}
    {nseg = 2}
    {pt3dclear()}
    {pt3dadd(-21.36, 21.16, -4.56, 1.84)}
    {pt3dadd(-26.22, 22.64, -5.72, 1.84)}
    {pt3dadd(-28.05, 24.29, -5.32, 1.84)}
    {pt3dadd(-30.80, 24.70, -3.24, 1.84)}
    {pt3dadd(-35.39, 24.70, -4.40, 1.84)}
    {pt3dadd(-37.23, 24.29, -4.44, 1.84)}

    {create dendA2_0010}
    {dendA2_001 connect dendA2_0010(0), 1}
    {access dendA2_0010}
    {nseg = 5}
    {pt3dclear()}
    {pt3dadd(-37.23, 24.29, -4.44, 1.84)}
    {pt3dadd(-39.52, 24.29, -0.48, 1.84)}
    {pt3dadd(-42.73, 25.11, 1.20, 1.84)}
    {pt3dadd(-43.19, 26.76, 2.44, 1.84)}
    {pt3dadd(-45.94, 25.52, 3.60, 1.84)}
    {pt3dadd(-47.32, 22.23, 3.84, 1.84)}
    {pt3dadd(-49.61, 22.23, 4.48, 0.92)}
    {pt3dadd(-51.45, 23.87, 5.08, 0.92)}
    {pt3dadd(-52.36, 26.76, 5.92, 0.92)}
    {pt3dadd(-53.28, 27.58, 9.04, 0.92)}
    {pt3dadd(-55.12, 26.76, 9.52, 0.92)}
    {pt3dadd(-56.95, 25.93, 9.92, 0.92)}
    {pt3dadd(-58.79, 27.17, 10.72, 0.92)}
    {pt3dadd(-60.62, 27.17, 11.68, 0.92)}
    {pt3dadd(-61.08, 28.81, 11.84, 0.92)}
    {pt3dadd(-58.79, 31.69, 11.96, 0.92)}
    {pt3dadd(-57.41, 31.69, 12.04, 0.92)}
    {pt3dadd(-59.24, 32.93, 13.04, 0.92)}
    {pt3dadd(-60.16, 34.57, 13.16, 0.92)}
    {pt3dadd(-60.62, 36.63, 13.72, 0.92)}

    {create dendA2_0011}
    {dendA2_001 connect dendA2_0011(0), 1}
    {access dendA2_0011}
    {nseg = 3}
    {pt3dclear()}
    {pt3dadd(-37.23, 24.29, -4.44, 1.84)}
    {pt3dadd(-39.52, 22.23, -6.44, 1.84)}
    {pt3dadd(-41.81, 20.58, -5.32, 1.84)}
    {pt3dadd(-43.65, 19.35, -5.24, 1.84)}
    {pt3dadd(-47.32, 16.88, -5.20, 1.84)}
    {pt3dadd(-50.53, 18.11, -5.20, 1.84)}
    {pt3dadd(-53.74, 18.11, -4.48, 1.84)}
    {pt3dadd(-56.03, 17.29, -7.44, 1.84)}
    {pt3dadd(-57.87, 15.64, -7.36, 1.84)}
    {pt3dadd(-59.24, 13.59, -7.68, 1.84)}
    {pt3dadd(-62.91, 11.12, -8.16, 0.92)}

    {create dendA2_00110}
    {dendA2_0011 connect dendA2_00110(0), 1}
    {access dendA2_00110}
    {nseg = 12}
    {pt3dclear()}
    {pt3dadd(-62.91, 11.12, -8.16, 0.92)}
    {pt3dadd(-66.58, 11.53, -9.00, 0.92)}
    {pt3dadd(-70.25, 10.71, -9.68, 0.92)}
    {pt3dadd(-73.46, 8.65, -10.16, 0.92)}
    {pt3dadd(-76.68, 9.06, -11.32, 0.92)}
    {pt3dadd(-79.43, 7.83, -12.04, 0.92)}
    {pt3dadd(-83.10, 7.00, -13.08, 0.92)}
    {pt3dadd(-85.39, 4.94, -13.76, 0.92)}
    {pt3dadd(-90.44, 2.48, -14.68, 0.92)}
    {pt3dadd(-92.73, 3.71, -15.64, 0.92)}
    {pt3dadd(-92.27, 1.24, -15.40, 0.92)}
    {pt3dadd(-95.94, -2.05, -15.36, 0.92)}
    {pt3dadd(-95.94, -4.11, -17.72, 0.92)}
    {pt3dadd(-100.07, -4.93, -19.24, 0.92)}
    {pt3dadd(-101.90, -6.58, -21.04, 0.92)}
    {pt3dadd(-105.12, -7.81, -21.72, 0.92)}
    {pt3dadd(-104.66, -8.64, -22.08, 0.92)}
    {pt3dadd(-107.87, -8.64, -22.12, 0.92)}
    {pt3dadd(-108.79, -11.10, -22.12, 0.92)}
    {pt3dadd(-106.95, -11.93, -22.04, 0.92)}
    {pt3dadd(-107.41, -14.40, -22.04, 0.92)}
    {pt3dadd(-109.70, -16.87, -23.96, 0.92)}
    {pt3dadd(-113.37, -18.51, -24.84, 0.92)}
    {pt3dadd(-114.29, -20.16, -24.96, 0.92)}
    {pt3dadd(-115.21, -21.39, -26.08, 0.92)}
    {pt3dadd(-117.04, -21.39, -24.44, 0.92)}
    {pt3dadd(-119.79, -20.57, -25.40, 0.92)}
    {pt3dadd(-121.63, -22.22, -26.12, 0.92)}
    {pt3dadd(-123.46, -23.45, -26.80, 0.92)}
    {pt3dadd(-125.76, -25.10, -28.24, 0.92)}
    {pt3dadd(-128.97, -27.57, -28.40, 0.92)}
    {pt3dadd(-132.64, -30.45, -28.80, 0.92)}
    {pt3dadd(-134.01, -34.15, -28.92, 0.92)}
    {pt3dadd(-134.01, -37.44, -29.08, 0.92)}
    {pt3dadd(-136.77, -43.20, -27.64, 0.92)}

    {create dendA2_00111}
    {dendA2_0011 connect dendA2_00111(0), 1}
    {access dendA2_00111}
    {nseg = 12}
    {pt3dclear()}
    {pt3dadd(-62.91, 11.12, -8.16, 1.84)}
    {pt3dadd(-64.75, 9.06, -7.56, 1.84)}
    {pt3dadd(-67.96, 7.41, -7.60, 0.92)}
    {pt3dadd(-71.63, 7.41, -7.64, 0.92)}
    {pt3dadd(-74.38, 6.59, -7.64, 0.92)}
    {pt3dadd(-78.05, 3.71, -7.64, 0.92)}
    {pt3dadd(-80.34, 2.06, -8.84, 0.92)}
    {pt3dadd(-82.64, 1.24, -9.40, 0.92)}
    {pt3dadd(-86.31, -1.64, -6.96, 0.92)}
    {pt3dadd(-89.06, -3.70, -6.48, 0.92)}
    {pt3dadd(-92.27, -7.40, -6.24, 0.92)}
    {pt3dadd(-94.57, -9.87, -5.64, 0.92)}
    {pt3dadd(-97.32, -13.99, -5.00, 0.92)}
    {pt3dadd(-101.90, -16.45, -4.24, 0.92)}
    {pt3dadd(-105.57, -18.92, -3.48, 0.92)}
    {pt3dadd(-107.41, -23.86, -6.52, 0.92)}
    {pt3dadd(-111.08, -24.68, -7.40, 0.92)}
    {pt3dadd(-112.46, -26.33, -5.04, 0.92)}
    {pt3dadd(-116.58, -29.21, -4.00, 0.92)}
    {pt3dadd(-123.01, -30.03, -2.48, 0.92)}
    {pt3dadd(-125.30, -29.62, -1.00, 0.92)}
    {pt3dadd(-130.34, -29.62, -2.28, 0.92)}
    {pt3dadd(-135.85, -30.03, -2.88, 0.92)}
    {pt3dadd(-137.23, -30.03, -3.08, 0.92)}
    {pt3dadd(-139.52, -27.15, -3.32, 0.92)}
    {pt3dadd(-143.19, -28.39, -0.64, 0.92)}
    {pt3dadd(-145.94, -29.62, 0.08, 0.92)}
    {pt3dadd(-152.36, -29.21, 0.84, 0.92)}
    {pt3dadd(-157.68, -29.82, -1.08, 0.92)}
    {pt3dadd(-163.18, -30.23, -1.28, 0.92)}

    {create dendA2_01}
    {dendA2_0 connect dendA2_01(0), 1}
    {access dendA2_01}
    {nseg = 3}
    {pt3dclear()}
    {pt3dadd(-23.65, 19.51, -5.96, 1.84)}
    {pt3dadd(-21.18, 20.94, -12.68, 1.84)}
    {pt3dadd(-19.34, 23.41, -12.68, 1.84)}
    {pt3dadd(-17.51, 19.71, -12.68, 1.84)}
    {pt3dadd(-13.84, 16.41, -13.24, 1.84)}
    {pt3dadd(-9.71, 12.30, -14.36, 1.84)}
    {pt3dadd(-5.58, 7.77, -15.72, 1.84)}

    {create dendA3_0}
    {somaA connect dendA3_0(0), 0.356944}
    {access dendA3_0}
    {nseg = 1}
    {pt3dclear()}
    {pt3dadd(-10.17, 18.88, -4.56, 2.76)}
    {pt3dadd(-12.92, 15.18, -6.00, 2.76)}
    {pt3dadd(-15.67, 13.53, -5.60, 2.76)}
    {pt3dadd(-20.26, 11.89, -5.52, 2.76)}

    {create dendA3_00}
    {dendA3_0 connect dendA3_00(0), 1}
    {access dendA3_00}
    {nseg = 2}
    {pt3dclear()}
    {pt3dadd(-20.26, 11.89, -5.52, 1.84)}
    {pt3dadd(-23.93, 11.48, -5.52, 1.84)}
    {pt3dadd(-26.22, 9.42, -5.52, 1.84)}
    {pt3dadd(-28.06, 7.77, -4.12, 1.84)}
    {pt3dadd(-32.19, 7.36, -2.92, 1.84)}
    {pt3dadd(-33.10, 6.95, -2.92, 1.84)}
    {pt3dadd(-35.86, 4.07, -3.64, 1.84)}

    {create dendA3_000}
    {dendA3_00 connect dendA3_000(0), 1}
    {access dendA3_000}
    {nseg = 8}
    {pt3dclear()}
    {pt3dadd(-35.86, 4.07, -3.64, 1.84)}
    {pt3dadd(-38.15, 4.48, -2.00, 1.84)}
    {pt3dadd(-38.15, 8.18, -0.12, 0.92)}
    {pt3dadd(-39.98, 9.42, 1.00, 0.92)}
    {pt3dadd(-42.28, 8.18, 1.52, 0.92)}
    {pt3dadd(-45.49, 7.77, 2.08, 0.92)}
    {pt3dadd(-51.45, 7.36, 2.16, 0.92)}
    {pt3dadd(-53.29, 5.71, 2.16, 0.92)}
    {pt3dadd(-57.87, 7.77, 3.40, 0.92)}
    {pt3dadd(-60.63, 8.18, 4.92, 0.92)}
    {pt3dadd(-62.46, 8.59, 5.92, 0.92)}
    {pt3dadd(-65.67, 5.71, 7.16, 0.92)}
    {pt3dadd(-67.97, 4.89, 7.80, 0.92)}
    {pt3dadd(-72.55, 4.89, 8.32, 0.92)}
    {pt3dadd(-74.39, 3.24, 8.36, 0.92)}
    {pt3dadd(-78.98, 2.42, 6.88, 0.92)}
    {pt3dadd(-81.73, 1.60, 6.60, 0.92)}
    {pt3dadd(-87.69, 1.19, 6.16, 0.92)}
    {pt3dadd(-90.90, 1.60, 5.92, 0.92)}
    {pt3dadd(-93.65, 2.01, 5.64, 0.92)}
    {pt3dadd(-99.16, 1.60, 7.80, 0.92)}
    {pt3dadd(-100.99, 0.78, 6.40, 0.92)}
    {pt3dadd(-104.66, 2.01, 6.44, 0.92)}

    {create dendA3_001}
    {dendA3_00 connect dendA3_001(0), 1}
    {access dendA3_001}
    {nseg = 8}
    {pt3dclear()}
    {pt3dadd(-35.86, 4.07, -3.64, 0.92)}
    {pt3dadd(-41.36, 3.24, -4.64, 0.92)}
    {pt3dadd(-43.20, 1.19, -4.56, 0.92)}
    {pt3dadd(-45.49, -0.87, -5.16, 0.92)}
    {pt3dadd(-48.24, -3.75, -4.20, 0.92)}
    {pt3dadd(-49.16, -7.45, -3.24, 0.92)}
    {pt3dadd(-50.08, -13.22, -4.16, 0.92)}
    {pt3dadd(-49.62, -16.10, -4.16, 0.92)}
    {pt3dadd(-51.45, -19.80, -4.16, 0.92)}
    {pt3dadd(-56.50, -23.92, -4.88, 0.92)}
    {pt3dadd(-61.54, -27.21, -3.68, 0.92)}
    {pt3dadd(-67.97, -29.27, -4.56, 0.92)}
    {pt3dadd(-78.52, -29.27, -4.56, 0.92)}
    {pt3dadd(-78.52, -32.56, -3.20, 0.92)}
    {pt3dadd(-84.02, -35.85, -3.52, 0.92)}

    {create dendA3_01}
    {dendA3_0 connect dendA3_01(0), 1}
    {access dendA3_01}
    {nseg = 2}
    {pt3dclear()}
    {pt3dadd(-20.26, 11.89, -5.52, 1.84)}
    {pt3dadd(-23.01, 9.01, -8.56, 1.84)}
    {pt3dadd(-23.47, 3.24, -7.96, 1.84)}
    {pt3dadd(-24.39, 2.42, -9.44, 1.84)}
    {pt3dadd(-29.43, 3.24, -12.04, 1.84)}
    {pt3dadd(-30.81, 3.66, -13.92, 1.84)}
    {pt3dadd(-34.48, 5.30, -13.72, 1.84)}

    {create dendA3_010}
    {dendA3_01 connect dendA3_010(0), 1}
    {access dendA3_010}
    {nseg = 14}
    {pt3dclear()}
    {pt3dadd(-34.48, 5.30, -13.72, 1.84)}
    {pt3dadd(-38.61, 2.01, -13.04, 1.84)}
    {pt3dadd(-42.74, 0.36, -13.36, 1.84)}
    {pt3dadd(-46.87, -1.28, -13.24, 1.84)}
    {pt3dadd(-49.62, -2.93, -11.88, 1.84)}
    {pt3dadd(-53.29, -5.40, -11.68, 0.92)}
    {pt3dadd(-55.12, -9.10, -14.56, 0.92)}
    {pt3dadd(-55.58, -9.92, -16.76, 0.92)}
    {pt3dadd(-59.71, -10.75, -17.32, 0.92)}
    {pt3dadd(-62.00, -13.63, -17.72, 0.92)}
    {pt3dadd(-67.51, -16.51, -17.72, 0.92)}
    {pt3dadd(-70.26, -18.57, -17.80, 0.92)}
    {pt3dadd(-73.47, -20.21, -17.84, 0.92)}
    {pt3dadd(-73.47, -21.03, -18.20, 0.92)}
    {pt3dadd(-76.22, -20.62, -19.20, 0.92)}
    {pt3dadd(-77.14, -22.68, -19.76, 0.92)}
    {pt3dadd(-78.98, -25.56, -20.44, 0.92)}
    {pt3dadd(-79.89, -29.27, -21.12, 0.92)}
    {pt3dadd(-84.48, -31.32, -21.32, 0.92)}
    {pt3dadd(-87.69, -32.56, -21.52, 0.92)}
    {pt3dadd(-90.44, -36.26, -21.56, 0.92)}
    {pt3dadd(-94.57, -39.14, -22.68, 0.92)}
    {pt3dadd(-96.41, -39.55, -20.00, 0.92)}
    {pt3dadd(-100.08, -39.55, -18.76, 0.92)}
    {pt3dadd(-104.66, -42.85, -18.36, 0.92)}
    {pt3dadd(-108.33, -45.31, -17.88, 0.92)}
    {pt3dadd(-111.09, -45.73, -19.20, 0.92)}
    {pt3dadd(-114.30, -46.96, -19.44, 0.92)}
    {pt3dadd(-118.88, -47.37, -19.44, 0.92)}
    {pt3dadd(-123.01, -51.08, -19.92, 0.92)}
    {pt3dadd(-125.52, -52.95, -20.88, 0.92)}
    {pt3dadd(-131.49, -55.83, -20.88, 0.92)}
    {pt3dadd(-136.07, -57.48, -20.88, 0.92)}
    {pt3dadd(-138.83, -57.07, -21.04, 0.92)}
    {pt3dadd(-141.12, -57.07, -21.12, 0.92)}
    {pt3dadd(-144.33, -59.95, -21.12, 0.92)}
    {pt3dadd(-146.17, -60.77, -22.80, 0.92)}
    {pt3dadd(-149.83, -62.42, -21.76, 0.92)}
    {pt3dadd(-155.34, -63.24, -22.44, 0.92)}

    {create dendA3_011}
    {dendA3_01 connect dendA3_011(0), 1}
    {access dendA3_011}
    {nseg = 4}
    {pt3dclear()}
    {pt3dadd(-34.48, 5.30, -13.72, 1.84)}
    {pt3dadd(-37.91, 5.90, -16.56, 1.84)}
    {pt3dadd(-42.04, 6.72, -18.88, 1.84)}
    {pt3dadd(-42.95, 8.78, -20.72, 1.84)}
    {pt3dadd(-41.12, 10.83, -21.68, 1.84)}
    {pt3dadd(-46.62, 10.42, -21.84, 1.84)}
    {pt3dadd(-48.46, 11.25, -25.28, 1.84)}
    {pt3dadd(-51.21, 13.71, -26.12, 1.84)}
    {pt3dadd(-53.05, 14.13, -28.56, 1.84)}
    {pt3dadd(-60.84, 16.18, -27.88, 1.84)}
    {pt3dadd(-65.89, 14.13, -27.68, 1.84)}
    {pt3dadd(-65.89, 17.01, -26.24, 0.92)}

    {create dendA4_0}
    {somaA connect dendA4_0(0), 0.000000}
    {access dendA4_0}
    {nseg = 3}
    {pt3dclear()}
    {pt3dadd(-3.05, 11.29, 6.08, 3.66)}
    {pt3dadd(-5.80, 11.70, 6.00, 3.66)}
    {pt3dadd(-9.01, 10.47, 5.00, 3.66)}
    {pt3dadd(-10.85, 7.59, 8.80, 3.66)}
    {pt3dadd(-12.68, 6.76, 6.24, 3.66)}
    {pt3dadd(-13.60, 1.83, 5.48, 3.66)}
    {pt3dadd(-16.35, -3.11, 3.92, 2.76)}
    {pt3dadd(-18.19, -5.58, 8.80, 2.76)}
    {pt3dadd(-22.32, -5.99, 9.08, 1.84)}
    {pt3dadd(-25.99, -5.99, 9.08, 1.84)}
    {pt3dadd(-29.20, -7.23, 8.48, 1.84)}

    {create dendA4_00}
    {dendA4_0 connect dendA4_00(0), 1}
    {access dendA4_00}
    {nseg = 9}
    {pt3dclear()}
    {pt3dadd(-29.20, -7.23, 8.48, 0.92)}
    {pt3dadd(-30.57, -12.17, 8.48, 0.92)}
    {pt3dadd(-31.03, -15.87, 8.64, 0.92)}
    {pt3dadd(-33.32, -19.16, 9.64, 0.92)}
    {pt3dadd(-33.78, -21.63, 10.44, 0.92)}
    {pt3dadd(-33.78, -26.57, 10.88, 0.92)}
    {pt3dadd(-35.62, -30.68, 8.60, 0.92)}
    {pt3dadd(-34.70, -35.21, 8.16, 0.92)}
    {pt3dadd(-35.62, -38.92, 7.88, 0.92)}
    {pt3dadd(-35.62, -43.03, 7.60, 0.92)}
    {pt3dadd(-36.54, -45.91, 9.64, 0.92)}
    {pt3dadd(-37.45, -48.79, 10.76, 0.92)}
    {pt3dadd(-38.37, -53.73, 10.88, 0.92)}
    {pt3dadd(-37.91, -57.02, 10.88, 0.92)}
    {pt3dadd(-39.51, -60.09, 12.28, 0.92)}
    {pt3dadd(-41.34, -62.14, 12.84, 0.92)}
    {pt3dadd(-44.55, -63.79, 12.88, 0.92)}
    {pt3dadd(-43.63, -66.67, 13.68, 0.92)}
    {pt3dadd(-47.30, -68.73, 14.56, 0.92)}
    {pt3dadd(-50.97, -71.20, 14.64, 0.92)}
    {pt3dadd(-60.61, -75.72, 13.80, 0.92)}

    {create dendA4_01}
    {dendA4_0 connect dendA4_01(0), 1}
    {access dendA4_01}
    {nseg = 2}
    {pt3dclear()}
    {pt3dadd(-29.20, -7.23, 8.48, 1.84)}
    {pt3dadd(-32.62, -7.82, 7.40, 1.84)}
    {pt3dadd(-35.38, -7.41, 7.44, 1.84)}
    {pt3dadd(-39.05, -11.12, 5.52, 1.84)}
    {pt3dadd(-41.80, -14.00, 9.76, 1.84)}
    {pt3dadd(-44.09, -18.52, 10.72, 1.84)}
    {pt3dadd(-48.22, -19.35, 10.68, 1.84)}

    {create dendA4_010}
    {dendA4_01 connect dendA4_010(0), 1}
    {access dendA4_010}
    {nseg = 7}
    {pt3dclear()}
    {pt3dadd(-48.22, -19.35, 10.68, 0.92)}
    {pt3dadd(-50.06, -21.81, 10.68, 0.92)}
    {pt3dadd(-53.73, -23.87, 10.68, 0.92)}
    {pt3dadd(-56.02, -25.93, 11.72, 0.92)}
    {pt3dadd(-58.77, -28.81, 9.64, 0.92)}
    {pt3dadd(-60.61, -31.28, 8.20, 0.92)}
    {pt3dadd(-62.44, -35.40, 9.52, 0.92)}
    {pt3dadd(-62.44, -38.69, 10.68, 0.92)}
    {pt3dadd(-66.11, -41.98, 10.40, 0.92)}
    {pt3dadd(-69.78, -45.68, 12.60, 0.92)}
    {pt3dadd(-73.91, -46.92, 12.96, 0.92)}
    {pt3dadd(-78.96, -47.33, 12.96, 0.92)}
    {pt3dadd(-81.71, -50.62, 13.00, 0.92)}
    {pt3dadd(-86.29, -53.09, 13.08, 0.92)}
    {pt3dadd(-89.96, -55.56, 13.24, 0.92)}
    {pt3dadd(-93.18, -56.79, 13.60, 0.92)}
    {pt3dadd(-93.63, -59.26, 13.60, 0.92)}
    {pt3dadd(-97.76, -59.26, 13.60, 0.92)}

    {create dendA4_011}
    {dendA4_01 connect dendA4_011(0), 1}
    {access dendA4_011}
    {nseg = 5}
    {pt3dclear()}
    {pt3dadd(-48.22, -19.35, 10.68, 0.92)}
    {pt3dadd(-50.97, -20.58, 10.56, 0.92)}
    {pt3dadd(-55.10, -20.58, 10.04, 0.92)}
    {pt3dadd(-59.69, -20.58, 11.04, 0.92)}
    {pt3dadd(-61.98, -22.23, 11.80, 0.92)}
    {pt3dadd(-65.19, -23.05, 11.80, 0.92)}
    {pt3dadd(-68.86, -22.23, 12.44, 0.92)}
    {pt3dadd(-69.78, -21.81, 12.88, 0.92)}
    {pt3dadd(-73.91, -22.64, 12.92, 0.92)}
    {pt3dadd(-75.29, -22.64, 13.24, 0.92)}
    {pt3dadd(-78.96, -20.17, 13.24, 0.92)}
    {pt3dadd(-84.00, -18.93, 13.96, 0.92)}
    {pt3dadd(-84.92, -16.47, 14.24, 0.92)}
    {pt3dadd(-87.67, -16.05, 14.24, 0.92)}
    {pt3dadd(-88.59, -19.35, 14.24, 0.92)}

    {create dendA5_0}
    {somaA connect dendA5_0(0), 0.000000}
    {access dendA5_0}
    {nseg = 1}
    {pt3dclear()}
    {pt3dadd(2.70, 5.35, -8.12, 4.58)}
    {pt3dadd(1.78, 2.05, -7.40, 3.66)}

    {create dendA5_00}
    {dendA5_0 connect dendA5_00(0), 1}
    {access dendA5_00}
    {nseg = 6}
    {pt3dclear()}
    {pt3dadd(1.78, 2.05, -7.40, 2.76)}
    {pt3dadd(-3.73, 0.41, -11.12, 2.76)}
    {pt3dadd(-7.85, -0.83, -8.52, 2.76)}
    {pt3dadd(-12.90, -3.30, -10.16, 1.84)}
    {pt3dadd(-14.73, -5.35, -9.48, 1.84)}
    {pt3dadd(-17.03, -6.59, -8.96, 1.84)}
    {pt3dadd(-20.24, -10.29, -8.20, 0.92)}
    {pt3dadd(-22.99, -11.12, -7.92, 0.92)}
    {pt3dadd(-25.74, -14.00, -7.88, 0.92)}
    {pt3dadd(-25.74, -16.05, -7.88, 0.92)}
    {pt3dadd(-27.58, -18.52, -7.88, 0.92)}
    {pt3dadd(-29.87, -22.23, -7.84, 0.92)}
    {pt3dadd(-31.25, -25.93, -7.84, 0.92)}
    {pt3dadd(-33.08, -29.22, -7.84, 0.92)}
    {pt3dadd(-32.62, -33.34, -7.84, 0.92)}
    {pt3dadd(-31.71, -36.63, -7.80, 0.92)}
    {pt3dadd(-31.71, -39.51, -7.72, 0.92)}
    {pt3dadd(-30.79, -43.63, -9.68, 0.92)}
    {pt3dadd(-30.79, -50.62, -11.44, 0.92)}

    {create dendA5_000}
    {dendA5_00 connect dendA5_000(0), 1}
    {access dendA5_000}
    {nseg = 10}
    {pt3dclear()}
    {pt3dadd(-30.79, -50.62, -11.44, 0.92)}
    {pt3dadd(-34.46, -53.50, -13.24, 0.92)}
    {pt3dadd(-33.54, -58.85, -13.76, 0.92)}
    {pt3dadd(-31.71, -60.91, -14.36, 0.92)}
    {pt3dadd(-31.71, -64.20, -14.88, 0.92)}
    {pt3dadd(-34.00, -65.85, -14.88, 0.92)}
    {pt3dadd(-37.21, -65.02, -17.16, 0.92)}
    {pt3dadd(-39.05, -65.44, -18.08, 0.92)}
    {pt3dadd(-43.63, -67.49, -18.84, 0.92)}
    {pt3dadd(-49.14, -68.32, -19.80, 0.92)}
    {pt3dadd(-55.10, -70.79, -20.12, 0.92)}
    {pt3dadd(-61.07, -72.84, -20.60, 0.92)}
    {pt3dadd(-64.28, -76.14, -20.12, 0.92)}
    {pt3dadd(-67.49, -79.43, -20.12, 0.92)}
    {pt3dadd(-72.53, -83.95, -22.04, 0.92)}
    {pt3dadd(-75.74, -84.78, -19.72, 0.92)}
    {pt3dadd(-81.25, -88.07, -19.32, 0.92)}
    {pt3dadd(-89.51, -91.36, -19.32, 0.92)}
    {pt3dadd(-93.18, -93.01, -19.96, 0.92)}
    {pt3dadd(-98.68, -97.53, -18.16, 0.92)}

    {create dendA5_001}
    {dendA5_00 connect dendA5_001(0), 1}
    {access dendA5_001}
    {nseg = 13}
    {pt3dclear()}
    {pt3dadd(-30.79, -50.62, -11.44, 0.92)}
    {pt3dadd(-31.25, -55.15, -10.04, 0.92)}
    {pt3dadd(-30.79, -60.09, -10.00, 0.92)}
    {pt3dadd(-32.17, -64.20, -9.64, 0.92)}
    {pt3dadd(-33.54, -67.49, -8.60, 0.92)}
    {pt3dadd(-33.54, -75.31, -8.28, 0.92)}
    {pt3dadd(-33.08, -79.02, -8.20, 0.92)}
    {pt3dadd(-32.62, -82.72, -8.08, 0.92)}
    {pt3dadd(-29.87, -86.42, -8.08, 0.92)}
    {pt3dadd(-30.33, -92.60, -7.76, 0.92)}
    {pt3dadd(-31.71, -97.12, -6.92, 0.92)}
    {pt3dadd(-29.87, -102.47, -6.16, 0.92)}
    {pt3dadd(-31.25, -109.47, -5.72, 0.92)}
    {pt3dadd(-29.41, -118.93, -5.24, 0.92)}
    {pt3dadd(-26.20, -124.28, -5.24, 0.92)}
    {pt3dadd(-24.83, -126.75, -5.20, 0.92)}
    {pt3dadd(-26.86, -130.71, -3.60, 0.92)}
    {pt3dadd(-25.95, -136.88, -3.08, 0.92)}
    {pt3dadd(-25.95, -145.11, -5.52, 0.92)}
    {pt3dadd(-25.49, -151.29, -5.20, 0.92)}
    {pt3dadd(-24.57, -154.17, -5.20, 0.92)}
    {pt3dadd(-24.11, -159.10, -5.96, 0.92)}
    {pt3dadd(-25.95, -162.40, -8.44, 0.92)}
    {pt3dadd(-28.70, -166.10, -9.40, 0.92)}
    {pt3dadd(-33.29, -169.80, -9.44, 0.92)}
    {pt3dadd(-34.66, -171.04, -10.44, 0.92)}

    {create dendA5_01}
    {dendA5_0 connect dendA5_01(0), 1}
    {access dendA5_01}
    {nseg = 1}
    {pt3dclear()}
    {pt3dadd(1.78, 2.05, -7.40, 0.92)}
    {pt3dadd(-3.93, -1.81, -11.48, 0.92)}

    {create dendA6_0}
    {somaA connect dendA6_0(0), 0.086730}
    {access dendA6_0}
    {nseg = 1}
    {pt3dclear()}
    {pt3dadd(16.03, 10.75, -0.84, 6.42)}
    {pt3dadd(22.00, 9.92, 0.08, 4.58)}
    {pt3dadd(26.12, 9.51, 0.20, 3.66)}

    {create dendA6_00}
    {dendA6_0 connect dendA6_00(0), 1}
    {access dendA6_00}
    {nseg = 5}
    {pt3dclear()}
    {pt3dadd(26.12, 9.51, 0.20, 2.76)}
    {pt3dadd(29.79, 13.63, 0.32, 2.76)}
    {pt3dadd(30.25, 16.10, 2.88, 2.76)}
    {pt3dadd(30.25, 18.98, 3.36, 2.76)}
    {pt3dadd(33.46, 18.57, 3.44, 1.84)}
    {pt3dadd(36.22, 20.62, 4.12, 1.84)}
    {pt3dadd(38.05, 22.68, 5.28, 1.84)}
    {pt3dadd(40.80, 22.27, 6.20, 1.84)}
    {pt3dadd(44.01, 22.68, 7.20, 1.84)}
    {pt3dadd(49.52, 24.33, 8.60, 1.84)}
    {pt3dadd(51.35, 24.74, 10.36, 1.84)}
    {pt3dadd(55.02, 23.92, 11.40, 1.84)}
    {pt3dadd(58.23, 22.68, 12.56, 1.84)}
    {pt3dadd(61.90, 23.50, 13.04, 0.92)}
    {pt3dadd(61.90, 24.74, 12.60, 0.92)}
    {pt3dadd(64.20, 23.92, 12.52, 0.92)}
    {pt3dadd(65.57, 25.56, 13.32, 0.92)}
    {pt3dadd(68.78, 24.33, 13.92, 0.92)}
    {pt3dadd(68.78, 20.62, 11.68, 0.92)}

    {create dendA6_01}
    {dendA6_0 connect dendA6_01(0), 1}
    {access dendA6_01}
    {nseg = 1}
    {pt3dclear()}
    {pt3dadd(26.12, 9.51, 0.20, 1.84)}
    {pt3dadd(28.88, 7.46, 0.40, 1.84)}
    {pt3dadd(33.00, 4.57, 0.40, 1.84)}
    {pt3dadd(36.67, 1.69, 4.04, 1.84)}

    {create dendA6_010}
    {dendA6_01 connect dendA6_010(0), 1}
    {access dendA6_010}
    {nseg = 15}
    {pt3dclear()}
    {pt3dadd(36.67, 1.69, 4.04, 0.92)}
    {pt3dadd(38.51, 6.22, 4.04, 0.92)}
    {pt3dadd(41.72, 6.22, 4.04, 0.92)}
    {pt3dadd(43.10, 7.04, 4.04, 0.92)}
    {pt3dadd(46.77, 10.34, 4.04, 0.92)}
    {pt3dadd(50.44, 13.22, 4.04, 0.92)}
    {pt3dadd(56.86, 16.92, 4.40, 0.92)}
    {pt3dadd(63.74, 21.04, 5.44, 0.92)}
    {pt3dadd(68.78, 23.50, 6.16, 0.92)}
    {pt3dadd(74.29, 27.21, 6.32, 0.92)}
    {pt3dadd(77.50, 30.50, 6.36, 0.92)}
    {pt3dadd(82.55, 32.15, 6.60, 0.92)}
    {pt3dadd(85.76, 35.03, 6.60, 0.92)}
    {pt3dadd(84.84, 39.14, 7.32, 0.92)}
    {pt3dadd(86.22, 44.90, 7.92, 0.92)}
    {pt3dadd(87.13, 51.08, 9.36, 0.92)}
    {pt3dadd(88.05, 56.43, 10.92, 0.92)}
    {pt3dadd(91.72, 58.90, 11.08, 0.92)}
    {pt3dadd(97.22, 60.54, 11.48, 0.92)}
    {pt3dadd(99.06, 63.83, 11.48, 0.92)}
    {pt3dadd(101.81, 68.36, 11.76, 0.92)}
    {pt3dadd(107.78, 74.95, 12.56, 0.92)}
    {pt3dadd(112.36, 79.06, 12.60, 0.92)}
    {pt3dadd(116.76, 80.54, 9.56, 0.92)}
    {pt3dadd(121.34, 86.30, 9.40, 0.92)}
    {pt3dadd(125.93, 90.42, 8.84, 0.92)}
    {pt3dadd(134.65, 95.77, 9.08, 0.92)}

    {create dendA6_011}
    {dendA6_01 connect dendA6_011(0), 1}
    {access dendA6_011}
    {nseg = 2}
    {pt3dclear()}
    {pt3dadd(36.67, 1.69, 4.04, 1.84)}
    {pt3dadd(38.31, -2.62, 4.04, 1.84)}
    {pt3dadd(41.52, -3.44, 4.04, 1.84)}
    {pt3dadd(45.64, -1.39, 4.04, 1.84)}
    {pt3dadd(47.48, -5.91, 4.04, 1.84)}
    {pt3dadd(48.86, -8.79, 8.36, 1.84)}
    {pt3dadd(48.86, -12.09, 10.72, 1.84)}

    {create dendA6_0110}
    {dendA6_011 connect dendA6_0110(0), 1}
    {access dendA6_0110}
    {nseg = 5}
    {pt3dclear()}
    {pt3dadd(48.86, -12.09, 10.72, 1.84)}
    {pt3dadd(54.82, -12.09, 10.72, 1.84)}
    {pt3dadd(57.57, -14.97, 13.00, 1.84)}
    {pt3dadd(58.49, -17.02, 7.64, 1.84)}
    {pt3dadd(59.87, -19.49, 6.80, 0.92)}
    {pt3dadd(63.08, -19.49, 6.40, 0.92)}
    {pt3dadd(66.75, -21.14, 4.96, 0.92)}
    {pt3dadd(69.50, -21.14, 3.80, 0.92)}
    {pt3dadd(71.33, -20.32, 3.04, 0.92)}
    {pt3dadd(74.54, -21.14, 2.96, 0.92)}
    {pt3dadd(77.76, -22.37, 1.68, 0.92)}
    {pt3dadd(81.42, -21.96, 0.72, 0.92)}
    {pt3dadd(85.09, -19.49, -0.80, 0.92)}
    {pt3dadd(88.76, -16.61, 1.76, 0.92)}

    {create dendA6_0111}
    {dendA6_011 connect dendA6_0111(0), 1}
    {access dendA6_0111}
    {nseg = 3}
    {pt3dclear()}
    {pt3dadd(48.86, -12.09, 10.72, 0.46)}
    {pt3dadd(46.56, -13.73, 8.24, 0.46)}
    {pt3dadd(46.56, -16.61, 9.56, 0.46)}
    {pt3dadd(44.73, -18.26, 8.36, 0.46)}
    {pt3dadd(40.14, -16.61, 11.40, 0.46)}

    {create dendA7_0}
    {somaA connect dendA7_0(0), 0.000000}
    {access dendA7_0}
    {nseg = 1}
    {pt3dclear()}
    {pt3dadd(14.91, 10.14, -4.44, 3.66)}
    {pt3dadd(19.96, 7.67, -4.44, 3.66)}
    {pt3dadd(25.00, 4.79, -2.16, 3.66)}

    {create dendA7_00}
    {dendA7_0 connect dendA7_00(0), 1}
    {access dendA7_00}
    {nseg = 1}
    {pt3dclear()}
    {pt3dadd(25.00, 4.79, -2.16, 2.76)}
    {pt3dadd(28.67, 1.08, -2.88, 2.76)}

    {create dendA7_000}
    {dendA7_00 connect dendA7_000(0), 1}
    {access dendA7_000}
    {nseg = 2}
    {pt3dclear()}
    {pt3dadd(28.67, 1.08, -2.88, 1.84)}
    {pt3dadd(28.21, -3.44, -2.88, 1.84)}
    {pt3dadd(27.76, -7.56, -2.88, 1.84)}
    {pt3dadd(27.76, -10.85, 2.76, 1.84)}
    {pt3dadd(28.21, -13.32, 2.84, 1.84)}

    {create dendA7_0000}
    {dendA7_000 connect dendA7_0000(0), 1}
    {access dendA7_0000}
    {nseg = 19}
    {pt3dclear()}
    {pt3dadd(28.21, -13.32, 2.84, 1.84)}
    {pt3dadd(27.30, -19.08, 3.64, 1.84)}
    {pt3dadd(25.46, -22.79, 4.36, 1.84)}
    {pt3dadd(23.17, -28.55, 2.32, 1.84)}
    {pt3dadd(21.79, -34.72, 2.32, 0.92)}
    {pt3dadd(18.12, -35.95, 2.32, 0.92)}
    {pt3dadd(15.37, -39.66, 2.32, 0.92)}
    {pt3dadd(11.70, -42.54, 1.08, 0.92)}
    {pt3dadd(10.32, -44.18, 1.08, 0.92)}
    {pt3dadd(8.95, -49.12, 4.08, 0.92)}
    {pt3dadd(7.11, -52.00, 4.36, 0.92)}
    {pt3dadd(5.28, -52.83, 4.60, 0.92)}
    {pt3dadd(3.44, -56.12, 5.56, 0.92)}
    {pt3dadd(1.61, -59.41, 6.28, 0.92)}
    {pt3dadd(0.69, -60.23, 6.56, 0.92)}
    {pt3dadd(-2.06, -62.70, 6.56, 0.92)}
    {pt3dadd(-2.06, -69.29, 6.56, 0.92)}
    {pt3dadd(-3.25, -76.91, 7.56, 0.92)}
    {pt3dadd(-5.54, -80.21, 7.56, 0.92)}
    {pt3dadd(-6.46, -83.50, 9.16, 0.92)}
    {pt3dadd(-10.13, -88.02, 7.84, 0.92)}
    {pt3dadd(-12.88, -93.37, 7.00, 0.92)}
    {pt3dadd(-13.80, -95.02, 6.76, 0.92)}
    {pt3dadd(-13.80, -97.49, 6.56, 0.92)}
    {pt3dadd(-16.55, -100.78, 9.56, 0.92)}
    {pt3dadd(-17.01, -102.84, 10.40, 0.92)}
    {pt3dadd(-20.22, -107.78, 11.08, 0.92)}
    {pt3dadd(-21.14, -112.30, 11.52, 0.92)}
    {pt3dadd(-22.97, -115.60, 11.72, 0.92)}
    {pt3dadd(-27.56, -118.48, 11.72, 0.92)}
    {pt3dadd(-29.85, -123.42, 12.96, 0.92)}
    {pt3dadd(-31.23, -125.88, 8.84, 0.92)}
    {pt3dadd(-33.06, -131.65, 8.32, 0.92)}
    {pt3dadd(-35.82, -136.58, 7.92, 0.92)}
    {pt3dadd(-35.82, -138.64, 9.56, 0.92)}
    {pt3dadd(-41.32, -143.58, 6.76, 0.92)}
    {pt3dadd(-43.36, -151.62, 5.36, 0.92)}
    {pt3dadd(-47.03, -154.09, 5.12, 0.92)}
    {pt3dadd(-49.33, -158.21, 3.92, 0.92)}
    {pt3dadd(-53.00, -163.56, 3.28, 0.92)}
    {pt3dadd(-57.13, -172.20, 2.60, 0.92)}
    {pt3dadd(-63.09, -173.85, 2.96, 0.92)}

    {create dendA7_0001}
    {dendA7_000 connect dendA7_0001(0), 1}
    {access dendA7_0001}
    {nseg = 3}
    {pt3dclear()}
    {pt3dadd(28.21, -13.32, 2.84, 1.84)}
    {pt3dadd(29.81, -15.56, 4.28, 1.84)}
    {pt3dadd(31.64, -12.27, 6.32, 1.84)}
    {pt3dadd(33.48, -11.04, 7.76, 1.84)}
    {pt3dadd(36.23, -14.74, 9.08, 1.84)}
    {pt3dadd(35.31, -15.15, 10.16, 1.84)}
    {pt3dadd(32.56, -15.56, 10.80, 1.84)}
    {pt3dadd(29.81, -16.80, 10.80, 1.84)}
    {pt3dadd(33.48, -18.86, 10.92, 1.84)}
    {pt3dadd(33.02, -23.79, 12.08, 1.84)}
    {pt3dadd(31.64, -28.32, 12.88, 0.92)}
    {pt3dadd(34.85, -32.02, 14.24, 0.92)}

    {create dendA7_001}
    {dendA7_00 connect dendA7_001(0), 1}
    {access dendA7_001}
    {nseg = 7}
    {pt3dclear()}
    {pt3dadd(28.67, 1.08, -2.88, 1.84)}
    {pt3dadd(33.48, -1.57, -2.52, 1.84)}
    {pt3dadd(37.14, -1.98, -3.08, 1.84)}
    {pt3dadd(42.65, -1.98, -3.96, 1.84)}
    {pt3dadd(46.32, -4.45, -4.20, 1.84)}
    {pt3dadd(52.28, -7.33, -4.36, 0.92)}
    {pt3dadd(54.58, -9.39, -4.60, 0.92)}
    {pt3dadd(59.62, -9.39, -7.72, 0.92)}
    {pt3dadd(63.29, -9.39, -8.56, 0.92)}
    {pt3dadd(70.63, -11.04, -8.44, 0.92)}
    {pt3dadd(73.84, -13.09, -8.44, 0.92)}
    {pt3dadd(78.89, -16.80, -10.40, 0.92)}
    {pt3dadd(82.56, -16.80, -10.56, 0.92)}
    {pt3dadd(83.48, -22.56, -11.48, 0.92)}
    {pt3dadd(84.85, -25.03, -12.96, 0.92)}
    {pt3dadd(88.06, -28.32, -14.80, 0.92)}
    {pt3dadd(90.36, -28.73, -15.16, 0.92)}

    {create dendA7_0010}
    {dendA7_001 connect dendA7_0010(0), 1}
    {access dendA7_0010}
    {nseg = 6}
    {pt3dclear()}
    {pt3dadd(90.36, -28.73, -15.16, 0.92)}
    {pt3dadd(95.86, -26.67, -16.44, 0.92)}
    {pt3dadd(99.99, -25.44, -17.52, 0.92)}
    {pt3dadd(104.58, -24.21, -14.44, 0.92)}
    {pt3dadd(107.79, -24.21, -12.80, 0.92)}
    {pt3dadd(110.54, -24.62, -11.24, 0.92)}
    {pt3dadd(116.96, -23.38, -11.32, 0.92)}
    {pt3dadd(122.92, -27.09, -11.40, 0.92)}
    {pt3dadd(133.93, -33.26, -11.24, 0.92)}
    {pt3dadd(137.14, -36.14, -11.20, 0.92)}

    {create dendA7_0011}
    {dendA7_001 connect dendA7_0011(0), 1}
    {access dendA7_0011}
    {nseg = 4}
    {pt3dclear()}
    {pt3dadd(90.36, -28.73, -15.16, 0.92)}
    {pt3dadd(93.11, -34.49, -16.52, 0.92)}
    {pt3dadd(97.24, -39.02, -15.64, 0.92)}
    {pt3dadd(97.70, -45.19, -19.00, 0.92)}
    {pt3dadd(93.11, -48.49, -20.12, 0.92)}
    {pt3dadd(88.98, -50.95, -21.84, 0.46)}
    {pt3dadd(86.23, -51.78, -22.20, 0.46)}

    {create dendA7_01}
    {dendA7_0 connect dendA7_01(0), 1}
    {access dendA7_01}
    {nseg = 1}
    {pt3dclear()}
    {pt3dadd(25.00, 4.79, -2.16, 1.84)}
    {pt3dadd(30.72, 2.96, -5.84, 1.84)}

    {create dendA7_010}
    {dendA7_01 connect dendA7_010(0), 1}
    {access dendA7_010}
    {nseg = 6}
    {pt3dclear()}
    {pt3dadd(30.72, 2.96, -5.84, 1.84)}
    {pt3dadd(34.39, 4.60, -5.84, 1.84)}
    {pt3dadd(38.98, 3.78, -5.84, 1.84)}
    {pt3dadd(42.65, 2.13, -5.84, 0.92)}
    {pt3dadd(47.24, 2.13, -7.48, 0.92)}
    {pt3dadd(52.74, 0.07, -7.96, 0.92)}
    {pt3dadd(60.08, -4.04, -8.80, 0.92)}
    {pt3dadd(62.37, -6.10, -9.80, 0.92)}
    {pt3dadd(65.59, -7.33, -10.64, 0.92)}
    {pt3dadd(70.63, -7.74, -10.32, 0.92)}
    {pt3dadd(76.59, -8.57, -11.28, 0.92)}
    {pt3dadd(81.64, -8.16, -12.60, 0.92)}
    {pt3dadd(84.85, -14.74, -12.80, 0.92)}
    {pt3dadd(89.44, -15.15, -15.36, 0.92)}

    {create dendA7_0100}
    {dendA7_010 connect dendA7_0100(0), 1}
    {access dendA7_0100}
    {nseg = 17}
    {pt3dclear()}
    {pt3dadd(89.44, -15.15, -15.36, 0.92)}
    {pt3dadd(94.94, -13.09, -14.56, 0.92)}
    {pt3dadd(97.24, -15.56, -13.72, 0.92)}
    {pt3dadd(101.82, -15.56, -17.44, 0.92)}
    {pt3dadd(105.49, -12.27, -18.56, 0.92)}
    {pt3dadd(108.70, -15.56, -19.44, 0.92)}
    {pt3dadd(110.54, -11.45, -20.12, 0.92)}
    {pt3dadd(113.75, -8.98, -20.76, 0.92)}
    {pt3dadd(116.96, -8.98, -21.84, 0.92)}
    {pt3dadd(118.80, -6.92, -23.80, 0.92)}
    {pt3dadd(122.92, -9.80, -27.44, 0.92)}
    {pt3dadd(130.72, -7.74, -28.40, 0.92)}
    {pt3dadd(134.39, -6.10, -29.44, 0.92)}
    {pt3dadd(138.52, -5.69, -30.52, 0.92)}
    {pt3dadd(143.11, -8.16, -31.24, 0.92)}
    {pt3dadd(145.40, -10.63, -32.00, 0.92)}
    {pt3dadd(149.99, -11.86, -32.60, 0.92)}
    {pt3dadd(156.41, -13.92, -33.16, 0.92)}
    {pt3dadd(159.84, -16.16, -39.88, 0.92)}
    {pt3dadd(164.43, -17.81, -42.56, 0.92)}
    {pt3dadd(167.64, -17.81, -43.64, 0.92)}
    {pt3dadd(170.39, -19.87, -45.68, 0.92)}
    {pt3dadd(173.60, -22.75, -46.88, 0.92)}
    {pt3dadd(174.98, -23.98, -48.00, 0.92)}
    {pt3dadd(181.40, -28.92, -50.28, 0.92)}
    {pt3dadd(194.70, -32.62, -54.32, 0.92)}
    {pt3dadd(196.08, -32.21, -56.12, 0.92)}
    {pt3dadd(201.12, -33.04, -58.64, 0.92)}
    {pt3dadd(210.76, -37.15, -59.68, 0.92)}
    {pt3dadd(211.22, -35.09, -60.84, 0.92)}
    {pt3dadd(216.26, -32.21, -62.68, 0.92)}
    {pt3dadd(225.44, -30.98, -67.04, 0.92)}

    {create dendA7_0101}
    {dendA7_010 connect dendA7_0101(0), 1}
    {access dendA7_0101}
    {nseg = 8}
    {pt3dclear()}
    {pt3dadd(89.44, -15.15, -15.36, 0.92)}
    {pt3dadd(94.24, -16.16, -14.20, 0.92)}
    {pt3dadd(97.91, -19.46, -14.20, 0.92)}
    {pt3dadd(102.50, -21.10, -11.36, 0.92)}
    {pt3dadd(107.55, -23.16, -10.68, 0.92)}
    {pt3dadd(114.43, -24.39, -9.44, 0.92)}
    {pt3dadd(118.56, -24.39, -9.20, 0.92)}
    {pt3dadd(122.68, -23.16, -12.00, 0.92)}
    {pt3dadd(127.73, -22.34, -13.16, 0.92)}
    {pt3dadd(130.48, -26.86, -14.48, 0.92)}
    {pt3dadd(135.07, -30.98, -11.08, 0.92)}
    {pt3dadd(143.33, -34.27, -8.32, 0.92)}
    {pt3dadd(148.83, -37.56, -8.48, 0.46)}
    {pt3dadd(155.25, -36.74, -9.20, 0.46)}

    {create dendA7_011}
    {dendA7_01 connect dendA7_011(0), 1}
    {access dendA7_011}
    {nseg = 8}
    {pt3dclear()}
    {pt3dadd(30.72, 2.96, -5.84, 1.84)}
    {pt3dadd(34.15, 0.30, -9.64, 1.84)}
    {pt3dadd(50.67, -4.64, -9.64, 1.84)}
    {pt3dadd(58.92, -7.93, -9.28, 0.92)}
    {pt3dadd(63.97, -12.05, -9.80, 0.92)}
    {pt3dadd(69.93, -17.40, -9.80, 0.92)}
    {pt3dadd(72.23, -29.74, -7.40, 0.92)}
    {pt3dadd(72.68, -37.56, -7.04, 0.92)}
    {pt3dadd(72.23, -41.27, -7.56, 0.46)}
    {pt3dadd(71.31, -46.20, -7.92, 1.84)}
    {pt3dadd(71.77, -50.73, -6.28, 1.84)}
    {pt3dadd(72.23, -54.02, -5.52, 1.84)}
    {pt3dadd(70.85, -58.96, -5.68, 1.84)}

    {create dendA7_0110}
    {dendA7_011 connect dendA7_0110(0), 1}
    {access dendA7_0110}
    {nseg = 4}
    {pt3dclear()}
    {pt3dadd(70.85, -58.96, -5.68, 0.92)}
    {pt3dadd(73.14, -63.08, -5.00, 0.92)}
    {pt3dadd(73.14, -70.48, -4.72, 0.92)}
    {pt3dadd(74.98, -73.78, -3.76, 0.92)}
    {pt3dadd(79.11, -74.60, -8.04, 0.92)}
    {pt3dadd(80.94, -79.13, -7.44, 0.92)}
    {pt3dadd(80.48, -86.12, -5.24, 0.92)}
    {pt3dadd(80.73, -87.95, -5.40, 0.92)}

    {create dendA7_0111}
    {dendA7_011 connect dendA7_0111(0), 1}
    {access dendA7_0111}
    {nseg = 4}
    {pt3dclear()}
    {pt3dadd(70.85, -58.96, -5.68, 0.92)}
    {pt3dadd(69.26, -64.08, -6.24, 0.92)}
    {pt3dadd(68.80, -68.20, -5.52, 0.92)}
    {pt3dadd(66.51, -72.31, -8.60, 0.92)}
    {pt3dadd(67.89, -77.25, -10.36, 0.92)}
    {pt3dadd(66.97, -81.78, -12.04, 0.92)}
    {pt3dadd(69.26, -84.66, -13.68, 0.92)}
    {pt3dadd(68.34, -87.95, -10.96, 0.92)}


    define_shape()
    objref mbSec
  """)

In [ ]:
# @title Create the hoc file `Model_specification.hoc`
# @markdown Execute this cell.
with open("Model_specification.hoc", "w") as file:
  file.write("""
    // --------------------------------------------------------------
    // passive & active membrane
    // --------------------------------------------------------------
    objref mbSec, dendritic_only
    //create somaA,iseg,hill,myelin[2],node[2],dendA1_0
    create iseg,hill,myelin[2],node[2]
    //access somaA

    dendA1_0000000000001 mbSec = new SectionRef()
    dendritic_only = new SectionList()

    //make sure no compartments exceed 20 uM length
    forall {
      //if(nseg < L/20) { print secname(), " not accurate" nseg=L/20+1 }
      dendritic_only.append()
    }    

    somaA  dendritic_only.remove()

    spA  = 2 // increase in membrane area due to spines

    ra = 80
    global_ra = ra
    rm = 30000
    c_m = 0.6
    cm_myelin = 0.04
    g_pas_node = 0.02

    v_init = -70
    celsius = 34      

    Ek = -90
    Ena = 60

    gna_dend = 27 
    gna_node = 30000   
    gna_soma = 54
    gna_myelin = 80  
    gkv_axon = 3000 
    gkv_soma = 600 
    gkv_dend = 30 

    gca_dend =1.5
    gkm_dend =.1
    gkca_dend=3.25

    gca_soma = 3 
    gkm_soma = 0.2 
    gkca_soma = 6.5 

    gka_soma = 0.06        
    gka_dend = 0.02
    gka_slope = 0 // no gradient

    tauR = 80
    caiExp = 4
    rA = 0.05
    rB = 0.1

    // --------------------------------------------------------------
    // Low Threshold Ca Channel to reproduce frequency effect (Larkum, Kaiser, Sakmann, PNAS,1999)
    // --------------------------------------------------------------
    vh1_it2=56
    vh2_it2=415
    ah_it2=30       
    v12m_it2=45
    v12h_it2=65  
    am_it2=3
    vshift_it2=10
    vm1_it2=50
    vm2_it2=125
    it2_init=0.0005   
    gca_init=4.5

    // --------------------------------------------------------------
    // initiation zone in the dendrite 
    // with slightly elevated Ca conductance densities
    // --------------------------------------------------------------

    proc InitZone() {
      mbSec.sec distance(0,1)
      forall for(x) if(distance(x) <100) {
        //print secname()
        gbar_sca = gca_init
        gcabar_it2 =0.0005
      }
    }

    // --------------------------------------------------------------
    // Axon geometry
    // Similar to Mainen et al (Neuron, 1995)
    // --------------------------------------------------------------
    n_axon_seg = 5

    proc create_axon() { local somaArea

        create iseg,hill,myelin[n_axon_seg],node[n_axon_seg]

        somaA {
            somaArea=0
            for(x) somaArea+=area(x)
            equiv_diam = sqrt(somaArea/(4*PI))
        }

        iseg {                
            pt3dclear() pt3dadd(0,0,0,diam) pt3dadd(0,-1000,0,diam)
            L = 15
            nseg = 5
            diam = equiv_diam/10    
        }

        hill {                
            pt3dclear() pt3dadd(0,0,0,diam) pt3dadd(0,-1000,0,diam)
            L = 10
            nseg = 5
            diam(0:1) = 5.4728284:1.3682071
        }

        for i=0,n_axon_seg-1 {
            myelin[i] { // myelin element
                pt3dclear() pt3dadd(0,0,0,diam) pt3dadd(0,-1000,0,diam)
                nseg = 5
                L = 100
                diam = iseg.diam         
            }
            node[i] { // nodes of Ranvier
                pt3dclear() pt3dadd(0,0,0,diam) pt3dadd(0,-1000,0,diam)
                nseg = 1
                L = 1.0
                diam = iseg.diam*.75 // nodes are thinner than axon
            }
        }
        somaA connect hill(0), 0.5
        hill connect iseg(0), 1
        iseg connect myelin[0](0), 1
        myelin[0] connect node[0](0), 1
        for i=0,n_axon_seg-2 { 
        node[i] connect myelin[i+1](0), 1 
        myelin[i+1] connect node[i+1](0), 1
        }
    }

    // --------------------------------------------------------------
    // Spines
    // --------------------------------------------------------------
    proc add_spines() { 

        // increase all dendritic conductances by factor spA
        // increase dendritic cm and g_pas by same
        // to account for increase in membrane area without changing distances etc

        forsec dendritic_only {
            cm *=spA
            g_pas *=spA
            gbar_na *=spA
            gbar_kv *=spA
            gbar_km *=spA
            gbar_kca *=spA
            gbar_sca *=spA
            gbar_kap *=spA
            gcabar_it2*=spA
        }
    }
    // --------------------------------------------------------------
    // Initialization routines
    // --------------------------------------------------------------
    proc init_cell() {  
        // passive
        forall {
            insert pas
            Ra = ra 
            cm = c_m 
            g_pas = 1/rm
            e_pas = v_init
        }
        // exceptions along the axon
        forsec "myelin" cm = cm_myelin
        forsec "node" g_pas = g_pas_node
        // active 
        // axon
        forall insert na
        forsec "myelin" gbar_na = gna_myelin
        forsec "hill" gbar_na = gna_node
        forsec "iseg" gbar_na = gna_node
        forsec "node" gbar_na = gna_node
        forsec "iseg" { insert kv  gbar_kv = gkv_axon }
        forsec "hill" { insert kv  gbar_kv = gkv_axon }

        // dendrites
        forsec dendritic_only {
            gbar_na = gna_dend
            insert kv    gbar_kv = gkv_dend 
            insert km    gbar_km  = gkm_dend
            insert kca   gbar_kca = gkca_dend
            insert kap   gkabar_kap = gka_soma
            insert sca   gbar_sca = gca_dend
            insert it2   gcabar_it2=0
            insert cad2
        }

        // soma
        somaA {
            gbar_na = gna_soma
            insert kv    gbar_kv = gkv_soma 
            insert km    gbar_km = gkm_soma
            insert kca   gbar_kca = gkca_soma
            insert kap   gkabar_kap = gka_soma
            insert sca   gbar_sca = gca_soma
            insert cad2
        }

        forall if(ismembrane("k_ion")) ek = Ek
        forall if(ismembrane("na_ion")) {
            ena = Ena
            vshift_na = -5
        }
        forall if(ismembrane("ca_ion")) {
          eca = 140
          ion_style("ca_ion",0,1,0,0,0)
          vshift_ca = 10
        }
        // ca diffusion  and kca parameters
        taur_cad2 = tauR
        caix_kca  = caiExp
        Ra_kca = rA
        Rb_kca = rB

        InitZone()
        add_spines()
    }

    proc init() { local saveDt, i
        finitialize(v_init)
        fcurrent()
        saveDt = dt
        dt = 10
        for i=0,99 fadvance()
        dt = saveDt
    }
  """)

In [ ]:
!rm -rf x86_64/
!nrnivmodl

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from neuron import h
h.load_file("stdrun.hoc")

In [ ]:
# @title Make nicer plots -- Execute this cell
def mystyle():
  """
  Create custom plotting style.

  Returns
  -------
  my_style : dict
      Dictionary with matplotlib parameters.

  """
  # color pallette
  style = {
      # Use LaTeX to write all text
      "text.usetex": False,
      "font.family": "DejaVu Sans",
      "font.weight": "bold",
      # Use 16pt font in plots, to match 16pt font in document
      "axes.labelsize": 16,
      "axes.titlesize": 20,
      "font.size": 16,
      # Make the legend/label fonts a little smaller
      "legend.fontsize": 14,
      "xtick.labelsize": 14,
      "ytick.labelsize": 14,
      "axes.linewidth": 2.5,
      "lines.markersize": 10.0,
      "lines.linewidth": 2.5,
      "xtick.major.width": 2.2,
      "ytick.major.width": 2.2,
      "axes.labelweight": "bold",
      "axes.spines.right": False,
      "axes.spines.top": False
  }

  return style


plt.style.use("seaborn-colorblind")
plt.rcParams.update(mystyle())

In [ ]:
h.xopen("morphology.nrn")
h.xopen("Model_specification.hoc")

In [ ]:
tstop = 1100
h.dt = 0.025

In [ ]:
for sec in h.allsec():
  if h.nseg < h.L/20:
    h.nseg = int(h.L/20 + 1)

h.create_axon()
h.init_cell()

In [ ]:
# Create a current Clamp procedure called "IatSoma" starting at 1005.1 ms and with duration=5ms. Amplitude is an argument.     
ic = h.IClamp(h.somaA(0.5))
ic.delay = 1005.1  # ms
ic.dur = 5  # ms
ic.amp = 1.8  # nA

In [ ]:
# Include an EPSP
syn = h.epsp(h.mbSec.sec(0))
syn.tau0 = 0.8
syn.tau1 = 4
syn.onset = 1007.1
syn.imax = 0 #0.6

In [ ]:
vsoma_vec = h.Vector().record(h.somaA(0.5)._ref_v)  # Membrane potential vector
vdend1_vec = h.Vector().record(h.mbSec.sec(0.5)._ref_v)  # Membrane potential vector
t_vec = h.Vector().record(h._ref_t)  # Time stamp vector

In [ ]:
sh1 = h.PlotShape(0)
sh1.size(-386.7, 313.3, -150.382, 1048.64)
sh1.variable("v")
{sh1.view(-386.7, -150.382, 700, 1199.02, 331, 30, 325, 570)}
h.fast_flush_list.append(sh1)
sh1.save_name("fast_flush_list.")
sh1.show(1)
sh1.exec_menu("Shape Plot")
sh1.plot(plt)
plt.show()

In [ ]:
sh2 = h.PlotShape(0)
sh2.size(-792.112, 571.207, -154.611, 1031.33)
sh2.variable("cai")
{sh2.view(-792.112, -154.611, 1363.32, 1185.94, 5, 340, 300, 260)}
h.fast_flush_list.append(sh2)
sh2.save_name("fast_flush_list.")
sh2.show(1)
sh2.exec_menu("Shape Plot")
sh2.scale(0, 3)
sh2.plot(plt)
plt.show()

In [ ]:
h.finitialize()
h.continuerun(tstop)

# Remove the first 1000 ms
tremove = 1000  # ms
vsoma_vec.remove(0, int(tremove/h.dt))
vdend1_vec.remove(0, int(tremove/h.dt))
t_vec.remove(0, int(tremove/h.dt))

plt.figure(figsize=(8, 6))
plt.plot(t_vec, vsoma_vec, color='black', label='soma')
plt.plot(t_vec, vdend1_vec, color='red', label='dend')
plt.xlabel('time (ms)')
plt.ylabel('voltage mV')
plt.legend()
plt.show()